# Logistic Regression — Chapter 4 (Math on Screen)

Chapter 4 wraps any plot animation in a **fixed template**: plot upper-left, math rows on the right, formula columns below.

## Abstractions (use for every clip)

| Module | Purpose |
|--------|---------|
| `tutorial_template.py` | `TutorialComposer`, `TutorialScene`, `TutorialTheme`, layout/typography/export |
| `handwrite_tutorial.py` | Patrick Hand + subscripts + symbol font + write-on reveal |
| `ch4_layout.py` | Chapter 4 defaults + `ch4_compose_tutorial_frame()` |

```python
scene = TutorialScene(plot=frame, math_right_blocks=..., math_bottom_blocks=...)
composer = make_composer("dark_rails")  # or any theme name
img = composer.render_scene(scene, write_progress=t)
```

All clips share **15.0×9.5 in @ 200 DPI → 3000×1900 px**. Typography and spacing live in `TutorialTypography`; colors/gradients in `TutorialTheme`.

## Exports

| Cell | Output |
|------|--------|
| **5** | Template PNG (`dark_rails`) |
| **6** | Handwrite demo + five theme MP4s |
| **7–13** | Likelihood story (one cell each): `ch4-likelihood-02` … `ch4-likelihood-07` → `ch4_02` … `ch4_07` |


In [223]:
import importlib
import json
from pathlib import Path

import handwrite_tutorial
import tutorial_template
import ch4_layout

importlib.reload(handwrite_tutorial)
importlib.reload(tutorial_template)
importlib.reload(ch4_layout)
from ch4_layout import *



Chapter 4 layout OK — handwriting: Patrick Hand


In [224]:
# Reuse all Chapter 3 builders (datasets, knobs, triptych, strip, duo, …).
_CH3_NB = Path("logistic-regression-chap3.ipynb")
if not _CH3_NB.is_file():
    raise FileNotFoundError(_CH3_NB.resolve())
_ch3_src = "".join(json.loads(_CH3_NB.read_text())["cells"][1]["source"])
exec(compile(_ch3_src, str(_CH3_NB), "exec"), globals())
del _CH3_NB, _ch3_src



Chapter 3 setup OK — 20 clean, 26 with noise.


logistic-regression-chap3.ipynb:580: SyntaxWarning: invalid escape sequence '\s'
  "cell_type": "code",
logistic-regression-chap3.ipynb:584: SyntaxWarning: invalid escape sequence '\s'
  "outputs": [],


In [225]:
def ch4_sample_mistakes_triptych_frame():
    """One representative frame from the ch3_25 mistakes triptych (split-screen) family.

    Returns ``(plot_img, w_st, w_el, b)`` for math-rail values.
    """
    spec = CH3_LOSS_SPECS["mistakes"]
    loss_fn = spec["fn"]
    study, exam, y = study_sep, exam_sep, y_sep
    wr, we, br = ch3_triptych_script_weights()
    seqs, triples, losses = {}, {}, {}
    for which in ("st", "el", "b"):
        c = ch3_active_value(which, wr, we, br)
        seqs[which] = ch3_quad_sweep(c, CH3_LOSS_SYM_DELTA, CH3_SWEEP_NSEG)
        triples[which] = [ch3_triplet(which, float(v)) for v in seqs[which]]
        losses[which] = [loss_fn(ws, we, bb, study, exam, y) for ws, we, bb in triples[which]]
    all_l = np.concatenate([losses["st"], losses["el"], losses["b"]])
    pad_y = 0.06 * max(1e-6, float(np.nanmax(all_l) - np.nanmin(all_l)))
    y_lo = float(np.nanmin(all_l) - pad_y)
    y_hi = float(np.nanmax(all_l) + pad_y)

    def _xlim(seq):
        span = float(np.max(seq) - np.min(seq))
        pad_x = max(0.06 * span, 0.08)
        return float(np.min(seq) - pad_x), float(np.max(seq) + pad_x)

    ws, we, bb = triples["el"][len(triples["el"]) // 2]
    n_st = len(seqs["st"])
    n_el = len(seqs["el"])
    n_b = max(2, len(seqs["b"]) // 2)
    rots = _ch3_triptych_pack_knob_rots(ws, we, bb, wr, we, br)
    frame = ch3_triptych_frame(
        ws,
        we,
        bb,
        study,
        exam,
        y,
        spec,
        show_colormap=bool(spec["colormap"]),
        xs_st=seqs["st"][:n_st],
        ys_st=losses["st"][:n_st],
        xs_el=seqs["el"][:n_el],
        ys_el=losses["el"][:n_el],
        xs_b=seqs["b"][:n_b],
        ys_b=losses["b"][:n_b],
        x_lim_st=_xlim(seqs["st"]),
        x_lim_el=_xlim(seqs["el"]),
        x_lim_b=_xlim(seqs["b"]),
        y_lo=y_lo,
        y_hi=y_hi,
        knob_rots=rots,
        knob_scales=[1.0, float(CH3_KNOB_ACTIVE_SCALE), 1.0],
        arrows=None,
        panel_visible=(True, True, True),
        emphasize_knob="el",
    )
    return frame, float(ws), float(we), float(bb)


def ch4_math_right_blocks(w_st, w_el, b, study, exam, y):
    """Right-rail rows: current weights, gradient, and NLL at ``(w_st, w_el, b)``."""
    z = logits_plane(w_st, w_el, b, study, exam)
    p = sigmoid(z)
    yy = y.astype(float)
    eps = 1e-12
    nll = float(-np.sum(yy * np.log(p + eps) + (1.0 - yy) * np.log(1.0 - p + eps)))
    resid = p - yy
    gw1 = float(np.sum(resid * study))
    gw2 = float(np.sum(resid * exam))
    gb = float(np.sum(resid))
    w_text = rf"$w_1={w_st:.2f}$" + "\n" + rf"$w_2={w_el:.2f}$" + "\n" + rf"$b={b:.2f}$"
    g_text = (
        rf"$\partial w_1={gw1:.2f}$"
        + "\n"
        + rf"$\partial w_2={gw2:.2f}$"
        + "\n"
        + rf"$\partial b={gb:.2f}$"
    )
    return [
        {"label": r"$w,\,b$", "text": w_text, "bold_lhs": True, "role": "weights"},
        {"label": r"$\nabla\mathrm{NLL}$", "text": g_text, "bold_lhs": True, "role": "gradient"},
        {"label": "NLL", "text": rf"$NLL={nll:.2f}$", "bold_lhs": True, "role": "nll"},
    ]

CH4_BOTTOM_FORMULA_BLOCKS = [
    {"text": r"$p(y_i \mid x_i)=\hat p_i^{\,y_i}(1-\hat p_i)^{1-y_i}$", "bold_lhs": True, "role": "formula"},
    {"text": r"$\mathrm{NLL}(w)=-\sum_i \log p(y_i \mid x_i)$", "bold_lhs": True, "role": "formula"},
    {"text": r"$\nabla_w\,\mathrm{NLL}=\sum_i(\hat p_i-y_i)\,x_i$", "bold_lhs": True, "role": "formula"},
]


def ch4_tutorial_scene():
    """Shared plot + math blocks for static PNG and animated MP4."""
    plot, w1, w2, b = ch4_sample_mistakes_triptych_frame()
    return {
        "plot": plot,
        "math_right_blocks": ch4_math_right_blocks(w1, w2, b, study_sep, exam_sep, y_sep),
        "math_bottom_blocks": CH4_BOTTOM_FORMULA_BLOCKS,
    }


def ch4_render_tutorial_frame(scene, write_progress=1.0, *, theme=None, composer=None):
    return ch4_compose_tutorial_frame(
        scene["plot"],
        math_right_blocks=scene["math_right_blocks"],
        math_bottom_blocks=scene["math_bottom_blocks"],
        write_progress=write_progress,
        theme=theme,
        composer=composer,
    )


def ch4_export_handwrite_demo_mp4(n_frames=32, ms_per_frame=100):
    scene = ch4_tutorial_scene()
    return CH4_COMPOSER.export_mp4(
        ch4_scene_from_dict(scene),
        "ch4_01_handwrite_demo.mp4",
        save_mp4=save_mp4,
        output_dir=OUTPUT_DIR,
        n_frames=n_frames,
        ms_per_frame=ms_per_frame,
    )


def ch4_export_all_theme_demos(n_frames=32, ms_per_frame=100):
    """Export handwrite demo for each of the 5 color themes."""
    return ch4_export_theme_demos(
        ch4_tutorial_scene(),
        save_mp4=save_mp4,
        n_frames=n_frames,
        ms_per_frame=ms_per_frame,
    )



















































































































# --- ch4_02: likelihood(w1,w2) landscape — duo left, tall 3D right ---
CH3_LIK_W12_W_ST0 = float(CH3_SCRIPT_K1_W_ST)
CH3_LIK_W12_W_EL0 = float(CH3_SCRIPT_K1_W_EL)
CH3_LIK_W12_B0 = float(CH3_SCRIPT_K1_B)
CH3_LIK_W12_W1_LO = float(CH3_SCRIPT_W1_WIDE[0])
CH3_LIK_W12_W1_HI = float(CH3_SCRIPT_W1_WIDE[1])
CH3_LIK_W12_W2_LO = float(CH3_LIK_W12_W_EL0 - CH3_SCRIPT_K1_STOP_SWEEP_DELTA)
CH3_LIK_W12_W2_HI = float(CH3_LIK_W12_W_EL0 + CH3_SCRIPT_K1_STOP_SWEEP_DELTA)
CH3_LIK_W12_B_HALF = float(CH3_SCRIPT_K1_TIGHT_SWEEP_DELTA * 20.0)
CH3_LIK_W12_FIGSIZE = EXPORT_FIGSIZE
CH3_LIK_W12_WIDTH_RATIOS = (1.22, 1.42)
CH3_LIK_W12_GRID_N = 52 if not _CH3_DRAFT else 28
CH3_LIK_W12_GRID_N_COARSE = 22 if not _CH3_DRAFT else 14
CH3_LIK_W12_GRID_N_FINE = 72 if not _CH3_DRAFT else 36
CH3_LIK_W12_QUAD_MID = 0.0
CH3_LIK_W12_CURVE_LW = 2.4
CH3_LIK_W12_SURFACE_ALPHA = 0.88 * 0.9
CH3_LIK_W12_SLICE_ALPHA = 0.42 * 0.9
CH3_LIK_W12_MS = 90 if not _CH3_DRAFT else 110
CH3_LIK_W12_N_HOLD = max(10, CH3_SCRIPT_N_HOLD // 4)
CH3_LIK_W12_N_KNOB = max(28, _smooth_n(22))
CH3_LIK_W12_N_ROT = max(32, _smooth_n(24))
CH3_LIK_W12_N_REVEAL = max(40, _smooth_n(30))
CH3_LIK_W12_N_FILL_LINES = CH3_LIK_W12_GRID_N
CH3_LIK_W12_N_FILL_TRACE = max(48, _smooth_n(36)) if not _CH3_DRAFT else max(24, _smooth_n(18))
CH3_LIK_W12_N_B_SLICES = 7 if not _CH3_DRAFT else 4
CH3_LIK_W12_N_ORBIT = max(48, _smooth_n(36))
CH3_LIK_W12_AZIM_W1 = 90.0
CH3_LIK_W12_AZIM_W2 = 180.0
CH3_LIK_W12_AZIM_BOTH = -54.0
CH3_LIK_W12_AZIM_STACK = 90.0
CH3_LIK_W12_ELEV_W1 = 0.0
CH3_LIK_W12_ELEV_W2 = 0.0
CH3_LIK_W12_ELEV_BOTH = 26.0
CH3_LIK_W12_ELEV_STACK = 0.0
CH3_LIK_W12_N_SQUISH = max(28, _smooth_n(22))
CH3_LIK_W12_CORNER_W1 = CH3_LIK_W12_W1_LO
CH3_LIK_W12_CORNER_W2 = CH3_LIK_W12_W2_LO
# 3-D knob 1 / knob 2 axis display (matches ch4_03 ±3 cube)
CH3_LIK_W12_3D_LO = -3.0
CH3_LIK_W12_3D_HI = 3.0
CH3_LIK_W12_CT_ELEV = 24.0
CH3_LIK_W12_CT_AZIM = -128.0


def _ch3_lik_w12_axis_refined(lo, hi, mid, n_coarse, n_fine, *, fine_lo_half):
    """Denser samples on one half-axis; ``mid`` is the split (typically 0)."""
    lo, hi, mid = float(lo), float(hi), float(mid)
    nc, nf = max(2, int(n_coarse)), max(2, int(n_fine))
    if fine_lo_half:
        g_a = np.linspace(lo, mid, nf, dtype=np.float64)
        g_b = np.linspace(mid, hi, nc, dtype=np.float64)
    else:
        g_a = np.linspace(lo, mid, nc, dtype=np.float64)
        g_b = np.linspace(mid, hi, nf, dtype=np.float64)
    return np.unique(np.concatenate([g_a, g_b[1:]]))


def ch3_lik_w12_mesh_pack(
    study, exam, y, b, *, w1_lo, w1_hi, w2_lo, w2_hi, grid_n=None, quadrant_fine=False,
):
    if quadrant_fine:
        mid = float(CH3_LIK_W12_QUAD_MID)
        nc = int(CH3_LIK_W12_GRID_N_COARSE)
        nf = int(CH3_LIK_W12_GRID_N_FINE)
        g1 = _ch3_lik_w12_axis_refined(w1_lo, w1_hi, mid, nc, nf, fine_lo_half=False)
        g2 = _ch3_lik_w12_axis_refined(w2_lo, w2_hi, mid, nc, nf, fine_lo_half=True)
    else:
        gn = int(CH3_LIK_W12_GRID_N if grid_n is None else grid_n)
        g1 = np.linspace(float(w1_lo), float(w1_hi), gn, dtype=np.float64)
        g2 = np.linspace(float(w2_lo), float(w2_hi), gn, dtype=np.float64)
    W1m, W2m = np.meshgrid(g1, g2, indexing="ij")
    bf = np.full(W1m.size, float(b), dtype=np.float64)
    Zf = _ch3_likelihood_on_flat_w12_grid(study, exam, y, W1m.ravel(), W2m.ravel(), bf)
    Z = Zf.reshape(W1m.shape)
    return {
        "W1m": W1m, "W2m": W2m, "Z": Z,
        "w1_lo": float(w1_lo), "w1_hi": float(w1_hi),
        "w2_lo": float(w2_lo), "w2_hi": float(w2_hi),
    }


def ch3_figure_lik_w12_3d():
    """Split screen: dataset+knobs (left), tall 3D likelihood ridge (right)."""
    fig = plt.figure(figsize=CH3_LIK_W12_FIGSIZE)
    gs = fig.add_gridspec(
        1, 2, width_ratios=CH3_LIK_W12_WIDTH_RATIOS, wspace=CH3_DUO_WSPACE,
    )
    g_left = GridSpecFromSubplotSpec(
        2, 1, subplot_spec=gs[0, 0], height_ratios=CH3_LEFT_HEIGHT_RATIOS, hspace=CH3_LEFT_HSPACE,
    )
    ax_data = fig.add_subplot(g_left[0,  0])
    g_k = GridSpecFromSubplotSpec(1, 3, subplot_spec=g_left[1, 0], wspace=CH3_KNOB_WSPACE)
    axes_k = tuple(fig.add_subplot(g_k[0, j]) for j in range(3))
    ax3d = fig.add_subplot(gs[0, 1], projection="3d")
    fig.subplots_adjust(left=0.05, right=0.97, top=0.93, bottom=0.06)
    _ch3_align_knob_axes_under_data(fig, ax_data, axes_k)
    ch3_layout_knob_axes_like_bridge_end(fig, ax_data, axes_k)
    from ch4_layout import ch4_duo_plot_layout_tune

    ch4_duo_plot_layout_tune(fig, ax_data, ax3d)
    return fig, ax_data, ax3d, axes_k


def ch4_figure_duo_weight3d():
    """Ch4 duo layout (same 2D/3D width ratios as ch4_02/ch4_03)."""
    return ch3_figure_lik_w12_3d()


def ch3_lik_w12_z_limits(Z, *, scale=1.0):
    z_hi = float(np.nanmax(Z)) * float(scale)
    pad = 0.10 * max(z_hi, 1e-15)
    return 0.0, z_hi + pad


def ch3_lik_w12_facecolors_full(W1m, W2m, reveal_u, *, rgba):
    t = float(np.clip(float(reveal_u), 0.0, 1.0))
    fc = np.empty(W1m.shape + (4,), dtype=float)
    rgba = mpl.colors.to_rgba(rgba)
    fc[..., :] = rgba
    fc[..., 3] = rgba[3] * t
    return fc


def ch3_lik_w12_stack_z_lim(z_lik_hi, *, pad_frac=0.06):
    """Fixed display box: all stacked surfaces compress into [0, z_hi]."""
    z_hi = float(z_lik_hi)
    pad = float(pad_frac) * max(z_hi, 1.0)
    return 0.0, z_hi + pad


def ch3_lik_w12_squish_z(Z, layer_i, n_layers, z_lo, z_hi, z_ref):
    """Map mistake height into layer_i of n_layers equal slots in [z_lo, z_hi]."""
    n = max(float(n_layers), 1.0)
    span = float(z_hi) - float(z_lo)
    slot = span / n
    z_base = float(z_lo) + float(layer_i) * slot
    scale = slot / max(float(z_ref), 1e-9)
    return z_base + np.asarray(Z, dtype=float) * scale


def ch3_lik_w12_squish_scalar(z_val, layer_i, n_layers, z_lo, z_hi, z_ref):
    n = max(float(n_layers), 1.0)
    span = float(z_hi) - float(z_lo)
    slot = span / n
    z_base = float(z_lo) + float(layer_i) * slot
    scale = slot / max(float(z_ref), 1e-9)
    return z_base + float(z_val) * scale


def _ch3_lik_w12_z_at(W1m, W2m, Z, w1, w2):
    d = (np.asarray(W1m, dtype=float) - float(w1)) ** 2 + (np.asarray(W2m, dtype=float) - float(w2)) ** 2
    return float(np.ravel(np.asarray(Z, dtype=float))[int(np.nanargmin(d))])


def ch3_lik_w12_knob3_z_ticks(stack_layers, n_layers, z_lo, z_hi):
    """Tick positions at stacked-layer centers; labels are Knob 3 (b) values."""
    if not stack_layers:
        return None, None
    n = max(float(n_layers), 1.0)
    span = float(z_hi) - float(z_lo)
    slot = span / n
    layers = sorted(
        (
            sl for sl in stack_layers
            if float(sl.get("reveal", 1.0)) > 1e-4 and sl.get("b") is not None
        ),
        key=lambda sl: int(sl.get("layer_i", 0)),
    )
    if not layers:
        return None, None
    tick_z, tick_lbl = [], []
    for sl in layers:
        li = int(sl.get("layer_i", 0))
        tick_z.append(float(z_lo) + (float(li) + 0.5) * slot)
        tick_lbl.append(f"{float(sl['b']):.2g}")
    return tick_z, tick_lbl


def ch3_lik_w12_facecolors_diag(W1m, W2m, reveal_u, *, w1_lo, w1_hi, w2_lo, w2_hi, rgba, origin="lo_lo"):
    """Diagonal surface reveal; ``origin='lo_hi'`` starts at (w1_lo, w2_hi)."""
    u1 = (W1m - float(w1_lo)) / max(float(w1_hi) - float(w1_lo), 1e-9)
    if str(origin) == "lo_hi":
        u2 = (float(w2_hi) - W2m) / max(float(w2_hi) - float(w2_lo), 1e-9)
    else:
        u2 = (W2m - float(w2_lo)) / max(float(w2_hi) - float(w2_lo), 1e-9)
    t = float(np.clip(float(reveal_u), 0.0, 1.0))
    mask = (u1 + u2) <= 2.0 * t + 1e-9
    fc = np.empty(W1m.shape + (4,), dtype=float)
    rgba = mpl.colors.to_rgba(rgba)
    fc[..., :] = rgba
    fc[..., 3] = rgba[3] * mask.astype(float)
    return fc


def ch3_frame_lik_w12_3d(
    study,
    exam,
    y,
    w_st,
    w_el,
    b,
    *,
    mesh_pack,
    z_lim,
    curves,
    elev,
    azim,
    emphasize_knob="st",
    landscape_reveal=0.0,
    landscape_reveal_origin="lo_lo",
    landscape_rgba=None,
    b_slices=None,
    slice_reveal=0.0,
    show_curves=True,
    marker=True,
    marker_z_offset=0.0,
    stack_layers=None,
    stack_n_layers=1.0,
    z_lik_ref=None,
    show_axis_labels=True,
    flat_surface=None,
    z_label=None,
    knob_pack=None,
    knob_scales=None,
    weight_axis_labels=False,
):
    spec = CH3_LOSS_SPECS["likelihood"]
    loss_fn = spec["fn"]
    w_st, w_el, b = float(w_st), float(w_el), float(b)
    W1m, W2m, Z = mesh_pack["W1m"], mesh_pack["W2m"], mesh_pack["Z"]
    w1_lo, w1_hi = mesh_pack["w1_lo"], mesh_pack["w1_hi"]
    w2_lo, w2_hi = mesh_pack["w2_lo"], mesh_pack["w2_hi"]
    z_lo_ax, z_hi_ax = float(z_lim[0]), float(z_lim[1])
    z_ref = float(np.nanmax(Z) if z_lik_ref is None else z_lik_ref)
    n_stack = max(float(stack_n_layers), 1.0)

    fig, ax_data, ax3d, axes_k = ch3_figure_lik_w12_3d()
    leg = legend_linear_equation_values_bold_param(w_st, w_el, b, emphasize_knob)
    ch3_draw_left_panel(
        ax_data, w_st, w_el, b, study, exam, y, leg,
        show_colormap=True, highlight_mistakes_flag=False,
    )
    ax_data.set_xlim(*xlim)
    ax_data.set_ylim(*ylim)
    finalize_style_legend_tex(ax_data)
    if knob_pack is None:
        knob_rgbs, canvas_sides = ch3_knob_asset_pack()
    else:
        knob_rgbs, canvas_sides = knob_pack
    if knob_scales is None:
        scales = ch3_knob_scales_emphasize(emphasize_knob, CH3_KNOB_ACTIVE_SCALE)
    else:
        scales = list(knob_scales)
    ch3_draw_knob_row(
        fig, axes_k, w_st, w_el, b, emphasize_knob,
        knob_rgbs, canvas_sides,
        rot_strip_deg=0.0, strip_scale=1.0,
        knob_rots=ch3_k1_knob_rots_at(w_st, w_el, b),
        knob_scales=scales, ax_data=ax_data,
    )

    ax3d.cla()
    lr = landscape_rgba if landscape_rgba is not None else (FAIL_COLOR, CH3_LIK_W12_SURFACE_ALPHA)
    z_pt = None
    if flat_surface is not None:
        Wb = flat_surface["W1m"]
        W2b = flat_surface["W2m"]
        Zb = np.asarray(flat_surface["Z"], dtype=float)
        if flat_surface.get("nll_heatmap") is not None:
            _ch4_nll_heatmap_plot_surface(
                ax3d, Wb, W2b, Zb, flat_surface["nll_heatmap"],
            )
        elif flat_surface.get("facecolors") is not None:
            fc_b = flat_surface["facecolors"]
            ax3d.plot_surface(
                Wb, W2b, Zb, facecolors=fc_b, shade=False,
                linewidth=0, antialiased=False, rstride=1, cstride=1, zorder=1,
            )
        else:
            fc_b = ch3_lik_w12_facecolors_full(
                Wb, W2b, 1.0,
                rgba=(FAIL_COLOR, CH3_LIK_W12_SURFACE_ALPHA * float(flat_surface.get("alpha_scale", 1.0))),
            )
            ax3d.plot_surface(
                Wb, W2b, Zb, facecolors=fc_b, shade=False,
                linewidth=0, antialiased=False, rstride=1, cstride=1, zorder=1,
            )
        z_pt = float(
            flat_surface["marker_z"]
            if flat_surface.get("marker_z") is not None
            else _ch3_lik_w12_z_at(Wb, W2b, Zb, w_st, w_el)
        )
    elif float(landscape_reveal) > 1e-4:
        fc = ch3_lik_w12_facecolors_diag(
            W1m, W2m, landscape_reveal,
            w1_lo=w1_lo, w1_hi=w1_hi, w2_lo=w2_lo, w2_hi=w2_hi, rgba=lr,
            origin=landscape_reveal_origin,
        )
        ax3d.plot_surface(
            W1m, W2m, Z, facecolors=fc, shade=False,
            linewidth=0, antialiased=False, rstride=1, cstride=1, zorder=1,
        )
    if stack_layers:
        for sl in stack_layers:
            rev = float(sl.get("reveal", 1.0)) * float(slice_reveal)
            if rev < 1e-4:
                continue
            pack_b = sl["pack"]
            Wb, Zb = pack_b["W1m"], pack_b["Z"]
            li = int(sl.get("layer_i", 0))
            Zplot = ch3_lik_w12_squish_z(Zb, li, n_stack, z_lo_ax, z_hi_ax, z_ref)
            fc_b = ch3_lik_w12_facecolors_full(
                Wb, pack_b["W2m"], rev,
                rgba=(FAIL_COLOR, CH3_LIK_W12_SURFACE_ALPHA * float(sl.get("alpha_scale", 1.0))),
            )
            ax3d.plot_surface(
                Wb, pack_b["W2m"], Zplot, facecolors=fc_b, shade=False,
                linewidth=0, antialiased=False, rstride=1, cstride=1, zorder=int(sl.get("zorder", 3)),
            )
    elif b_slices:
        for sl in b_slices:
            bb = float(sl["b"])
            rev = float(sl.get("reveal", 1.0)) * float(slice_reveal)
            if rev < 1e-4:
                continue
            pack_b = sl.get("pack")
            if pack_b is None:
                pack_b = ch3_lik_w12_mesh_pack(
                    study, exam, y, bb, w1_lo=w1_lo, w1_hi=w1_hi, w2_lo=w2_lo, w2_hi=w2_hi,
                )
            Wb, Zb = pack_b["W1m"], pack_b["Z"]
            z0 = float(sl.get("z_base", 0.0))
            fc_b = ch3_lik_w12_facecolors_full(
                Wb, pack_b["W2m"], rev,
                rgba=(FAIL_COLOR, CH3_LIK_W12_SLICE_ALPHA * float(sl.get("alpha_scale", 1.0))),
            )
            ax3d.plot_surface(
                Wb, pack_b["W2m"], Zb + z0, facecolors=fc_b, shade=False,
                linewidth=0, antialiased=False, rstride=1, cstride=1, zorder=3,
            )
    if show_curves and curves:
        for cw1, cw2, cz in curves:
            cw1 = np.asarray(cw1, dtype=float)
            cw2 = np.asarray(cw2, dtype=float)
            cz = np.asarray(cz, dtype=float) + float(marker_z_offset)
            if cw1.size >= 2:
                ax3d.plot(
                    cw1, cw2, cz, color=FAIL_COLOR, linewidth=CH3_LIK_W12_CURVE_LW,
                    alpha=0.95, zorder=8,
                )
    if z_pt is None:
        z_raw = float(loss_fn(w_st, w_el, b, study, exam, y))
        if stack_layers:
            top_i = max(int(sl.get("layer_i", 0)) for sl in stack_layers if float(sl.get("reveal", 0)) > 1e-4)
            z_pt = ch3_lik_w12_squish_scalar(z_raw, top_i, n_stack, z_lo_ax, z_hi_ax, z_ref)
        else:
            z_pt = z_raw + float(marker_z_offset)
    if marker:
        ax3d.scatter(
            [w_st], [w_el], [z_pt],
            color=FAIL_COLOR, edgecolors="white", linewidths=2.0,
            s=260.0, depthshade=False, zorder=20,
        )
    ax3d.set_xlim(w1_lo, w1_hi)
    ax3d.set_ylim(w2_lo, w2_hi)
    ax3d.set_zlim(float(z_lim[0]), float(z_lim[1]))
    knob3_zticks, knob3_zlabels = ch3_lik_w12_knob3_z_ticks(stack_layers, n_stack, z_lo_ax, z_hi_ax)
    if show_axis_labels:
        fs_ax = float(AXIS_LABEL_SIZE) * float(CH3_LIK_3D_AXIS_LABEL_SCALE)
        if weight_axis_labels:
            ax3d.set_xlabel(r"$w_{\mathrm{ST}}$", fontsize=fs_ax, labelpad=10)
            ax3d.set_ylabel(r"$w_{\mathrm{EL}}$", fontsize=fs_ax, labelpad=10)
        else:
            ax3d.set_xlabel("Knob 1", fontsize=AXIS_LABEL_SIZE, labelpad=10)
            ax3d.set_ylabel("Knob 2", fontsize=AXIS_LABEL_SIZE, labelpad=10)
        if knob3_zticks:
            ax3d.set_zlabel("Knob 3", fontsize=AXIS_LABEL_SIZE, labelpad=10)
            ax3d.set_zticks(knob3_zticks)
            ax3d.set_zticklabels(knob3_zlabels)
        elif weight_axis_labels:
            from matplotlib.ticker import MaxNLocator

            ax3d.set_zlabel(str(z_label or r"$b$"), fontsize=fs_ax, labelpad=10)
            ax3d.zaxis.set_major_locator(MaxNLocator(nbins=5))
        else:
            ax3d.set_zlabel(str(z_label or "Likelihood"), fontsize=AXIS_LABEL_SIZE, labelpad=10)
        ax3d.tick_params(axis="both", which="major", labelsize=FONT_SIZE)
        ax3d.grid(True)
    else:
        ax3d.set_xlabel("")
        ax3d.set_ylabel("")
        ax3d.set_zlabel("")
        ax3d.set_xticklabels([])
        ax3d.set_yticklabels([])
        ax3d.set_zticklabels([])
    ax3d.view_init(elev=float(elev), azim=float(azim))
    return fig_to_image(fig, dpi=CH3_ANIM_DPI)


def _ch3_lik_w12_azim_shortest_delta(az0, az1):
    """Signed azimuth delta in (-180, 180] — shortest rotation."""
    return (float(az1) - float(az0) + 180.0) % 360.0 - 180.0


def _ch3_lik_w12_lerp_azim_shortest(az0, az1, u):
    return float(az0) + _ch3_lik_w12_azim_shortest_delta(az0, az1) * float(u)


def ch3_lik_w12_frame_opening(mesh_pack, z_lim, *, w_st=None, w_el=None):
    """Opening frame: empty 3-D at ``(w_st, w_el)`` (defaults: script corner)."""
    w1_c = float(CH3_LIK_W12_CORNER_W1 if w_st is None else w_st)
    w2_c = float(CH3_LIK_W12_CORNER_W2 if w_el is None else w_el)
    b0 = float(CH3_LIK_W12_B0)
    return ch3_frame_lik_w12_3d(
        study_sep, exam_sep, y_sep, w1_c, w2_c, b0,
        mesh_pack=mesh_pack, z_lim=z_lim, curves=[],
        elev=CH3_LIK_W12_ELEV_W1, azim=CH3_LIK_W12_AZIM_W1,
        emphasize_knob="st", show_curves=False, marker=False,
    )


def _ch3_lik_w12_trace_knob1(study, exam, y, w2_fix, b, w1_from, w1_to, n):
    w1s = np.linspace(float(w1_from), float(w1_to), int(n), dtype=float)
    z = [_ch3_likelihood_on_flat_w12_grid(study, exam, y, [w], [w2_fix], [b])[0] for w in w1s]
    return w1s, np.full_like(w1s, float(w2_fix)), np.asarray(z, dtype=float)


def _ch3_lik_w12_trace_knob2(study, exam, y, w1_fix, b, w2_from, w2_to, n):
    w2s = np.linspace(float(w2_from), float(w2_to), int(n), dtype=float)
    z = [_ch3_likelihood_on_flat_w12_grid(study, exam, y, [w1_fix], [w], [b])[0] for w in w2s]
    return np.full_like(w2s, float(w1_fix)), w2s, np.asarray(z, dtype=float)


def _ch4_lik_02_03_handoff_state():
    """Shared end-of-02 / start-of-03 pose: weights (3, -3, 0), ±3 mesh, no marker."""
    study, exam, y = study_sep, exam_sep, y_sep
    lo = float(CH3_LIK_W12_3D_LO)
    hi = float(CH3_LIK_W12_3D_HI)
    b0 = float(CH3_LIK_W12_B0)
    mesh = ch3_lik_w12_mesh_pack(
        study, exam, y, b0,
        w1_lo=lo, w1_hi=hi, w2_lo=lo, w2_hi=hi,
        quadrant_fine=True,
    )
    return {
        "study": study, "exam": exam, "y": y,
        "w_st": hi, "w_el": lo, "b": b0,
        "mesh": mesh, "z_lim": ch3_lik_w12_z_limits(mesh["Z"], scale=1.0),
    }


def _ch4_lik_02_03_handoff_plot(*, knob_labeled_blend=None, emphasize_knob="st"):
    """Full likelihood landscape at CT view — matches 02 end and 03 opening."""
    st = _ch4_lik_02_03_handoff_state()
    lo = float(CH3_LIK_W12_3D_LO)
    hi = float(CH3_LIK_W12_3D_HI)
    knob_pack = None
    if knob_labeled_blend is not None:
        from ch4_layout import ch4_knob_asset_pack, ch4_knob_asset_pack_blended

        knob_pack = ch4_knob_asset_pack_blended(
            ch3_knob_asset_pack(),
            ch4_knob_asset_pack(),
            knob_labeled_blend,
        )
    return ch3_frame_lik_w12_3d(
        st["study"], st["exam"], st["y"], st["w_st"], st["w_el"], st["b"],
        mesh_pack=st["mesh"], z_lim=st["z_lim"], curves=[],
        elev=float(CH3_LIK_W12_CT_ELEV), azim=float(CH3_LIK_W12_CT_AZIM),
        emphasize_knob=emphasize_knob,
        landscape_reveal=1.0, landscape_reveal_origin="lo_hi",
        show_curves=False, marker=False,
        knob_pack=knob_pack,
        knob_scales=[1.0, 1.0, 1.0],
    )


def _ch4_02_compose_plot(plot_img):
    """Same shell as ch4_03 opening (layout_u=0, full-width plot, no rails)."""
    return _ch4_lik_03_opening_compose(plot_img, layout_u=0.0)


def ch3_build_frames_likelihood_w12_landscape_story():
    study, exam, y = study_sep, exam_sep, y_sep
    b0 = float(CH3_LIK_W12_B0)
    lo = float(CH3_LIK_W12_3D_LO)
    hi = float(CH3_LIK_W12_3D_HI)
    w1s, w2s = hi, lo  # start (knob1, knob2) = (3, -3)
    el_ct = float(CH3_LIK_W12_CT_ELEV)
    az_ct = float(CH3_LIK_W12_CT_AZIM)
    el_w1 = float(CH3_LIK_W12_ELEV_W1)
    az_w1 = float(CH3_LIK_W12_AZIM_W1)
    el_w2 = float(CH3_LIK_W12_ELEV_W2)
    az_w2 = float(CH3_LIK_W12_AZIM_W2)

    mesh0 = ch3_lik_w12_mesh_pack(
        study, exam, y, b0,
        w1_lo=lo, w1_hi=hi,
        w2_lo=lo, w2_hi=hi,
        quadrant_fine=True,
    )
    z_lik_hi = float(np.nanmax(mesh0["Z"]))
    z_lim_full = ch3_lik_w12_z_limits(mesh0["Z"], scale=1.0)
    n_trace = max(24, CH3_LIK_W12_N_KNOB)

    frames = []
    curves = []

    def emit(
        ws, we, bb, *, elev, azim, emp="st", lrev=0.0, show_curves=True,
        z_lim=None, marker=True, srev=0.0, marker_z_offset=0.0, stack_layers=None,
        stack_n_layers=1.0, z_lik_ref=None, landscape_reveal_origin="lo_lo",
    ):
        fr = ch3_frame_lik_w12_3d(
            study, exam, y, ws, we, bb,
            mesh_pack=mesh0, z_lim=z_lim or z_lim_full,
            curves=list(curves) if show_curves else [],
            elev=elev, azim=azim, emphasize_knob=emp,
            landscape_reveal=lrev, landscape_reveal_origin=landscape_reveal_origin,
            slice_reveal=srev,
            show_curves=show_curves, marker=marker,
            marker_z_offset=marker_z_offset, stack_layers=stack_layers,
            stack_n_layers=stack_n_layers,
            z_lik_ref=z_lik_hi if z_lik_ref is None else z_lik_ref,
        )
        frames.append(_ch4_02_compose_plot(fr))

    def hold(ws, we, bb, n, **kw):
        for _ in range(int(n)):
            emit(ws, we, bb, **kw)

    # 1–2: empty 3D at (3, -3)
    open_fr = ch3_lik_w12_frame_opening(mesh0, z_lim_full, w_st=w1s, w_el=w2s)
    open_comp = _ch4_02_compose_plot(open_fr)
    for _ in range(CH3_LIK_W12_N_HOLD):
        frames.append(open_comp.copy())

    # 3: knob 1 — w1: 3 → -3 (w2 = -3)
    for tv in np.linspace(0.0, 1.0, CH3_LIK_W12_N_KNOB, endpoint=True):
        u = ch3_knob_smoothstep(float(tv))
        w1c = ch3_lerp(hi, lo, u)
        cw1, cw2, cz = _ch3_lik_w12_trace_knob1(study, exam, y, w2s, b0, hi, w1c, n_trace)
        if curves:
            curves[-1] = (cw1, cw2, cz)
        else:
            curves.append((cw1, cw2, cz))
        emit(w1c, w2s, b0, elev=el_w1, azim=az_w1, emp="st")
    hold(lo, w2s, b0, CH3_LIK_W12_N_HOLD // 2, elev=el_w1, azim=az_w1)
    curves[-1] = _ch3_lik_w12_trace_knob1(study, exam, y, w2s, b0, hi, lo, n_trace)

    # 3b: knob 1 — w1: -3 → 3
    for tv in np.linspace(0.0, 1.0, CH3_LIK_W12_N_KNOB, endpoint=True):
        u = ch3_knob_smoothstep(float(tv))
        w1c = ch3_lerp(lo, hi, u)
        emit(w1c, w2s, b0, elev=el_w1, azim=az_w1, emp="st")
    hold(w1s, w2s, b0, CH3_LIK_W12_N_HOLD // 2, elev=el_w1, azim=az_w1)

    # 4: shortest rotation to knob-2 view
    for tv in np.linspace(0.0, 1.0, CH3_LIK_W12_N_ROT, endpoint=True):
        u = ch3_knob_smoothstep(float(tv))
        emit(
            w1s, w2s, b0,
            elev=ch3_lerp(el_w1, el_w2, u),
            azim=_ch3_lik_w12_lerp_azim_shortest(az_w1, az_w2, u),
        )

    # 5: knob 2 — w2: -3 → 3 (w1 = 3)
    curves.append(_ch3_lik_w12_trace_knob2(study, exam, y, w1s, b0, w2s, w2s, 2))
    for tv in np.linspace(0.0, 1.0, CH3_LIK_W12_N_KNOB, endpoint=True):
        u = ch3_knob_smoothstep(float(tv))
        w2c = ch3_lerp(w2s, hi, u)
        cw1, cw2, cz = _ch3_lik_w12_trace_knob2(study, exam, y, w1s, b0, w2s, w2c, n_trace)
        curves[-1] = (cw1, cw2, cz)
        emit(w1s, w2c, b0, elev=el_w2, azim=az_w2, emp="el")
    hold(w1s, hi, b0, CH3_LIK_W12_N_HOLD // 2, elev=el_w2, azim=az_w2)

    # 5b: knob 2 — w2: 3 → -3
    for tv in np.linspace(0.0, 1.0, CH3_LIK_W12_N_KNOB, endpoint=True):
        u = ch3_knob_smoothstep(float(tv))
        w2c = ch3_lerp(hi, w2s, u)
        emit(w1s, w2c, b0, elev=el_w2, azim=az_w2, emp="el")
    hold(w1s, w2s, b0, CH3_LIK_W12_N_HOLD // 2, elev=el_w2, azim=az_w2)

    # 6: shortest rotation to ch4_03 CT view — keep for fill + landscape
    for tv in np.linspace(0.0, 1.0, CH3_LIK_W12_N_ROT, endpoint=True):
        u = ch3_knob_smoothstep(float(tv))
        emit(
            w1s, w2s, b0,
            elev=ch3_lerp(el_w2, el_ct, u),
            azim=_ch3_lik_w12_lerp_azim_shortest(az_w2, az_ct, u),
        )

    # 7: raster fill — evenly spaced curves (surface mesh stays finer in one quadrant)
    w1_vals = np.linspace(lo, hi, int(CH3_LIK_W12_N_FILL_LINES), dtype=np.float64)
    w2_vals = np.linspace(lo, hi, int(CH3_LIK_W12_N_FILL_LINES), dtype=np.float64)
    n_trace_fill = int(CH3_LIK_W12_N_FILL_TRACE)
    for w1v in w1_vals:
        cw1, cw2, cz = _ch3_lik_w12_trace_knob2(
            study, exam, y, float(w1v), b0, lo, hi, n_trace_fill,
        )
        curves.append((cw1, cw2, cz))
        emit(w1s, w2s, b0, elev=el_ct, azim=az_ct, emp="st", marker=True)
    for w2v in w2_vals:
        cw1, cw2, cz = _ch3_lik_w12_trace_knob1(
            study, exam, y, float(w2v), b0, lo, hi, n_trace_fill,
        )
        curves.append((cw1, cw2, cz))
        emit(w1s, w2s, b0, elev=el_ct, azim=az_ct, emp="el", marker=True)

    hold(w1s, w2s, b0, CH3_LIK_W12_N_HOLD // 2, elev=el_ct, azim=az_ct, marker=True)

    # 8: diagonal landscape reveal from (-3, 3); then drop marker
    for tv in np.linspace(0.0, 1.0, CH3_LIK_W12_N_REVEAL, endpoint=True):
        u = ch3_knob_smoothstep(float(tv))
        emit(
            w1s, w2s, b0, elev=el_ct, azim=az_ct,
            lrev=u, show_curves=True, marker=False,
            landscape_reveal_origin="lo_hi",
        )
    emit(
        w1s, w2s, b0, elev=el_ct, azim=az_ct, lrev=1.0,
        show_curves=False, marker=False, landscape_reveal_origin="lo_hi",
    )
    handoff_comp = _ch4_02_compose_plot(_ch4_lik_02_03_handoff_plot())
    for _ in range(CH3_LIK_W12_N_HOLD):
        frames.append(handoff_comp.copy())
    return frames


def ch4_preview_likelihood_w12_landscape_last_frame():
    return _ch4_02_compose_plot(_ch4_lik_02_03_handoff_plot())


def ch4_export_likelihood_w12_landscape():
    frames = ch3_build_frames_likelihood_w12_landscape_story()
    fn = "ch4_02_likelihood_w12_landscape.mp4"
    save_mp4(frames, fn, duration=int(CH3_LIK_W12_MS))
    print("wrote", OUTPUT_DIR / fn)
    return OUTPUT_DIR / fn


# --- ch4_03/04: likelihood ch4 notation/NLL morph → 3D measurements + trajectory ---

CH3_LIK_CH4_MS = 90 if not _CH3_DRAFT else 110
CH3_LIK_CH4_W_LO = -3.0
CH3_LIK_CH4_W_HI = 3.0
CH3_LIK_CH4_PLANE_B_MID = 0.0
CH3_LIK_CH4_PLANE_B = -3.0
CH3_LIK_CH4_SURFACE_ALPHA = 0.6
CH3_LIK_CH4_CT_ELEV = 24.0
CH3_LIK_CH4_CT_AZIM = -128.0
# Shared CT slice mesh (matches ch4_05a axis sweeps + voxel cut planes).
CH3_LIK_CT_GRID = 18 if _CH3_DRAFT else 32
CH3_LIK_CT_PLANE_ALPHA = 0.5
CH3_LIK_CT_VIEW_BOUNDS = (-3.0, 3.0, -3.0, 3.0, -3.0, 3.0)
CH3_LIK_3D_CAM_AZIM0 = CH3_LIK_CH4_CT_AZIM
CH3_LIK_CH4_N_HEATMAP_REVEAL = 8 if _CH3_DRAFT else max(36, _smooth_n(28))
CH3_LIK_CH4_N_SQUISH_PLANE = 8 if _CH3_DRAFT else max(32, _smooth_n(24))
CH3_LIK_CH4_N_PLANE_DROP = 8 if _CH3_DRAFT else max(32, _smooth_n(24))
CH3_LIK_3D_MS = 90 if not _CH3_DRAFT else 110
CH3_LIK_CH4_N_MORPH = 8 if _CH3_DRAFT else max(28, _smooth_n(22))
CH3_LIK_CH4_N_LOG = 8 if _CH3_DRAFT else max(32, _smooth_n(24))
CH3_LIK_CH4_N_NLL = 8 if _CH3_DRAFT else max(32, _smooth_n(24))
CH3_LIK_CH4_N_NOTATION_MOVE = 8 if _CH3_DRAFT else max(24, _smooth_n(18))
CH3_LIK_CH4_N_CORNER_WRITE = 8 if _CH3_DRAFT else max(32, _smooth_n(24))
CH3_LIK_CH4_N_KNOB_SWAP = 8 if _CH3_DRAFT else max(20, _smooth_n(14))
CH3_LIK_CH4_N_PROB_WRITE = 8 if _CH3_DRAFT else max(40, _smooth_n(32))
CH3_LIK_3D_N_RIGHT_TITLE = 6 if _CH3_DRAFT else max(14, _smooth_n(10))
CH3_LIK_3D_N_RIGHT_WRITE = 8 if _CH3_DRAFT else max(36, _smooth_n(28))
CH3_LIK_3D_N_TRANS = 10 if _CH3_DRAFT else max(36, _smooth_n(28))
CH3_LIK_3D_N_PATH = 24 if _CH3_DRAFT else max(120, _smooth_n(90))
CH3_LIK_3D_N_COLOR = 8 if _CH3_DRAFT else max(40, _smooth_n(32))
CH3_LIK_3D_CAM_PATH_ROT = 90.0
CH3_LIK_3D_N_INTRO_ZOOM = 6 if _CH3_DRAFT else max(22, _smooth_n(16))
CH3_LIK_3D_N_INTRO_SPIN = 10 if _CH3_DRAFT else max(64, _smooth_n(48))
CH3_LIK_3D_N_INTRO_ZOOM_OUT = 6 if _CH3_DRAFT else max(22, _smooth_n(16))
CH3_LIK_3D_N_05_HANDOFF = 8 if _CH3_DRAFT else max(36, _smooth_n(28))
CH3_LIK_3D_BALL_NSAMPLE = 16 if _CH3_DRAFT else 40
CH3_LIK_3D_BALL_R_SCALE = 0.085
CH3_LIK_3D_BALL_QUIVER_LEN = 0.075
CH3_LIK_3D_MS_BALL = 110 if not _CH3_DRAFT else 130
CH3_LIK_3D_AXIS_LABEL_SCALE = 1.45
CH3_LIK_3D_POINT_COLOR = FAIL_COLOR
CH3_LIK_3D_PATH_START = (-0.5, 0.33, -0.5)
CH3_LIK_GD_STEP = 0.06
CH3_LIK_GD_N_ITERS = 10 if not _CH3_DRAFT else 3
CH3_LIK_GD_N_HOLD_ARROWS = 4 if _CH3_DRAFT else 12
CH3_LIK_GD_N_PARAM_STEP = 4 if _CH3_DRAFT else 10
CH3_LIK_GD_N_COMBINE = 4 if _CH3_DRAFT else 16
CH3_LIK_GD_SUBSTEPS = 6 if _CH3_DRAFT else 12
CH3_LIK_3D_MS_GD = 120 if not _CH3_DRAFT else 140

# ch4_02 bookends + ch4_03 step 1 share layout_u=0; ch4_03 morphs the plot into the slot.
CH4_LIK_PLOT_START_RECT = (0.0, 0.0, 1.0, 1.0)


def _ch3_lik_w12_z_limits_signed(Z, *, pad_frac=0.10):
    z_lo = float(np.nanmin(Z))
    z_hi = float(np.nanmax(Z))
    span = max(z_hi - z_lo, 1e-9)
    pad = float(pad_frac) * span
    return z_lo - pad, z_hi + pad


def _ch3_lik_w12_z_morph_limits(Zlik, Zlog, Znll, log_u, nll_u):
    """Blend z-axis limits: ℒ (0…max) → log ℒ (min…max) → NLL (0…max)."""
    mu_log = float(np.clip(log_u, 0.0, 1.0))
    mu_nll = float(np.clip(nll_u, 0.0, 1.0))
    lim_lik = ch3_lik_w12_z_limits(Zlik)
    lim_log = _ch3_lik_w12_z_limits_signed(Zlog)
    lim_nll = ch3_lik_w12_z_limits(Znll)
    if mu_nll > 1e-9:
        lim_a, lim_b, mu = lim_log, lim_nll, mu_nll
    elif mu_log > 1e-9:
        lim_a, lim_b, mu = lim_lik, lim_log, mu_log
    else:
        return lim_lik
    return (
        (1.0 - mu) * lim_a[0] + mu * lim_b[0],
        (1.0 - mu) * lim_a[1] + mu * lim_b[1],
    )


def _ch3_lik_w12_z_morph_surface(Zlik, Zlog, Znll, log_u, nll_u, z_lim):
    """Morph surface height in normalized z, with axis limits lerped separately."""
    mu_log = float(np.clip(log_u, 0.0, 1.0))
    mu_nll = float(np.clip(nll_u, 0.0, 1.0))
    lim_lik = ch3_lik_w12_z_limits(Zlik)
    lim_log = _ch3_lik_w12_z_limits_signed(Zlog)
    lim_nll = ch3_lik_w12_z_limits(Znll)
    zlo, zhi = float(z_lim[0]), float(z_lim[1])

    def _norm(Z, lo, hi):
        return (np.asarray(Z, dtype=float) - float(lo)) / max(float(hi) - float(lo), 1e-9)

    def _denorm(t):
        return t * (zhi - zlo) + zlo

    if mu_nll >= 1.0 - 1e-9:
        return np.asarray(Znll, dtype=float)
    if mu_log >= 1.0 - 1e-9 and mu_nll <= 1e-9:
        return np.asarray(Zlog, dtype=float)
    if mu_log <= 1e-9 and mu_nll <= 1e-9:
        return np.asarray(Zlik, dtype=float)
    if mu_nll > 1e-9:
        t = (1.0 - mu_nll) * _norm(Zlog, *lim_log) + mu_nll * _norm(Znll, *lim_nll)
    elif mu_log > 1e-9:
        t = (1.0 - mu_log) * _norm(Zlik, *lim_lik) + mu_log * _norm(Zlog, *lim_log)
    else:
        t = _norm(Zlik, *lim_lik)
    return _denorm(t)


def _ch3_lik86_terminal_state():
    """Ch4_03 story state — handoff from ch4_02: (w1, w2, b) = (3, -3, 0)."""
    st = _ch4_lik_02_03_handoff_state()
    return {
        **st,
        "w1_lo": float(CH3_LIK_W12_W1_LO),
        "w1_hi": float(CH3_LIK_W12_W1_HI),
        "w2_lo": float(CH3_LIK_W12_W2_LO),
        "w2_hi": float(CH3_LIK_W12_W2_HI),
    }


def _ch3_lik_w12_loglik_nll_grids(study, exam, y, w1m, w2m, b):
    """Σ log p and Σ −log p on a mesh — avoid ``log(likelihood product)`` underflow."""
    w1f = np.asarray(w1m, dtype=np.float64).ravel()
    w2f = np.asarray(w2m, dtype=np.float64).ravel()
    bf = np.full(w1f.size, float(b), dtype=np.float64)
    sh = np.asarray(w1m, dtype=np.float64).shape
    zlog = _ch3_loglik_on_flat_w12_grid(study, exam, y, w1f, w2f, bf).reshape(sh)
    znll = _ch3_nll_sum_on_flat_grid(study, exam, y, w1f, w2f, bf).reshape(sh)
    return zlog, znll


def _ch3_lik_ch4_plane_mesh(state, *, b=None):
    """±3 (w_ST, w_EL) grid at fixed ``b`` — same NLL grid as ch4_05a ``b`` slices."""
    study, exam, y = state["study"], state["exam"], state["y"]
    bb = float(CH3_LIK_CH4_PLANE_B if b is None else b)
    mesh = ch3_lik_w12_mesh_pack(
        study, exam, y, bb,
        w1_lo=float(CH3_LIK_CH4_W_LO), w1_hi=float(CH3_LIK_CH4_W_HI),
        w2_lo=float(CH3_LIK_CH4_W_LO), w2_hi=float(CH3_LIK_CH4_W_HI),
        grid_n=int(CH3_LIK_CT_GRID),
    )
    W1m, W2m = mesh["W1m"], mesh["W2m"]
    _, nll = _ch3_lik_w12_loglik_nll_grids(study, exam, y, W1m, W2m, bb)
    mesh["nll"] = nll
    return mesh


def ch4_lik_ct_view_init(ax3d, *, cam_azim_u=0.0, cam_spin_deg=0.0):
    """Canonical CT camera — ch4_03 plane-drop end pose when idle."""
    u = float(np.clip(float(cam_azim_u), 0.0, 1.0))
    spin = float(cam_spin_deg)
    if abs(spin) > 1e-9:
        azim = _ch3_lik_cam_azim(u, total_deg=spin)
    elif abs(u) > 1e-9:
        azim = _ch3_lik_cam_azim(u, total_deg=0.0)
    else:
        azim = float(CH3_LIK_CH4_CT_AZIM)
    ax3d.view_init(elev=float(CH3_LIK_CH4_CT_ELEV), azim=float(azim))


def _ch4_nll_heatmap_plot_surface(ax3d, w1, w2, z, nll, *, alpha=None, zorder=6):
    """Shared NLL heatmap surface coloring — matches ch4_05a CT (canonical)."""
    from ch4_layout import ch4_nll_heatmap_cmap

    al = float(CH3_LIK_CT_PLANE_ALPHA if alpha is None else alpha)
    lo, hi = ch4_nll_global_scale()
    cmap = ch4_nll_heatmap_cmap()
    span = max(float(hi) - float(lo), 1e-9)
    arr = np.asarray(nll, dtype=float)
    normed = np.clip((arr - float(lo)) / span, 0.0, 1.0)
    face = cmap(normed)
    ax3d.plot_surface(
        w1, w2, z,
        facecolors=face,
        rstride=1,
        cstride=1,
        linewidth=0,
        antialiased=False,
        shade=False,
        alpha=float(al),
        zorder=int(zorder),
    )


def _ch3_lik_w12_plot_crop(img):
    """Crop the right 3D panel from a split-screen ch3_86 frame."""
    img = img.convert("RGB")
    w, h = img.size
    x0 = int(round(w * 0.405))
    return img.crop((x0, int(h * 0.04), w - int(w * 0.02), h - int(h * 0.05)))


def ch3_frame_lik_w12_single_surface(
    state,
    *,
    log_u=0.0,
    nll_u=0.0,
    heatmap_u=0.0,
    squish_u=0.0,
    plane_drop_u=0.0,
    keep_layers=1,
    elev=None,
    azim=None,
    show_axis_labels=True,
    knob_labeled_blend=None,
):
    """One NLL surface; 3-D axes ±3, fixed CT camera (z ticks left), 2-D knobs unchanged."""
    study, exam, y = state["study"], state["exam"], state["y"]
    ws, we, bb = state["w_st"], state["w_el"], state["b"]
    mesh = state["mesh"]
    W1m, W2m, Z = mesh["W1m"], mesh["W2m"], mesh["Z"]
    Zlik = np.asarray(Z, dtype=float)
    mu_log = float(np.clip(log_u, 0.0, 1.0))
    mu_nll = float(np.clip(nll_u, 0.0, 1.0))
    su = ch3_knob_smoothstep(float(np.clip(float(squish_u), 0.0, 1.0)))
    pu = ch3_knob_smoothstep(float(np.clip(float(plane_drop_u), 0.0, 1.0)))
    b_mid = float(CH3_LIK_CH4_PLANE_B_MID)
    b_end = float(CH3_LIK_CH4_PLANE_B)
    z_flat = (1.0 - pu) * b_mid + pu * b_end
    if su > 1e-6:
        plane = _ch3_lik_ch4_plane_mesh(state, b=z_flat)
        W1m, W2m = plane["W1m"], plane["W2m"]
        Zlik_p = np.asarray(plane["Z"], dtype=float)
        Zlog_p, Znll = _ch3_lik_w12_loglik_nll_grids(study, exam, y, W1m, W2m, z_flat)
        z_lim_nll = _ch3_lik_w12_z_morph_limits(Zlik_p, Zlog_p, Znll, mu_log, mu_nll)
        Zmix_p = _ch3_lik_w12_z_morph_surface(Zlik_p, Zlog_p, Znll, mu_log, mu_nll, z_lim_nll)
        Zplot = (1.0 - su) * Zmix_p + su * np.full_like(Zmix_p, float(z_flat))
    else:
        Zlog, Znll = _ch3_lik_w12_loglik_nll_grids(study, exam, y, W1m, W2m, bb)
        z_lim_nll = _ch3_lik_w12_z_morph_limits(Zlik, Zlog, Znll, mu_log, mu_nll)
        Zmix = _ch3_lik_w12_z_morph_surface(Zlik, Zlog, Znll, mu_log, mu_nll, z_lim_nll)
        Zplot = np.asarray(Zmix, dtype=float)
    z_lo_b, z_hi_b = float(CH3_LIK_CH4_W_LO), float(CH3_LIK_CH4_W_HI)
    z_lim = (
        (1.0 - su) * float(z_lim_nll[0]) + su * z_lo_b,
        (1.0 - su) * float(z_lim_nll[1]) + su * z_hi_b,
    )
    if su >= 0.5:
        z_lab = r"$b$"
    elif mu_nll > 0.5:
        z_lab = "NLL"
    elif mu_log > 0.5:
        z_lab = "log likelihood"
    else:
        z_lab = "Likelihood"
    hu = ch3_knob_smoothstep(float(np.clip(float(heatmap_u), 0.0, 1.0)))
    surf_alpha = float(
        CH3_LIK_CT_PLANE_ALPHA if su > 1e-6 else CH3_LIK_CH4_SURFACE_ALPHA
    )
    rgba_red = mpl.colors.to_rgba(FAIL_COLOR)
    fc_red = np.empty(W1m.shape + (4,), dtype=float)
    fc_red[..., :3] = rgba_red[:3]
    fc_red[..., 3] = surf_alpha
    if hu > 1e-6:
        from ch4_layout import ch4_nll_heatmap_facecolors

        g_lo, g_hi = ch4_nll_global_scale()
        fc_heat = ch4_nll_heatmap_facecolors(
            Znll, vmin=g_lo, vmax=g_hi, alpha=surf_alpha,
        )
        fc = fc_red * (1.0 - hu) + fc_heat * hu
    else:
        fc = fc_red
    el = float(CH3_LIK_CH4_CT_ELEV if elev is None else elev)
    az = float(CH3_LIK_CH4_CT_AZIM if azim is None else azim)
    knob_pack = None
    if knob_labeled_blend is not None:
        from ch4_layout import ch4_knob_asset_pack, ch4_knob_asset_pack_blended

        knob_pack = ch4_knob_asset_pack_blended(
            ch3_knob_asset_pack(),
            ch4_knob_asset_pack(),
            knob_labeled_blend,
        )
    use_weight_axis_labels = (
        knob_labeled_blend is not None
        and all(float(x) >= 1.0 - 1e-6 for x in knob_labeled_blend)
    )
    b_show = float(z_flat) if su > 1e-4 else float(bb)
    if su > 1e-6 and hu > 1.0 - 1e-6:
        flat_surface = {"W1m": W1m, "W2m": W2m, "Z": Zplot, "nll_heatmap": Znll}
    else:
        flat_surface = {"W1m": W1m, "W2m": W2m, "Z": Zplot, "facecolors": fc}
    return ch3_frame_lik_w12_3d(
        study, exam, y, ws, we, b_show,
        mesh_pack=mesh, z_lim=z_lim, curves=[],
        elev=el, azim=az, emphasize_knob="all",
        landscape_reveal=0.0, show_curves=False, marker=False,
        stack_layers=None, stack_n_layers=1.0, slice_reveal=1.0,
        show_axis_labels=show_axis_labels,
        flat_surface=flat_surface,
        z_label=z_lab,
        knob_pack=knob_pack,
        knob_scales=[1.0, 1.0, 1.0],
        weight_axis_labels=use_weight_axis_labels,
    )


def _ch4_lik_03_opening_plot(*, knob_labeled_blend=None):
    """Plot panel for ch4_03 frame 0 — same pose/camera as ch4_02 end."""
    return _ch4_lik_02_03_handoff_plot(knob_labeled_blend=knob_labeled_blend)


def _ch4_lik_03_opening_compose(plot_img, *, layout_u=0.0):
    """Compose wrapper for ch4_03 opening / ch4_02 bookends (no rails yet)."""
    from ch4_layout import compose_tutorial

    return compose_tutorial(
        plot_img,
        right_blocks=[],
        bottom_blocks=[],
        layout_u=float(layout_u),
        panel_u=0.0,
        title_write_progress=0.0,
        write_progress=0.0,
        plot_start_rect=CH4_LIK_PLOT_START_RECT,
        theme="classic_light",
    )


def _ch3_lik_stage_tex(stages, frame_i, n_frames):
    """Pick one of ``stages`` for frame index (hard cut, no crossfade)."""
    n = max(int(n_frames), 1)
    idx = min(int(frame_i * len(stages) / n), len(stages) - 1)
    return stages[idx]


def _ch3_lik_emit_ch4_03_formulas(
    plot_img,
    *,
    right_blocks,
    bottom_prog,
    bottom_blocks=None,
    right_title="Notation",
    bottom_title="Formulas",
    corner_blocks=None,
    corner_title=None,
    right_blocks_empty=False,
):
    """Compose ch4_03 with three-column handwritten formulas."""
    from ch4_layout import (
        CH4_FORMULAS_SECTION_TITLE,
        CH4_LIK_PLOT_START_RECT,
        CH4_NOTATION_SECTION_TITLE,
        ch4_formula_blocks_ch4_03_three_col,
        compose_tutorial,
    )

    blocks = bottom_blocks if bottom_blocks is not None else ch4_formula_blocks_ch4_03_three_col()
    kw = dict(
        plot_img=plot_img,
        right_blocks=[] if right_blocks_empty else right_blocks,
        bottom_blocks=blocks,
        layout_u=1.0,
        panel_u=1.0,
        write_progress=1.0,
        plot_start_rect=CH4_LIK_PLOT_START_RECT,
        progress_override={"bottom": bottom_prog},
        theme="classic_light",
    )
    if corner_blocks is not None:
        kw.update(
            corner_blocks=corner_blocks,
            bottom_title=bottom_title or CH4_FORMULAS_SECTION_TITLE,
            corner_title=corner_title or CH4_NOTATION_SECTION_TITLE,
        )
    else:
        kw.update(right_title=right_title, bottom_title=bottom_title)
    return compose_tutorial(**kw)


def ch3_build_frames_likelihood_ch4_nll_story():
    from ch4_layout import (
        CH4_COMPOSER,
        CH4_FORMULAS_SECTION_TITLE,
        CH4_LIK_PLOT_START_RECT,
        CH4_NOTATION_SECTION_TITLE,
        ch4_blend_images,
        ch4_bottom_prog_ch4_03_three_col,
        ch4_cached_notation_corner_blocks,
        ch4_formula_blocks_3d_story,
        ch4_formula_blocks_ch4_03_three_col,
        ch4_formula_blocks_nll_story,
        ch4_blocks_write_from_slot,
        ch4_bottom_per_block_progress,
        ch4_group_write_from_slot,
        ch4_notation_blocks_basic,
        ch4_notation_blocks_expanded,
        compose_tutorial,
    )

    state = _ch3_lik86_terminal_state()
    frames = []
    style = CH4_COMPOSER.handwrite_style()
    bottom_3col = ch4_formula_blocks_ch4_03_three_col()

    def plot_surface(
        *,
        log_u=0.0,
        nll_u=0.0,
        heatmap_u=0.0,
        squish_u=0.0,
        plane_drop_u=0.0,
        knob_labeled_blend=None,
    ):
        return ch3_frame_lik_w12_single_surface(
            state,
            log_u=log_u,
            nll_u=nll_u,
            heatmap_u=heatmap_u,
            squish_u=squish_u,
            plane_drop_u=plane_drop_u,
            show_axis_labels=True,
            knob_labeled_blend=knob_labeled_blend,
        )

    def emit(
        plot_img,
        *,
        layout_u=1.0,
        panel_u=1.0,
        title_write_progress=None,
        write_progress=0.0,
        right_blocks=None,
        bottom_blocks=None,
        right_write_progress=None,
        bottom_write_progress=None,
        progress_override=None,
    ):
        frames.append(
            compose_tutorial(
                plot_img,
                right_blocks=right_blocks if right_blocks is not None else ch4_notation_blocks_basic(),
                bottom_blocks=bottom_blocks if bottom_blocks is not None else bottom_3col,
                right_title="Notation",
                bottom_title="Formulas",
                layout_u=layout_u,
                panel_u=panel_u,
                title_write_progress=title_write_progress,
                write_progress=write_progress,
                plot_start_rect=CH4_LIK_PLOT_START_RECT,
                right_write_progress=right_write_progress,
                bottom_write_progress=bottom_write_progress,
                progress_override=progress_override,
                theme="classic_light",
            )
        )

    plot0 = _ch4_lik_03_opening_plot()
    n_titles = max(14, _smooth_n(12))
    n_write = max(40, _smooth_n(32))
    n_expand = max(24, _smooth_n(18))
    basic_slots = 6  # weights (3 lines) + yi/xi (3 lines) on right rail

    # 1 — full ch4_02 figure resizes into the template plot slot (no rails yet)
    for tv in np.linspace(0.0, 1.0, CH3_LIK_CH4_N_MORPH, endpoint=True):
        u = ch3_knob_smoothstep(float(tv))
        emit(plot0, layout_u=u, panel_u=0.0, title_write_progress=0.0, write_progress=0.0)

    # 2 — section titles appear and stay (rails visible, no block text yet)
    for tv in np.linspace(0.0, 1.0, n_titles, endpoint=True):
        u = ch3_knob_smoothstep(float(tv))
        emit(plot0, layout_u=1.0, panel_u=1.0, title_write_progress=u, write_progress=0.0)

    # 3 — handwrite basic notation + column-1 likelihood
    for tv in np.linspace(0.0, 1.0, n_write, endpoint=True):
        u = ch3_knob_smoothstep(float(tv))
        emit(
            plot0,
            layout_u=1.0,
            panel_u=1.0,
            title_write_progress=1.0,
            write_progress=1.0,
            right_blocks=ch4_notation_blocks_basic(),
            progress_override={
                "bottom": ch4_bottom_prog_ch4_03_three_col(lik_u=u),
            },
        )

    # 4 — add expanded notation (y_i, x_i); keep ℒ column written
    right_exp = ch4_notation_blocks_expanded()
    for tv in np.linspace(0.0, 1.0, n_expand, endpoint=True):
        u = ch3_knob_smoothstep(float(tv))
        prog = ch4_group_write_from_slot([right_exp], basic_slots, u, style=style)
        emit(
            plot0,
            layout_u=1.0,
            panel_u=1.0,
            title_write_progress=1.0,
            write_progress=1.0,
            right_blocks=right_exp,
            progress_override={
                "right": prog[0],
                "bottom": ch4_bottom_prog_ch4_03_three_col(lik_u=1.0),
            },
        )

    # 5 — surface → log; handwrite log ℒ column (single line, two-phase reveal)
    n_log = CH3_LIK_CH4_N_LOG
    for i, tv in enumerate(np.linspace(0.0, 1.0, n_log, endpoint=True)):
        u = ch3_knob_smoothstep(float(tv))
        plot_u = plot_surface(log_u=u, nll_u=0.0)
        if u < 0.5:
            log_us = (u * 2.0, 0.0)
        else:
            log_us = (1.0, (u - 0.5) * 2.0)
        frames.append(_ch3_lik_emit_ch4_03_formulas(
            plot_u,
            right_blocks=right_exp,
            bottom_prog=ch4_bottom_prog_ch4_03_three_col(lik_u=1.0, log_line_us=log_us, nll_u=0.0),
        ))

    # 6 — surface → NLL; handwrite NLL in column 3
    n_nll = CH3_LIK_CH4_N_NLL
    for i, tv in enumerate(np.linspace(0.0, 1.0, n_nll, endpoint=True)):
        u = ch3_knob_smoothstep(float(tv))
        plot_u = plot_surface(log_u=1.0, nll_u=u)
        frames.append(_ch3_lik_emit_ch4_03_formulas(
            plot_u,
            right_blocks=right_exp,
            bottom_prog=ch4_bottom_prog_ch4_03_three_col(lik_u=1.0, log_line_us=(1.0, 1.0), nll_u=u),
        ))

    plot_nll = plot_surface(log_u=1.0, nll_u=1.0)
    bottom_nll = ch4_formula_blocks_nll_story()
    corner = ch4_cached_notation_corner_blocks()

    # 7 — swap numbered knobs → w_ST / w_EL / b (before notation moves down)
    for slot in range(3):
        for tv in np.linspace(0.0, 1.0, CH3_LIK_CH4_N_KNOB_SWAP, endpoint=True):
            u = ch3_knob_smoothstep(float(tv))
            blends = tuple(1.0 if i < slot else (u if i == slot else 0.0) for i in range(3))
            emit(
                plot_surface(log_u=1.0, nll_u=1.0, knob_labeled_blend=blends),
                layout_u=1.0,
                panel_u=1.0,
                title_write_progress=1.0,
                write_progress=1.0,
                right_blocks=right_exp,
                bottom_blocks=bottom_nll,
            )

    # 8 — notation → corner; erase ℒ+log columns, NLL moves to left column
    frame_right_3col = _ch3_lik_emit_ch4_03_formulas(
        plot_nll,
        right_blocks=right_exp,
        bottom_prog=ch4_bottom_prog_ch4_03_three_col(lik_u=1.0, log_line_us=(1.0, 1.0), nll_u=1.0),
    )
    frame_corner_nll = compose_tutorial(
        plot_nll,
        right_blocks=[],
        bottom_blocks=bottom_nll,
        corner_blocks=corner,
        bottom_title=CH4_FORMULAS_SECTION_TITLE,
        corner_title=CH4_NOTATION_SECTION_TITLE,
        layout_u=1.0,
        panel_u=1.0,
        write_progress=1.0,
        plot_start_rect=CH4_LIK_PLOT_START_RECT,
        progress_override={"corner": ch4_blocks_write_from_slot(corner, 0, 0.0, style=style)},
        theme="classic_light",
    )
    for tv in np.linspace(0.0, 1.0, CH3_LIK_CH4_N_NOTATION_MOVE, endpoint=True):
        u = ch3_knob_smoothstep(float(tv))
        frames.append(ch4_blend_images(frame_right_3col, frame_corner_nll, u))

    plot_corner = plot_surface(log_u=1.0, nll_u=1.0, knob_labeled_blend=(1.0, 1.0, 1.0))
    for tv in np.linspace(0.0, 1.0, CH3_LIK_CH4_N_CORNER_WRITE, endpoint=True):
        u = ch3_knob_smoothstep(float(tv))
        corner_prog = ch4_blocks_write_from_slot(corner, 0, u, style=style)
        frames.append(
            compose_tutorial(
                plot_corner,
                right_blocks=[],
                bottom_blocks=bottom_nll,
                corner_blocks=corner,
                bottom_title=CH4_FORMULAS_SECTION_TITLE,
                corner_title=CH4_NOTATION_SECTION_TITLE,
                layout_u=1.0,
                panel_u=1.0,
                write_progress=1.0,
                plot_start_rect=CH4_LIK_PLOT_START_RECT,
                progress_override={"corner": corner_prog},
                theme="classic_light",
            )
        )

    # 9 — handwrite p(y_i | x_i) below NLL (same mathtext style as ch4_04)
    bottom_3d = ch4_formula_blocks_3d_story()
    for tv in np.linspace(0.0, 1.0, CH3_LIK_CH4_N_PROB_WRITE, endpoint=True):
        u = ch3_knob_smoothstep(float(tv))
        prog = ch4_bottom_per_block_progress(bottom_3d, {0: 1.0, 1: u})
        corner_full = ch4_blocks_write_from_slot(corner, 0, 1.0, style=style)
        frames.append(
            compose_tutorial(
                plot_corner,
                right_blocks=[],
                bottom_blocks=bottom_3d,
                corner_blocks=corner,
                bottom_title=CH4_FORMULAS_SECTION_TITLE,
                corner_title=CH4_NOTATION_SECTION_TITLE,
                layout_u=1.0,
                panel_u=1.0,
                write_progress=1.0,
                plot_start_rect=CH4_LIK_PLOT_START_RECT,
                progress_override={"bottom": prog, "corner": corner_full},
                theme="classic_light",
            )
        )

    from ch4_layout import CH4_NLL_HEATMAP_SECTION_TITLE

    nll_legend = ch4_nll_global_legend_blocks()

    def emit_ch4_03_end(plot_img, *, right_blocks=None, right_title=None, right_title_single_line=None):
        frames.append(
            compose_tutorial(
                plot_img,
                right_blocks=[] if right_blocks is None else right_blocks,
                bottom_blocks=bottom_3d,
                corner_blocks=corner,
                bottom_title=CH4_FORMULAS_SECTION_TITLE,
                corner_title=CH4_NOTATION_SECTION_TITLE,
                right_title=right_title,
                right_title_single_line=right_title_single_line,
                layout_u=1.0,
                panel_u=1.0,
                write_progress=1.0,
                plot_start_rect=CH4_LIK_PLOT_START_RECT,
                progress_override={
                    "bottom": ch4_bottom_per_block_progress(bottom_3d, {0: 1.0, 1: 1.0}),
                    "corner": ch4_blocks_write_from_slot(corner, 0, 1.0, style=style),
                },
                theme="classic_light",
            )
        )

    # 10 — heatmap reveal on NLL surface + NLL color scale legend
    for tv in np.linspace(0.0, 1.0, CH3_LIK_CH4_N_HEATMAP_REVEAL, endpoint=True):
        u = ch3_knob_smoothstep(float(tv))
        plot_h = plot_surface(
            log_u=1.0, nll_u=1.0, heatmap_u=u,
            knob_labeled_blend=(1.0, 1.0, 1.0),
        )
        emit_ch4_03_end(
            plot_h,
            right_blocks=nll_legend if u > 0.02 else [],
            right_title=CH4_NLL_HEATMAP_SECTION_TITLE if u > 0.02 else None,
            right_title_single_line=True,
        )

    # 11 — collapse NLL surface → flat plane at b = 0
    for tv in np.linspace(0.0, 1.0, CH3_LIK_CH4_N_SQUISH_PLANE, endpoint=True):
        u = ch3_knob_smoothstep(float(tv))
        plot_s = plot_surface(
            log_u=1.0, nll_u=1.0, heatmap_u=1.0, squish_u=u, plane_drop_u=0.0,
            knob_labeled_blend=(1.0, 1.0, 1.0),
        )
        emit_ch4_03_end(
            plot_s,
            right_blocks=nll_legend,
            right_title=CH4_NLL_HEATMAP_SECTION_TITLE,
            right_title_single_line=True,
        )

    # 12 — slide plane from b = 0 down to b = -3 (handoff to ch4_05a CT scan)
    for tv in np.linspace(0.0, 1.0, CH3_LIK_CH4_N_PLANE_DROP, endpoint=True):
        u = ch3_knob_smoothstep(float(tv))
        plot_d = plot_surface(
            log_u=1.0, nll_u=1.0, heatmap_u=1.0, squish_u=1.0, plane_drop_u=u,
            knob_labeled_blend=(1.0, 1.0, 1.0),
        )
        emit_ch4_03_end(
            plot_d,
            right_blocks=nll_legend,
            right_title=CH4_NLL_HEATMAP_SECTION_TITLE,
            right_title_single_line=True,
        )

    if frames:
        last = frames[-1]
        for _ in range(max(8, CH3_SCRIPT_N_HOLD // 4)):
            frames.append(last.copy())
    return frames


def _ch3_lik_margin_bounds(bounds, *, frac=0.07):
    """Inset axis limits so zig-zag waypoints stay comfortably inside."""
    dlo1, dhi1, dlo2, dhi2, dlob, dhib = bounds
    out = []
    for lo, hi in ((dlo1, dhi1), (dlo2, dhi2), (dlob, dhib)):
        span = max(float(hi) - float(lo), 1e-9)
        m = float(frac) * span
        out.extend([float(lo) + m, float(hi) - m])
    return tuple(out)


def _ch3_lik_zigzag_path_3d(bounds, *, end, n_pts=None):
    """Zig-zag through weight space inside ``bounds``, ending at ``end``."""
    n_pts = int(CH3_LIK_3D_N_PATH if n_pts is None else n_pts)
    end = np.asarray(end, dtype=float).reshape(3)
    dlo1, dhi1, dlo2, dhi2, dlob, dhib = _ch3_lik_margin_bounds(bounds)
    mid1 = 0.5 * (dlo1 + dhi1)
    mid2 = 0.5 * (dlo2 + dhi2)
    midb = 0.5 * (dlob + dhib)
    waypoints = np.array([
        [dhi1, dlo2, dhib],
        [dlo1, dhi2, dlob],
        [dhi1, dhi2, midb],
        [mid1, dlo2, dhib],
        [dlo1, mid2, dlob],
        [mid1, dhi2, midb],
        end,
    ], dtype=float)
    seg = np.linalg.norm(np.diff(waypoints, axis=0), axis=1)
    cum = np.concatenate([[0.0], np.cumsum(seg)])
    total = float(cum[-1])
    if total < 1e-12:
        return np.tile(end, (n_pts, 1)).astype(float)
    targets = np.linspace(0.0, total, n_pts, endpoint=True)
    pts = np.empty((n_pts, 3), dtype=float)
    for j, ut in enumerate(targets):
        i = int(np.searchsorted(cum, ut, side="right") - 1)
        i = min(max(i, 0), len(waypoints) - 2)
        t = float((ut - cum[i]) / max(seg[i], 1e-12))
        pts[j] = (1.0 - t) * waypoints[i] + t * waypoints[i + 1]
    pts[-1] = end
    return pts.astype(float)


def _ch3_lik_fun_path_3d(n_pts=None, start=None, bounds=None):
    """Path in (w_ST, w_EL, b); ends at ``start`` (default ``CH3_LIK_3D_PATH_START``)."""
    end = CH3_LIK_3D_PATH_START if start is None else start
    if bounds is None:
        t = np.linspace(0.0, 1.0, int(CH3_LIK_3D_N_PATH if n_pts is None else n_pts), endpoint=True)
        w1_c, w2_c, b_c = (float(end[0]), float(end[1]), float(end[2]))
        w1 = w1_c + 1.8 * np.sin(2.0 * np.pi * t) * (0.35 + 0.65 * t)
        w2 = w2_c + 1.6 * np.cos(2.4 * np.pi * t + 0.6) * (0.35 + 0.65 * t)
        b = b_c + 0.55 * np.sin(4.0 * np.pi * t + 1.1)
        return np.column_stack([w1, w2, b]).astype(float)
    return _ch3_lik_zigzag_path_3d(bounds, end=end, n_pts=n_pts)


def _ch3_lik_style_ax3d(ax3d, dlo1, dhi1, dlo2, dhi2, dlob, dhib):
    """3-D axis labels (+45%) and sparser z ticks."""
    from matplotlib.ticker import MaxNLocator

    fs = float(AXIS_LABEL_SIZE) * float(CH3_LIK_3D_AXIS_LABEL_SCALE)
    ax3d.set_xlim(dlo1, dhi1)
    ax3d.set_ylim(dlo2, dhi2)
    ax3d.set_zlim(dlob, dhib)
    ax3d.set_xlabel(r"$w_{\mathrm{ST}}$", fontsize=fs, labelpad=8)
    ax3d.set_ylabel(r"$w_{\mathrm{EL}}$", fontsize=fs, labelpad=8)
    ax3d.set_zlabel(r"$b$", fontsize=fs, labelpad=8)
    ax3d.zaxis.set_major_locator(MaxNLocator(nbins=5))


def _ch3_lik_ax3d_fixed_bounds(ref_lo1, hi1, lo2, hi2, lob, hib, path_xyz, *, extra_pts=None):
    """Axis limits that contain the full path (and grid), with padding."""
    ref_lo = np.array([float(ref_lo1), float(lo2), float(lob)], dtype=float)
    ref_hi = np.array([float(hi1), float(hi2), float(hib)], dtype=float)
    ref_span = np.maximum(ref_hi - ref_lo, 1e-9)
    pts = [np.asarray(path_xyz, dtype=float).reshape(-1, 3)]
    if extra_pts is not None:
        pts.append(np.asarray(extra_pts, dtype=float).reshape(-1, 3))
    P = np.vstack(pts)
    P = P[np.isfinite(P).all(axis=1)]
    if P.shape[0] == 0:
        return float(ref_lo1), float(hi1), float(lo2), float(hi2), float(lob), float(hib)
    lo = np.minimum(P.min(axis=0), ref_lo)
    hi = np.maximum(P.max(axis=0), ref_hi)
    span = np.maximum(hi - lo, CH3_KERAS_NLL3D_ZOOM_MIN_SPAN_FRAC * ref_span)
    center = 0.5 * (lo + hi)
    lo = center - 0.5 * span
    hi = center + 0.5 * span
    pad = CH3_KERAS_NLL3D_VIEW_PAD_FRAC * (hi - lo)
    return (
        float(lo[0] - pad[0]),
        float(hi[0] + pad[0]),
        float(lo[1] - pad[1]),
        float(hi[1] + pad[1]),
        float(lo[2] - pad[2]),
        float(hi[2] + pad[2]),
    )


def _ch3_lik_cam_azim(cam_u, *, total_deg=CH3_LIK_3D_CAM_PATH_ROT, base=CH3_LIK_3D_CAM_AZIM0):
    u = float(np.clip(float(cam_u), 0.0, 1.0))
    return float(base + float(total_deg) * u)


def _ch3_lik_lerp_bounds(bounds_a, bounds_b, u):
    u = float(np.clip(float(u), 0.0, 1.0))
    a = tuple(float(v) for v in bounds_a)
    b = tuple(float(v) for v in bounds_b)
    return tuple(float(x + (y - x) * u) for x, y in zip(a, b))


def _ch3_lik_bounds_span(bounds):
    return np.array(
        [bounds[1] - bounds[0], bounds[3] - bounds[2], bounds[5] - bounds[4]],
        dtype=float,
    )


def _ch3_lik_ball_zoom_bounds(ws, we, bb, wide_bounds, *, r_scale=CH3_LIK_3D_BALL_R_SCALE):
    """Tight axis limits framing the NLL ball around one weight-space point."""
    center = np.array([float(ws), float(we), float(bb)], dtype=float)
    span_ref = float(np.max(_ch3_lik_bounds_span(wide_bounds)))
    R = float(r_scale) * span_ref
    half = max(R * 2.35, span_ref * 0.055)
    lo = center - half
    hi = center + half
    pad = 0.08 * (hi - lo)
    return (
        float(lo[0] - pad[0]), float(hi[0] + pad[0]),
        float(lo[1] - pad[1]), float(hi[1] + pad[1]),
        float(lo[2] - pad[2]), float(hi[2] + pad[2]),
    )


def _ch3_lik_ball_vector_field(study, exam, y, ws, we, bb, wide_bounds):
    """Sample points on a sphere + negative-NLL gradient vectors at each sample."""
    span_ref = float(np.max(_ch3_lik_bounds_span(wide_bounds)))
    R = float(CH3_LIK_3D_BALL_R_SCALE) * span_ref
    offs = _ch3_ball_unit_offsets(int(CH3_LIK_3D_BALL_NSAMPLE))
    center = np.array([float(ws), float(we), float(bb)], dtype=float)
    P = center[np.newaxis, :] + offs * R
    L = _ch3_nll_sum_on_flat_grid(study, exam, y, P[:, 0], P[:, 1], P[:, 2])
    U = np.empty(P.shape[0], dtype=float)
    V = np.empty(P.shape[0], dtype=float)
    W = np.empty(P.shape[0], dtype=float)
    for i, p in enumerate(P):
        g1, g2, gb = _ch3_nll_sum_grad_at_point(study, exam, y, p[0], p[1], p[2])
        U[i], V[i], W[i] = -float(g1), -float(g2), -float(gb)
    return P, L, U, V, W


_BALL_FIELD_CACHE: dict[tuple, tuple] = {}


def _ch3_lik_ball_vector_field_cached(study, exam, y, ws, we, bb, wide_bounds):
    key = (
        round(float(ws), 5), round(float(we), 5), round(float(bb), 5),
        round(float(wide_bounds[0]), 4), round(float(wide_bounds[1]), 4),
        round(float(wide_bounds[2]), 4), round(float(wide_bounds[3]), 4),
        round(float(wide_bounds[4]), 4), round(float(wide_bounds[5]), 4),
        int(CH3_LIK_3D_BALL_NSAMPLE),
    )
    hit = _BALL_FIELD_CACHE.get(key)
    if hit is not None:
        return hit
    out = _ch3_lik_ball_vector_field(study, exam, y, ws, we, bb, wide_bounds)
    _BALL_FIELD_CACHE[key] = out
    return out


def _ch3_lik_ball_nll_limits(study, exam, y, path, wide_bounds):
    chunks = []
    for r in np.asarray(path, dtype=float):
        _, L, _, _, _ = _ch3_lik_ball_vector_field_cached(
            study, exam, y, float(r[0]), float(r[1]), float(r[2]), wide_bounds,
        )
        chunks.append(L)
    flat = np.concatenate(chunks) if chunks else np.array([0.0])
    return float(np.min(flat)), float(np.max(flat))


def _ch3_lik_gd_ax3d_bounds(pack, *, specs_fn=None):
    """Fixed 3-D limits for ch4_06/07: contain full GD motion + arrow tips."""
    if specs_fn is None:
        specs_fn = _ch3_lik_gd_frame_specs
    study, exam, y = pack["study"], pack["exam"], pack["y"]
    eta = float(pack["gd_eta"])
    pts = [np.asarray(pack["path"], dtype=float).reshape(-1, 3)]
    for r in pack["path"]:
        ws, we, bb = float(r[0]), float(r[1]), float(r[2])
        g1, g2, gb = _ch3_nll_sum_grad_at_point(study, exam, y, ws, we, bb)
        pts.append(np.array([
            [ws - eta * g1, we, bb],
            [ws, we - eta * g2, bb],
            [ws, we, bb - eta * gb],
            [ws - eta * g1, we - eta * g2, bb - eta * gb],
        ], dtype=float))
    for spec in specs_fn(pack):
        pts.append(np.array([[spec["ws"], spec["we"], spec["bb"]]], dtype=float))
    all_pts = np.vstack(pts)
    k_lo1, k_hi1 = float(pack["W1m"].min()), float(pack["W1m"].max())
    k_lo2, k_hi2 = float(pack["W2m"].min()), float(pack["W2m"].max())
    k_lob, k_hib = float(pack["Bm"].min()), float(pack["Bm"].max())
    return _ch3_lik_ax3d_fixed_bounds(
        k_lo1, k_hi1, k_lo2, k_hi2, k_lob, k_hib, all_pts,
    )


def _ch3_lik_gd_bounds_source(study, exam, y, W1m, W2m, Bm):
    """Minimal pack fields for ``_ch3_lik_gd_ax3d_bounds`` (matches ch4_06, unchanged)."""
    gd_trail = _ch3_lik_gd_path(
        study, exam, y, CH3_LIK_3D_PATH_START, CH3_LIK_GD_N_ITERS, CH3_LIK_GD_STEP,
    )
    return {
        "study": study, "exam": exam, "y": y,
        "W1m": W1m, "W2m": W2m, "Bm": Bm,
        "path": gd_trail,
        "gd_start": CH3_LIK_3D_PATH_START,
        "gd_n_iters": CH3_LIK_GD_N_ITERS,
        "gd_eta": CH3_LIK_GD_STEP,
    }


def _ch3_lik_story_view_bounds(pack):
    """3-D axis limits shared with ch4_06 opening (``gd_ax3d_bounds``)."""
    b = pack.get("gd_ax3d_bounds")
    if b is not None:
        return b
    return pack["ax3d_bounds"]


def _ch3_lik_draw_gd_axis_arrow(ax3d, x0, y0, z0, dx, dy, dz, color, *, head_len, alpha=1.0):
    """Draw one GD arrow: line shaft + quiver head (works for short vectors)."""
    mag = float(np.hypot(dx, np.hypot(dy, dz)))
    if mag < 1e-12:
        return
    a = float(np.clip(float(alpha), 0.0, 1.0))
    x1, y1, z1 = float(x0 + dx), float(y0 + dy), float(z0 + dz)
    ax3d.plot(
        [x0, x1], [y0, y1], [z0, z1],
        color=color, linewidth=3.2, alpha=a * 0.95, zorder=18, solid_capstyle="round",
    )
    ux, uy, uz = dx / mag, dy / mag, dz / mag
    hl = float(min(max(float(head_len), 0.04 * mag), 0.42 * mag))
    ax3d.quiver(
        x1 - ux * hl, y1 - uy * hl, z1 - uz * hl,
        ux * hl, uy * hl, uz * hl,
        color=color, arrow_length_ratio=0.42, linewidth=2.6,
        normalize=False, alpha=a * 0.95, zorder=19,
    )


def _ch3_lik_draw_gd_combined_arrow(ax3d, ws, we, bb, grad, eta, color, *, bounds, alpha=1.0):
    g1, g2, gb = (float(grad[0]), float(grad[1]), float(grad[2]))
    span = float(np.max(_ch3_lik_bounds_span(bounds))) if bounds is not None else 1.0
    head_len = 0.06 * span
    _ch3_lik_draw_gd_axis_arrow(
        ax3d, float(ws), float(we), float(bb),
        -float(eta) * g1, -float(eta) * g2, -float(eta) * gb,
        color, head_len=head_len, alpha=alpha,
    )


def _ch3_lik_draw_gd_step_arrows(ax3d, ws, we, bb, grad, eta, *, visible=(True, True, True), bounds=None, alpha=1.0):
    """Axis-aligned GD step arrows: length = α × ∂NLL/∂(coord), direction of the update."""
    from ch4_layout import CH4_GD_ARROW_B_COLOR, CH4_GD_ARROW_EL_COLOR, CH4_GD_ARROW_ST_COLOR

    g1, g2, gb = (float(grad[0]), float(grad[1]), float(grad[2]))
    eta = float(eta)
    span = float(np.max(_ch3_lik_bounds_span(bounds))) if bounds is not None else 1.0
    head_len = 0.06 * span
    specs = (
        (-eta * g1, 0.0, 0.0, CH4_GD_ARROW_ST_COLOR, bool(visible[0])),
        (0.0, -eta * g2, 0.0, CH4_GD_ARROW_EL_COLOR, bool(visible[1])),
        (0.0, 0.0, -eta * gb, CH4_GD_ARROW_B_COLOR, bool(visible[2])),
    )
    for du, dv, dw, color, show in specs:
        if not show:
            continue
        _ch3_lik_draw_gd_axis_arrow(
            ax3d, float(ws), float(we), float(bb), du, dv, dw, color, head_len=head_len, alpha=alpha,
        )


def _ch3_lik_draw_gd_arrows_for_spec(
    ax3d, ws, we, bb, grad, eta, *, bounds, arrow_mode, visible, transition_u=0.0,
):
    from ch4_layout import CH4_GD_GRADIENT_COLOR

    bounds = bounds
    if arrow_mode == "none":
        return
    if arrow_mode == "split":
        _ch3_lik_draw_gd_step_arrows(
            ax3d, ws, we, bb, grad, eta, visible=visible, bounds=bounds,
        )
        return
    if arrow_mode == "combined":
        _ch3_lik_draw_gd_combined_arrow(
            ax3d, ws, we, bb, grad, eta, CH4_GD_GRADIENT_COLOR, bounds=bounds,
        )
        return
    if arrow_mode == "transition":
        u = float(np.clip(float(transition_u), 0.0, 1.0))
        if u < 1.0 - 1e-9:
            _ch3_lik_draw_gd_step_arrows(
                ax3d, ws, we, bb, grad, eta, visible=visible, bounds=bounds, alpha=1.0 - u,
            )
        if u > 1e-9:
            _ch3_lik_draw_gd_combined_arrow(
                ax3d, ws, we, bb, grad, eta, CH4_GD_GRADIENT_COLOR, bounds=bounds, alpha=u,
            )


def _ch3_lik_gd_frame_specs(pack):
    """Frame descriptors for sequential per-parameter GD (10 iterations)."""
    study, exam, y = pack["study"], pack["exam"], pack["y"]
    ws, we, bb = (float(pack["gd_start"][0]), float(pack["gd_start"][1]), float(pack["gd_start"][2]))
    eta = float(pack["gd_eta"])
    n_iters = int(pack["gd_n_iters"])
    hold_n = max(int(CH3_LIK_GD_N_HOLD_ARROWS), 1)
    step_n = max(int(CH3_LIK_GD_N_PARAM_STEP), 2)
    specs = []

    def _append(ws_i, we_i, bb_i, grad, *, arrows, bold, arrow_mode="split", grad_red=False, bold_all=False, transition_u=0.0):
        specs.append({
            "ws": float(ws_i), "we": float(we_i), "bb": float(bb_i),
            "grad": (float(grad[0]), float(grad[1]), float(grad[2])),
            "eta": eta,
            "arrows": tuple(bool(v) for v in arrows),
            "bold": bold,
            "bold_all": bool(bold_all),
            "grad_red": bool(grad_red),
            "arrow_mode": str(arrow_mode),
            "transition_u": float(transition_u),
        })

    for _ in range(n_iters):
        g1, g2, gb = _ch3_nll_sum_grad_at_point(study, exam, y, ws, we, bb)
        grad = (g1, g2, gb)
        for _ in range(hold_n):
            _append(ws, we, bb, grad, arrows=(True, True, True), bold=None)

        ws0 = ws
        for si in range(step_n):
            u = ch3_knob_smoothstep(float(si) / float(step_n - 1))
            _append(
                ws0 - u * eta * g1, we, bb, grad,
                arrows=(False, True, True), bold=0,
            )
        ws = ws0 - eta * g1

        we0 = we
        for si in range(step_n):
            u = ch3_knob_smoothstep(float(si) / float(step_n - 1))
            _append(
                ws, we0 - u * eta * g2, bb, grad,
                arrows=(False, False, True), bold=1,
            )
        we = we0 - eta * g2

        bb0 = bb
        for si in range(step_n):
            u = ch3_knob_smoothstep(float(si) / float(step_n - 1))
            _append(
                ws, we, bb0 - u * eta * gb, grad,
                arrows=(False, False, False), bold=2,
            )
        bb = bb0 - eta * gb

    return specs


def _ch3_lik_gd_combined_frame_specs(pack):
    """ch4_07: split arrows once, then red combined gradient arrow + simultaneous GD."""
    study, exam, y = pack["study"], pack["exam"], pack["y"]
    ws, we, bb = (float(pack["gd_start"][0]), float(pack["gd_start"][1]), float(pack["gd_start"][2]))
    eta = float(pack["gd_eta"])
    n_iters = int(pack["gd_n_iters"])
    hold_n = max(int(CH3_LIK_GD_N_HOLD_ARROWS), 1)
    step_n = max(int(CH3_LIK_GD_N_PARAM_STEP), 2)
    combine_n = max(int(CH3_LIK_GD_N_COMBINE), 2)
    specs = []

    def _append(ws_i, we_i, bb_i, grad, *, arrow_mode, grad_red=False, bold_all=False, transition_u=0.0):
        specs.append({
            "ws": float(ws_i), "we": float(we_i), "bb": float(bb_i),
            "grad": (float(grad[0]), float(grad[1]), float(grad[2])),
            "eta": eta,
            "arrows": (True, True, True),
            "bold": None,
            "bold_all": bool(bold_all),
            "grad_red": bool(grad_red),
            "arrow_mode": str(arrow_mode),
            "transition_u": float(transition_u),
        })

    g1, g2, gb = _ch3_nll_sum_grad_at_point(study, exam, y, ws, we, bb)
    grad = (g1, g2, gb)
    for _ in range(hold_n):
        _append(ws, we, bb, grad, arrow_mode="split", grad_red=False)

    for si in range(combine_n):
        u = ch3_knob_smoothstep(float(si) / float(combine_n - 1))
        _append(
            ws, we, bb, grad,
            arrow_mode="transition",
            transition_u=u,
            grad_red=u >= 0.5,
        )

    for _ in range(n_iters):
        g1, g2, gb = _ch3_nll_sum_grad_at_point(study, exam, y, ws, we, bb)
        grad = (g1, g2, gb)
        for _ in range(hold_n):
            _append(ws, we, bb, grad, arrow_mode="combined", grad_red=True)

        ws0, we0, bb0 = ws, we, bb
        for si in range(step_n):
            u = ch3_knob_smoothstep(float(si) / float(step_n - 1))
            _append(
                ws0 - u * eta * g1, we0 - u * eta * g2, bb0 - u * eta * gb,
                grad,
                arrow_mode="none",
                grad_red=True,
                bold_all=True,
            )
        ws = ws0 - eta * g1
        we = we0 - eta * g2
        bb = bb0 - eta * gb

    return specs


def _ch3_lik_draw_ball_vectors(
    ax3d, study, exam, y, ws, we, bb, wide_bounds, *,
    vmin, vmax, cmap=None, alpha=0.86,
    ball_field=None,
):
    if ball_field is None:
        ball_field = _ch3_lik_ball_vector_field_cached(study, exam, y, ws, we, bb, wide_bounds)
    P, L, U, V, W = ball_field
    gn = np.sqrt(U * U + V * V + W * W)
    keep = gn > 1e-14
    if not np.any(keep):
        return
    P = P[keep]
    L = L[keep]
    U = U[keep]
    V = V[keep]
    W = W[keep]
    from ch4_layout import ch4_nll_heatmap_cmap

    cmap = ch4_nll_heatmap_cmap() if cmap is None else cmap
    span = max(float(vmax) - float(vmin), 1e-9)
    cols = cmap((L - float(vmin)) / span)
    ax3d.scatter(
        P[:, 0], P[:, 1], P[:, 2],
        c=L, cmap=cmap, vmin=float(vmin), vmax=float(vmax),
        s=22.0, alpha=float(alpha) * 0.55, linewidths=0, depthshade=False, zorder=6,
    )
    ax3d.quiver(
        P[:, 0], P[:, 1], P[:, 2],
        U, V, W,
        colors=cols,
        length=float(CH3_LIK_3D_BALL_QUIVER_LEN),
        normalize=True,
        alpha=float(alpha),
        linewidth=0.95,
        arrow_length_ratio=0.34,
        zorder=7,
    )


def _ch3_lik_3d_measurements_pack(*, ball_colormap_limits=False):
    state = _ch3_lik86_terminal_state()
    study, exam, y = state["study"], state["exam"], state["y"]
    ws, we, bb = state["w_st"], state["w_el"], state["b"]
    gn = 10 if _CH3_DRAFT else 14
    w1g = np.linspace(float(state["w1_lo"]), float(state["w1_hi"]), gn)
    w2g = np.linspace(float(state["w2_lo"]), float(state["w2_hi"]), gn)
    bg = np.linspace(-float(CH3_LIK_W12_B_HALF), float(CH3_LIK_W12_B_HALF), gn)
    W1m, W2m, Bm = np.meshgrid(w1g, w2g, bg, indexing="ij")
    Lf = _ch3_nll_sum_on_flat_grid(study, exam, y, W1m.ravel(), W2m.ravel(), Bm.ravel()).reshape(W1m.shape)
    vmin, vmax = float(np.nanmin(Lf)), float(np.nanmax(Lf))
    k_lo1, k_hi1 = float(W1m.min()), float(W1m.max())
    k_lo2, k_hi2 = float(W2m.min()), float(W2m.max())
    k_lob, k_hib = float(Bm.min()), float(Bm.max())
    gd_ax3d_bounds = _ch3_lik_gd_ax3d_bounds(
        _ch3_lik_gd_bounds_source(study, exam, y, W1m, W2m, Bm),
    )
    path = _ch3_lik_fun_path_3d(start=CH3_LIK_3D_PATH_START, bounds=gd_ax3d_bounds)
    ax3d_bounds = _ch3_lik_ax3d_fixed_bounds(
        k_lo1, k_hi1, k_lo2, k_hi2, k_lob, k_hib, path,
        extra_pts=np.array([[path[0, 0], path[0, 1], path[0, 2]]], dtype=float),
    )
    path_nll = np.array([
        float(-loss_log_likelihood(float(r[0]), float(r[1]), float(r[2]), study, exam, y))
        for r in path
    ], dtype=float)
    if ball_colormap_limits:
        ball_vmin, ball_vmax = _ch3_lik_ball_nll_limits(study, exam, y, path, gd_ax3d_bounds)
    else:
        ball_vmin, ball_vmax = float(vmin), float(vmax)
    span = max(float(ball_vmax) - float(ball_vmin), 1e-9)
    from ch4_layout import ch4_nll_heatmap_cmap

    nll_cmap = ch4_nll_heatmap_cmap()
    path_colors = [nll_cmap(float((v - ball_vmin) / span)) for v in path_nll]
    n_slow = max(CH3_LIK_3D_N_PATH // 3, 20)
    n_fast = CH3_LIK_3D_N_PATH - n_slow
    path_us = list(np.linspace(0.02, 0.35, n_slow, endpoint=True)) + list(
        np.linspace(0.35, 1.0, n_fast, endpoint=True)
    )
    return {
        "study": study, "exam": exam, "y": y,
        "W1m": W1m, "W2m": W2m, "Bm": Bm, "Lf": Lf,
        "vmin": vmin, "vmax": vmax,
        "path": path, "path_colors": path_colors,
        "ax3d_bounds": ax3d_bounds,
        "gd_ax3d_bounds": gd_ax3d_bounds,
        "path_us": path_us, "n_slow": n_slow, "n_fast": n_fast,
        "ball_vmin": ball_vmin, "ball_vmax": ball_vmax,
    }


_CH4_NLL_GLOBAL_SCALE = None


def ch4_nll_global_scale():
    """Canonical NLL heatmap limits — same as ch4_05a CT scan ``pack['vmin'/'vmax']``."""
    global _CH4_NLL_GLOBAL_SCALE
    if _CH4_NLL_GLOBAL_SCALE is None:
        pack = _ch3_lik_3d_measurements_pack(ball_colormap_limits=False)
        _CH4_NLL_GLOBAL_SCALE = (float(pack["vmin"]), float(pack["vmax"]))
    return _CH4_NLL_GLOBAL_SCALE


def ch4_nll_global_legend_blocks(*, stacked=False, pre_gap_pt=None):
    """Right-rail NLL color scale — shared range and labels everywhere."""
    from ch4_layout import ch4_nll_heatmap_legend_blocks

    lo, hi = ch4_nll_global_scale()
    return ch4_nll_heatmap_legend_blocks(lo, hi)


def ch3_frame_lik_weight3d_measurements(
    study, exam, y, ws, we, bb, *,
    W1m, W2m, Bm, Lf, vmin, vmax,
    path_xyz=None,
    path_colors=None,
    path_u=1.0,
    notation_condensed=False,
    measurements=None,
    write_progress=1.0,
    plot_alpha=1.0,
    ax3d_bounds=None,
    show_weight_grid=False,
    show_ball_vectors=False,
    wide_bounds=None,
    ball_vmin=None,
    ball_vmax=None,
    ball_field=None,
    elev=CH3_LIK_CH4_CT_ELEV,
    azim=None,
    cam_azim_u=0.0,
    cam_rot_deg=CH3_LIK_3D_CAM_PATH_ROT,
    grad=None,
    step_size=None,
    gd_formulas=False,
    gd_arrows_grad=None,
    gd_arrows_visible=None,
    gd_bold_update_idx=None,
    gd_bold_all_updates=False,
    gd_grad_red=False,
    gd_arrow_mode="split",
    gd_arrow_transition_u=0.0,
    gd_bottom_blocks=None,
    progress_override=None,
    right_write_progress=None,
    title_write_progress=None,
    bottom_write_progress=None,
    show_path_line=True,
    voxel_draw=None,
    point_s=None,
    point_color=None,
    right_title=None,
    right_title_single_line=None,
):
    from ch4_layout import (
        CH4_FORMULAS_SECTION_TITLE,
        CH4_HERE_SECTION_TITLE,
        CH4_NOTATION_SECTION_TITLE,
        ch4_cached_formula_blocks_3d_story,
        ch4_cached_formula_blocks_gd_story,
        ch4_cached_notation_corner_blocks,
        ch4_formula_blocks_gd_story,
        ch4_knob_asset_pack,
        ch4_nll_heatmap_cmap,
        ch4_rails_cache_key,
        ch4_rails_cache_key_gd,
        ch4_we_are_here_blocks,
        compose_tutorial,
    )

    fig, ax_data, ax3d, axes_k = ch4_figure_duo_weight3d()
    leg = legend_linear_equation_values_bold_param(ws, we, bb, "all")
    ch3_draw_left_panel(ax_data, ws, we, bb, study, exam, y, leg, show_colormap=True, highlight_mistakes_flag=False)
    ax_data.set_xlim(*xlim)
    ax_data.set_ylim(*ylim)
    finalize_style_legend_tex(ax_data)
    knob_rgbs, canvas_sides = ch4_knob_asset_pack()
    ch3_draw_knob_row(
        fig, axes_k, ws, we, bb, "st", knob_rgbs, canvas_sides,
        rot_strip_deg=0.0, strip_scale=1.0,
        knob_rots=ch3_k1_knob_rots_at(ws, we, bb), knob_scales=[1.0, 1.0, 1.0], ax_data=ax_data,
    )
    if show_weight_grid:
        ax3d.scatter(
            W1m.ravel(), W2m.ravel(), Bm.ravel(),
            c=Lf.ravel(), cmap=ch4_nll_heatmap_cmap(), vmin=float(vmin), vmax=float(vmax),
            s=28.0, alpha=0.45, linewidths=0, depthshade=False, zorder=1,
        )
    k_lo1, k_hi1 = float(W1m.min()), float(W1m.max())
    k_lo2, k_hi2 = float(W2m.min()), float(W2m.max())
    k_lob, k_hib = float(Bm.min()), float(Bm.max())
    if ax3d_bounds is not None:
        dlo1, dhi1, dlo2, dhi2, dlob, dhib = ax3d_bounds
    else:
        dlo1, dhi1, dlo2, dhi2, dlob, dhib = _ch3_keras_ax3d_zoom_bounds(
            k_lo1, k_hi1, k_lo2, k_hi2, k_lob, k_hib, np.array([[ws, we, bb]], dtype=float),
        )
    _ch3_lik_style_ax3d(ax3d, dlo1, dhi1, dlo2, dhi2, dlob, dhib)
    if azim is None:
        if abs(float(cam_rot_deg)) > 1e-9 or abs(float(cam_azim_u)) > 1e-9:
            azim = _ch3_lik_cam_azim(cam_azim_u, total_deg=float(cam_rot_deg))
        else:
            azim = float(CH3_LIK_CH4_CT_AZIM)
    ax3d.view_init(elev=float(elev), azim=float(azim))
    ref_bounds = wide_bounds if wide_bounds is not None else (
        ax3d_bounds if ax3d_bounds is not None else (dlo1, dhi1, dlo2, dhi2, dlob, dhib)
    )
    if gd_arrows_grad is not None:
        vis = gd_arrows_visible if gd_arrows_visible is not None else (True, True, True)
        _ch3_lik_draw_gd_arrows_for_spec(
            ax3d, ws, we, bb, gd_arrows_grad, float(step_size if step_size is not None else CH3_LIK_GD_STEP),
            bounds=(dlo1, dhi1, dlo2, dhi2, dlob, dhib),
            arrow_mode=str(gd_arrow_mode),
            visible=vis,
            transition_u=float(gd_arrow_transition_u),
        )
    elif show_ball_vectors:
        _ch3_lik_draw_ball_vectors(
            ax3d, study, exam, y, ws, we, bb, ref_bounds,
            vmin=float(ball_vmin if ball_vmin is not None else vmin),
            vmax=float(ball_vmax if ball_vmax is not None else vmax),
            ball_field=ball_field,
        )
    if voxel_draw is not None:
        xs = voxel_draw.get("xs")
        if xs is not None and len(xs):
            ax3d.bar3d(
                voxel_draw["xs"], voxel_draw["ys"], voxel_draw["zs"],
                float(voxel_draw["dx"]), float(voxel_draw["dy"]), float(voxel_draw["dz"]),
                color=voxel_draw["colors"],
                shade=False,
                linewidth=0.0,
                edgecolor=(0.0, 0.0, 0.0, 0.0),
                zorder=6,
            )
    pt_s = float(320 if point_s is None else point_s)
    pt_c = CH3_LIK_3D_POINT_COLOR if point_color is None else str(point_color)
    ax3d.scatter(
        [ws], [we], [bb], s=pt_s, c=[pt_c], edgecolors="white",
        linewidths=2.0, depthshade=False, zorder=20,
    )
    if show_path_line and path_xyz is not None and len(path_xyz) >= 2:
        P = np.asarray(path_xyz, dtype=float)
        n_keep = max(2, int(round(float(path_u) * (P.shape[0] - 1))) + 1)
        Pk = P[:n_keep]
        if path_colors is not None and len(path_colors) >= n_keep:
            cols = np.asarray(path_colors[:n_keep])
            for i in range(Pk.shape[0] - 1):
                ax3d.plot(
                    Pk[i:i + 2, 0], Pk[i:i + 2, 1], Pk[i:i + 2, 2],
                    color=cols[i], linewidth=3.2, alpha=0.95,
                )
        else:
            ax3d.plot(Pk[:, 0], Pk[:, 1], Pk[:, 2], color=CH3_LIK_3D_POINT_COLOR, linewidth=3.0, alpha=0.9)
    plot_img = fig_to_image(fig, dpi=CH3_ANIM_DPI)
    plt.close(fig)
    nll = float(-loss_log_likelihood(ws, we, bb, study, exam, y))
    bvmin = float(ball_vmin if ball_vmin is not None else vmin)
    bvmax = float(ball_vmax if ball_vmax is not None else vmax)
    if measurements is not None:
        right = measurements
    else:
        right = ch4_we_are_here_blocks(
            ws, we, bb, nll,
            nll_vmin=bvmin, nll_vmax=bvmax,
            point_color=CH3_LIK_3D_POINT_COLOR,
            grad=grad, step_size=step_size,
        )
    if gd_bottom_blocks is not None:
        bottom = gd_bottom_blocks
        rails_key = None
    elif gd_formulas:
        bottom = ch4_formula_blocks_gd_story(
            highlight_update_idx=gd_bold_update_idx,
            highlight_all_updates=gd_bold_all_updates,
            grad_red=gd_grad_red,
        )
        rails_key = (
            ch4_rails_cache_key_gd(
                highlight_update_idx=gd_bold_update_idx,
                highlight_all=gd_bold_all_updates,
                grad_red=gd_grad_red,
            )
            if float(write_progress) >= 1.0 - 1e-9
            else None
        )
    elif notation_condensed:
        bottom = ch4_cached_formula_blocks_3d_story()
        rails_key = ch4_rails_cache_key(gd_formulas=False) if float(write_progress) >= 1.0 - 1e-9 else None
    else:
        bottom = ch4_cached_formula_blocks_3d_story()
        rails_key = ch4_rails_cache_key(gd_formulas=False) if float(write_progress) >= 1.0 - 1e-9 else None
    if notation_condensed:
        rt = CH4_HERE_SECTION_TITLE if right_title is None else right_title
        kw = dict(
            plot_img=plot_img,
            right_blocks=right,
            bottom_blocks=bottom,
            corner_blocks=ch4_cached_notation_corner_blocks(),
            right_title=rt,
            bottom_title=CH4_FORMULAS_SECTION_TITLE,
            corner_title=CH4_NOTATION_SECTION_TITLE,
            right_title_color=CH3_LIK_3D_POINT_COLOR,
            write_progress=write_progress,
            plot_alpha=plot_alpha,
            theme="classic_light",
            rails_cache_key=rails_key,
            progress_override=progress_override,
            right_write_progress=right_write_progress,
            title_write_progress=title_write_progress,
            bottom_write_progress=bottom_write_progress,
        )
        if right_title_single_line is not None:
            kw["right_title_single_line"] = bool(right_title_single_line)
        return compose_tutorial(**kw)
    return compose_tutorial(
        plot_img,
        right_blocks=right,
        bottom_blocks=bottom,
        corner_blocks=ch4_cached_notation_corner_blocks(),
        right_title=CH4_HERE_SECTION_TITLE,
        bottom_title=CH4_FORMULAS_SECTION_TITLE,
        corner_title=CH4_NOTATION_SECTION_TITLE,
        right_title_color=CH3_LIK_3D_POINT_COLOR,
        write_progress=write_progress,
        plot_alpha=plot_alpha,
        theme="classic_light",
        rails_cache_key=rails_key,
        progress_override=progress_override,
        right_write_progress=right_write_progress,
        title_write_progress=title_write_progress,
        bottom_write_progress=bottom_write_progress,
    )


def _ch3_lik_we_are_here_at(pack, ws, we, bb):
    from ch4_layout import ch4_we_are_here_blocks

    nll = float(-loss_log_likelihood(ws, we, bb, pack["study"], pack["exam"], pack["y"]))
    return ch4_we_are_here_blocks(
        ws, we, bb, nll,
        nll_vmin=pack["ball_vmin"], nll_vmax=pack["ball_vmax"],
        point_color=CH3_LIK_3D_POINT_COLOR,
    )


def _ch3_lik_append_path_frames(
    frames, pack, *,
    show_ball_vectors=False,
    cam_rot_deg=CH3_LIK_3D_CAM_PATH_ROT,
    gd_formulas=False,
):
    path = pack["path"]
    path_us = pack["path_us"]
    n_path = max(len(path_us) - 1, 1)
    cols = pack["path_colors"]
    bounds = _ch3_lik_story_view_bounds(pack)
    for i, pu in enumerate(path_us):
        pu = float(pu)
        idx = min(len(path) - 1, max(0, int(round(pu * (len(path) - 1)))))
        ws_i, we_i, bb_i = path[idx]
        frames.append(
            ch3_frame_lik_weight3d_measurements(
                pack["study"], pack["exam"], pack["y"], ws_i, we_i, bb_i,
                W1m=pack["W1m"], W2m=pack["W2m"], Bm=pack["Bm"], Lf=pack["Lf"],
                vmin=pack["vmin"], vmax=pack["vmax"],
                path_xyz=path, path_colors=cols, path_u=pu,
                notation_condensed=True,
                measurements=_ch3_lik_we_are_here_at(pack, ws_i, we_i, bb_i),
                ax3d_bounds=bounds,
                wide_bounds=bounds,
                show_ball_vectors=show_ball_vectors,
                ball_vmin=pack["ball_vmin"], ball_vmax=pack["ball_vmax"],
                cam_azim_u=float(i) / float(n_path),
                cam_rot_deg=float(cam_rot_deg),
                gd_formulas=gd_formulas,
            )
        )


def _ch3_lik_append_04_right_intro(frames, pack):
    """Handwrite We-are-here title + weights + NLL before the path animation."""
    path = pack["path"]
    ws0, we0, bb0 = path[0]
    right = _ch3_lik_we_are_here_at(pack, ws0, we0, bb0)
    bounds = _ch3_lik_story_view_bounds(pack)

    def _emit(*, title_u=1.0, right_u=1.0):
        frames.append(ch3_frame_lik_weight3d_measurements(
            pack["study"], pack["exam"], pack["y"], ws0, we0, bb0,
            W1m=pack["W1m"], W2m=pack["W2m"], Bm=pack["Bm"], Lf=pack["Lf"],
            vmin=pack["vmin"], vmax=pack["vmax"],
            path_xyz=path, path_colors=pack["path_colors"], path_u=0.0,
            notation_condensed=True,
            measurements=right,
            ax3d_bounds=bounds,
            wide_bounds=bounds,
            show_ball_vectors=False,
            ball_vmin=pack["ball_vmin"], ball_vmax=pack["ball_vmax"],
            cam_azim_u=0.0,
            gd_formulas=False,
            write_progress=1.0,
            title_write_progress=title_u,
            right_write_progress=right_u,
        ))

    for tv in np.linspace(0.0, 1.0, CH3_LIK_3D_N_RIGHT_TITLE, endpoint=True):
        _emit(title_u=ch3_knob_smoothstep(float(tv)), right_u=0.0)
    for tv in np.linspace(0.0, 1.0, CH3_LIK_3D_N_RIGHT_WRITE, endpoint=True):
        _emit(title_u=1.0, right_u=ch3_knob_smoothstep(float(tv)))


def ch3_build_frames_likelihood_3d_measurements_story():
    pack = _ch3_lik_3d_measurements_pack()
    frames = []
    _ch3_lik_append_04_right_intro(frames, pack)
    _ch3_lik_append_path_frames(frames, pack, show_ball_vectors=False)
    if frames:
        last = frames[-1]
        for _ in range(max(10, CH3_SCRIPT_N_HOLD // 3)):
            frames.append(last.copy())
    return frames


def _ch3_lik_gd_path(study, exam, y, start, n_steps, eta):
    """Gradient-descent trajectory from ``start`` in (w_ST, w_EL, b)."""
    ws, we, bb = float(start[0]), float(start[1]), float(start[2])
    pts = [[ws, we, bb]]
    eta = float(eta)
    for _ in range(int(n_steps)):
        g1, g2, gb = _ch3_nll_sum_grad_at_point(study, exam, y, ws, we, bb)
        ws -= eta * float(g1)
        we -= eta * float(g2)
        bb -= eta * float(gb)
        pts.append([ws, we, bb])
    return np.asarray(pts, dtype=float)


def _ch3_lik_3d_gd_pack():
    pack = dict(_ch3_lik_3d_measurements_pack(ball_colormap_limits=False))
    trail_path = _ch3_lik_gd_path(
        pack["study"], pack["exam"], pack["y"],
        CH3_LIK_3D_PATH_START, CH3_LIK_GD_N_ITERS, CH3_LIK_GD_STEP,
    )
    study, exam, y = pack["study"], pack["exam"], pack["y"]
    path_nll = np.array([
        float(-loss_log_likelihood(float(r[0]), float(r[1]), float(r[2]), study, exam, y))
        for r in trail_path
    ], dtype=float)
    pack.update({
        "gd_start": CH3_LIK_3D_PATH_START,
        "gd_n_iters": CH3_LIK_GD_N_ITERS,
        "gd_eta": CH3_LIK_GD_STEP,
        "ball_vmin": float(np.min(path_nll)),
        "ball_vmax": float(np.max(path_nll)),
        "path": trail_path,
    })
    pack["gd_ax3d_bounds"] = CH3_LIK_CT_VIEW_BOUNDS
    pack["ax3d_bounds"] = CH3_LIK_CT_VIEW_BOUNDS
    return pack


def _ch3_lik_3d_gd_combined_pack():
    pack = _ch3_lik_3d_gd_pack()
    pack["gd_ax3d_bounds"] = CH3_LIK_CT_VIEW_BOUNDS
    pack["ax3d_bounds"] = CH3_LIK_CT_VIEW_BOUNDS
    return pack


def _ch3_lik_gd_render_frame(pack, spec, *, cam_azim_u=0.0):
    from ch4_layout import ch4_we_are_here_blocks, ch4_we_are_here_grad_line_colors

    study, exam, y = pack["study"], pack["exam"], pack["y"]
    ws, we, bb = float(spec["ws"]), float(spec["we"]), float(spec["bb"])
    grad = spec["grad"]
    eta = float(spec["eta"])
    grad_red = bool(spec.get("grad_red", False))
    nll = float(-loss_log_likelihood(ws, we, bb, study, exam, y))
    grad_colors = ch4_we_are_here_grad_line_colors(
        grad_red=grad_red,
        transition_u=float(spec.get("transition_u", 0.0)),
    )
    right = ch4_we_are_here_blocks(
        ws, we, bb, nll,
        nll_vmin=pack["ball_vmin"], nll_vmax=pack["ball_vmax"],
        point_color=CH3_LIK_3D_POINT_COLOR,
        grad=grad, step_size=eta,
        grad_line_colors=grad_colors,
    )
    view_bounds = CH3_LIK_CT_VIEW_BOUNDS
    arrow_mode = str(spec.get("arrow_mode", "split"))
    show_grad = arrow_mode != "none"
    return ch3_frame_lik_weight3d_measurements(
        study, exam, y, ws, we, bb,
        W1m=pack["W1m"], W2m=pack["W2m"], Bm=pack["Bm"], Lf=pack["Lf"],
        vmin=pack["vmin"], vmax=pack["vmax"],
        notation_condensed=True,
        measurements=right,
        ax3d_bounds=view_bounds,
        wide_bounds=view_bounds,
        show_ball_vectors=False,
        ball_vmin=pack["ball_vmin"], ball_vmax=pack["ball_vmax"],
        cam_azim_u=float(cam_azim_u),
        grad=grad,
        step_size=eta,
        gd_formulas=True,
        gd_arrows_grad=grad if show_grad else None,
        gd_arrows_visible=spec.get("arrows", (True, True, True)),
        gd_bold_update_idx=spec.get("bold"),
        gd_bold_all_updates=bool(spec.get("bold_all", False)),
        gd_grad_red=grad_red,
        gd_arrow_mode=arrow_mode,
        gd_arrow_transition_u=float(spec.get("transition_u", 0.0)),
    )


def _ch3_lik_ball_intro_frames(pack, *, gd_formulas=False):
    """Zoom → 360° spin → zoom out; shared by ch4_05 and ch4_06."""
    _BALL_FIELD_CACHE.clear()
    path = pack["path"]
    ws0, we0, bb0 = path[0]
    wide = _ch3_lik_story_view_bounds(pack)
    tight = _ch3_lik_ball_zoom_bounds(ws0, we0, bb0, wide)
    intro_ball = _ch3_lik_ball_vector_field_cached(
        pack["study"], pack["exam"], pack["y"], ws0, we0, bb0, wide,
    )
    frames = []

    def _intro_frame(bounds, azim, *, path_u=0.0):
        if gd_formulas:
            from ch4_layout import ch4_we_are_here_blocks
            g1, g2, gb = _ch3_nll_sum_grad_at_point(
                pack["study"], pack["exam"], pack["y"], ws0, we0, bb0,
            )
            nll0 = float(-loss_log_likelihood(ws0, we0, bb0, pack["study"], pack["exam"], pack["y"]))
            right = ch4_we_are_here_blocks(
                ws0, we0, bb0, nll0,
                nll_vmin=pack["ball_vmin"], nll_vmax=pack["ball_vmax"],
                point_color=CH3_LIK_3D_POINT_COLOR,
                grad=(g1, g2, gb), step_size=CH3_LIK_GD_STEP,
            )
        else:
            right = _ch3_lik_we_are_here_at(pack, ws0, we0, bb0)
        return ch3_frame_lik_weight3d_measurements(
            pack["study"], pack["exam"], pack["y"], ws0, we0, bb0,
            W1m=pack["W1m"], W2m=pack["W2m"], Bm=pack["Bm"], Lf=pack["Lf"],
            vmin=pack["vmin"], vmax=pack["vmax"],
            path_xyz=path, path_u=path_u,
            notation_condensed=True,
            measurements=right,
            ax3d_bounds=bounds,
            wide_bounds=wide,
            show_ball_vectors=True,
            ball_vmin=pack["ball_vmin"], ball_vmax=pack["ball_vmax"],
            ball_field=intro_ball,
            azim=float(azim),
            gd_formulas=gd_formulas,
        )

    for tv in np.linspace(0.0, 1.0, CH3_LIK_3D_N_INTRO_ZOOM, endpoint=True):
        u = ch3_knob_smoothstep(float(tv))
        frames.append(_intro_frame(_ch3_lik_lerp_bounds(wide, tight, u), CH3_LIK_3D_CAM_AZIM0))

    spin_n = max(int(CH3_LIK_3D_N_INTRO_SPIN), 2)
    for tv in np.linspace(0.0, 1.0, spin_n, endpoint=True):
        u = ch3_knob_smoothstep(float(tv))
        frames.append(_intro_frame(tight, CH3_LIK_3D_CAM_AZIM0 + 360.0 * float(u)))

    for tv in np.linspace(0.0, 1.0, CH3_LIK_3D_N_INTRO_ZOOM_OUT, endpoint=True):
        u = ch3_knob_smoothstep(float(tv))
        frames.append(_intro_frame(_ch3_lik_lerp_bounds(tight, wide, u), CH3_LIK_3D_CAM_AZIM0))

    return frames


def _ch3_lik_append_gd_frames(
    frames, pack, *,
    cam_rot_deg=CH3_LIK_3D_CAM_PATH_ROT,
    specs_fn=None,
):
    del cam_rot_deg  # camera pan is encoded in render_frame cam_azim_u
    if specs_fn is None:
        specs_fn = _ch3_lik_gd_frame_specs
    from ch4_export_pipeline import build_tutorial_frames

    rendered = build_tutorial_frames(
        pack,
        specs_fn(pack),
        _ch3_lik_gd_render_frame,
        prewarm="gd",
        progress_label="ch4_gd",
    )
    frames.extend(rendered)


def _ch3_lik_append_gd_combined_frames(frames, pack):
    _ch3_lik_append_gd_frames(frames, pack, specs_fn=_ch3_lik_gd_combined_frame_specs)


def _ch3_lik_story_hold(frames):
    if frames:
        last = frames[-1]
        for _ in range(max(10, CH3_SCRIPT_N_HOLD // 3)):
            frames.append(last.copy())
    return frames


def _ch3_lik_pack_path_frame(pack, path_index=0, *, show_ball_vectors=False, gd_formulas=False, **frame_kw):
    """Render one path frame from a measurements/GD pack (no full story build)."""
    path = pack["path"]
    path_us = pack["path_us"]
    i = int(np.clip(path_index, 0, len(path_us) - 1))
    pu = float(path_us[i])
    idx = min(len(path) - 1, max(0, int(round(pu * (len(path) - 1)))))
    ws_i, we_i, bb_i = path[idx]
    n_path = max(len(path_us) - 1, 1)
    bounds = _ch3_lik_story_view_bounds(pack)
    return ch3_frame_lik_weight3d_measurements(
        pack["study"], pack["exam"], pack["y"], ws_i, we_i, bb_i,
        W1m=pack["W1m"], W2m=pack["W2m"], Bm=pack["Bm"], Lf=pack["Lf"],
        vmin=pack["vmin"], vmax=pack["vmax"],
        path_xyz=path, path_colors=pack["path_colors"], path_u=pu,
        notation_condensed=True,
        measurements=_ch3_lik_we_are_here_at(pack, ws_i, we_i, bb_i),
        ax3d_bounds=bounds,
        wide_bounds=bounds,
        show_ball_vectors=show_ball_vectors,
        ball_vmin=pack["ball_vmin"], ball_vmax=pack["ball_vmax"],
        cam_azim_u=float(i) / float(n_path),
        gd_formulas=gd_formulas,
        **frame_kw,
    )


def ch4_preview_likelihood_morph_surface_frame(
    *,
    log_u=0.0,
    nll_u=0.0,
    heatmap_u=0.0,
    squish_u=0.0,
    plane_drop_u=0.0,
):
    """Plot panel only — likelihood / log / NLL morph stages (ch4_03 surface checks)."""
    state = _ch3_lik86_terminal_state()
    return ch3_frame_lik_w12_single_surface(
        state,
        log_u=float(log_u),
        nll_u=float(nll_u),
        heatmap_u=float(heatmap_u),
        squish_u=float(squish_u),
        plane_drop_u=float(plane_drop_u),
        show_axis_labels=True,
        knob_labeled_blend=(1.0, 1.0, 1.0),
    )


def ch4_preview_likelihood_notation_nll_plane_frame(
    *,
    heatmap_u=1.0,
    squish_u=0.0,
    plane_drop_u=0.0,
):
    """Composed ch4_03 frame for heatmap / squish / plane-drop checks."""
    from ch4_layout import (
        CH4_FORMULAS_SECTION_TITLE,
        CH4_LIK_PLOT_START_RECT,
        CH4_NOTATION_SECTION_TITLE,
        CH4_NLL_HEATMAP_SECTION_TITLE,
        ch4_cached_notation_corner_blocks,
        ch4_formula_blocks_3d_story,
        compose_tutorial,
    )

    state = _ch3_lik86_terminal_state()
    plot = ch3_frame_lik_w12_single_surface(
        state,
        log_u=1.0,
        nll_u=1.0,
        heatmap_u=float(heatmap_u),
        squish_u=float(squish_u),
        plane_drop_u=float(plane_drop_u),
        show_axis_labels=True,
        knob_labeled_blend=(1.0, 1.0, 1.0),
    )
    show_legend = float(heatmap_u) > 0.02
    return compose_tutorial(
        plot,
        right_blocks=ch4_nll_global_legend_blocks() if show_legend else [],
        bottom_blocks=ch4_formula_blocks_3d_story(),
        corner_blocks=ch4_cached_notation_corner_blocks(),
        bottom_title=CH4_FORMULAS_SECTION_TITLE,
        corner_title=CH4_NOTATION_SECTION_TITLE,
        right_title=CH4_NLL_HEATMAP_SECTION_TITLE if show_legend else None,
        right_title_single_line=True,
        layout_u=1.0,
        panel_u=1.0,
        write_progress=1.0,
        plot_start_rect=CH4_LIK_PLOT_START_RECT,
        theme="classic_light",
    )


def ch4_preview_likelihood_notation_nll_last_frame():
    """Last ch4_03 frame for layout checks — does not build the full MP4 story."""
    from ch4_layout import (
        CH4_FORMULAS_SECTION_TITLE,
        CH4_LIK_PLOT_START_RECT,
        CH4_NOTATION_SECTION_TITLE,
        CH4_NLL_HEATMAP_SECTION_TITLE,
        ch4_cached_notation_corner_blocks,
        ch4_formula_blocks_3d_story,
        compose_tutorial,
    )

    state = _ch3_lik86_terminal_state()
    plot = ch3_frame_lik_w12_single_surface(
        state,
        log_u=1.0,
        nll_u=1.0,
        heatmap_u=1.0,
        squish_u=1.0,
        plane_drop_u=1.0,
        show_axis_labels=True,
        knob_labeled_blend=(1.0, 1.0, 1.0),
    )
    return compose_tutorial(
        plot,
        right_blocks=ch4_nll_global_legend_blocks(),
        bottom_blocks=ch4_formula_blocks_3d_story(),
        corner_blocks=ch4_cached_notation_corner_blocks(),
        bottom_title=CH4_FORMULAS_SECTION_TITLE,
        corner_title=CH4_NOTATION_SECTION_TITLE,
        right_title=CH4_NLL_HEATMAP_SECTION_TITLE,
        right_title_single_line=True,
        layout_u=1.0,
        panel_u=1.0,
        write_progress=1.0,
        plot_start_rect=CH4_LIK_PLOT_START_RECT,
        theme="classic_light",
    )


def ch4_preview_likelihood_3d_measurements_frame(path_index=0):
    """Single ch4_04 frame for layout checks — does not build the full MP4 story."""
    pack = _ch3_lik_3d_measurements_pack(ball_colormap_limits=False)
    return _ch3_lik_pack_path_frame(pack, path_index, show_ball_vectors=False, gd_formulas=False)


def ch4_preview_likelihood_3d_ball_vectors_frame(*, intro=True, path_index=0, gd_start=False):
    """Single ch4_05 frame — intro zoom start, one path index, or GD opening pose."""
    pack = _ch3_lik_3d_measurements_pack(ball_colormap_limits=True)
    if gd_start:
        from ch4_layout import ch4_formula_blocks_3d_story
        return _ch3_lik_emit_05_formula_frame(
            pack,
            bottom_blocks=ch4_formula_blocks_3d_story(),
            bottom_prog={0: 1.0, 1: 1.0},
            at_gd_start=True,
        )
    if intro:
        path = pack["path"]
        ws0, we0, bb0 = path[0]
        wide = _ch3_lik_story_view_bounds(pack)
        intro_ball = _ch3_lik_ball_vector_field_cached(
            pack["study"], pack["exam"], pack["y"], ws0, we0, bb0, wide,
        )
        return ch3_frame_lik_weight3d_measurements(
            pack["study"], pack["exam"], pack["y"], ws0, we0, bb0,
            W1m=pack["W1m"], W2m=pack["W2m"], Bm=pack["Bm"], Lf=pack["Lf"],
            vmin=pack["vmin"], vmax=pack["vmax"],
            path_xyz=path, path_u=0.0,
            notation_condensed=True,
            measurements=_ch3_lik_we_are_here_at(pack, ws0, we0, bb0),
            ax3d_bounds=wide,
            wide_bounds=wide,
            show_ball_vectors=True,
            ball_vmin=pack["ball_vmin"], ball_vmax=pack["ball_vmax"],
            ball_field=intro_ball,
            azim=float(CH3_LIK_3D_CAM_AZIM0),
            gd_formulas=False,
        )
    return _ch3_lik_pack_path_frame(pack, path_index, show_ball_vectors=True, gd_formulas=False)


def ch4_preview_likelihood_3d_gd_frame(*, intro=True, gd_spec_index=0):
    """Single ch4_06 frame — first hold (all arrows) or one sequential-GD frame."""
    pack = _ch3_lik_3d_gd_pack()
    specs = _ch3_lik_gd_frame_specs(pack)
    idx = 0 if intro else int(np.clip(gd_spec_index, 0, len(specs) - 1))
    return _ch3_lik_gd_render_frame(pack, specs[idx], cam_azim_u=0.0)


def ch4_preview_likelihood_3d_gd_combined_frame(*, split_hold=True, combined_hold=False, gd_spec_index=0):
    """Single ch4_07 frame — split-arrow hold, combined hold, or one spec index."""
    pack = _ch3_lik_3d_gd_combined_pack()
    specs = _ch3_lik_gd_combined_frame_specs(pack)
    if split_hold:
        idx = 0
    elif combined_hold:
        hold_n = max(int(CH3_LIK_GD_N_HOLD_ARROWS), 1)
        combine_n = max(int(CH3_LIK_GD_N_COMBINE), 2)
        idx = hold_n + combine_n
    else:
        idx = int(np.clip(gd_spec_index, 0, len(specs) - 1))
    return _ch3_lik_gd_render_frame(pack, specs[idx], cam_azim_u=0.0)


def ch3_build_frames_likelihood_3d_gd_combined_story():
    pack = _ch3_lik_3d_gd_combined_pack()
    frames = []
    _ch3_lik_append_gd_combined_frames(frames, pack)
    return _ch3_lik_story_hold(frames)


def ch4_export_likelihood_3d_gd_combined():
    from ch4_export_pipeline import export_mp4_from_specs

    pack = _ch3_lik_3d_gd_combined_pack()
    return export_mp4_from_specs(
        pack,
        _ch3_lik_gd_combined_frame_specs(pack),
        _ch3_lik_gd_render_frame,
        save_mp4=save_mp4,
        filename="ch4_07_likelihood_3d_gd_combined.mp4",
        duration_ms=int(CH3_LIK_3D_MS_GD),
        story_hold_fn=_ch3_lik_story_hold,
        prewarm="gd",
        progress_label="ch4_07",
        render_fn_name="_ch3_lik_gd_render_frame",
    )


def _ch3_lik_05_end_path_index(pack):
    return max(len(pack["path_us"]) - 1, 0)


def _ch3_lik_append_05_gd_handoff(frames, pack):
    """Move ch4_05 terminal pose to ch4_06 opening (position, camera, no ball vectors)."""
    path = pack["path"]
    path_us = pack["path_us"]
    bounds = _ch3_lik_story_view_bounds(pack)
    cols = pack["path_colors"]
    n_path = max(len(path_us) - 1, 1)
    i_end = _ch3_lik_05_end_path_index(pack)
    pu_end = float(path_us[i_end])
    idx_end = min(len(path) - 1, max(0, int(round(pu_end * (len(path) - 1)))))
    ws_e, we_e, bb_e = path[idx_end]
    cam_u_end = float(i_end) / float(n_path)
    ws_t, we_t, bb_t = (float(CH3_LIK_3D_PATH_START[0]), float(CH3_LIK_3D_PATH_START[1]), float(CH3_LIK_3D_PATH_START[2]))
    for tv in np.linspace(0.0, 1.0, CH3_LIK_3D_N_05_HANDOFF, endpoint=True):
        u = ch3_knob_smoothstep(float(tv))
        ws = ws_e + u * (ws_t - ws_e)
        we = we_e + u * (we_t - we_e)
        bb = bb_e + u * (bb_t - bb_e)
        cam_u = cam_u_end + u * (0.0 - cam_u_end)
        frames.append(
            ch3_frame_lik_weight3d_measurements(
                pack["study"], pack["exam"], pack["y"], ws, we, bb,
                W1m=pack["W1m"], W2m=pack["W2m"], Bm=pack["Bm"], Lf=pack["Lf"],
                vmin=pack["vmin"], vmax=pack["vmax"],
                path_xyz=path, path_colors=cols, path_u=pu_end,
                notation_condensed=True,
                measurements=_ch3_lik_we_are_here_at(pack, ws, we, bb),
                ax3d_bounds=bounds,
                wide_bounds=bounds,
                show_ball_vectors=float(u) < 0.35,
                ball_vmin=pack["ball_vmin"], ball_vmax=pack["ball_vmax"],
                cam_azim_u=cam_u,
                gd_formulas=False,
            )
        )


def ch3_build_frames_likelihood_3d_ball_vectors_story():
    pack = _ch3_lik_3d_measurements_pack(ball_colormap_limits=True)
    frames = _ch3_lik_ball_intro_frames(pack, gd_formulas=False)
    _ch3_lik_append_path_frames(frames, pack, show_ball_vectors=True, gd_formulas=False)
    _ch3_lik_append_05_gd_handoff(frames, pack)
    return _ch3_lik_story_hold(frames)


def ch3_build_frames_likelihood_3d_gd_story():
    pack = _ch3_lik_3d_gd_pack()
    frames = []
    _ch3_lik_append_gd_frames(frames, pack)
    return _ch3_lik_story_hold(frames)


def ch4_export_likelihood_notation_nll():
    frames = ch3_build_frames_likelihood_ch4_nll_story()
    fn = "ch4_03_likelihood_notation_nll.mp4"
    save_mp4(frames, fn, duration=int(CH3_LIK_CH4_MS))
    print("wrote", OUTPUT_DIR / fn)
    return OUTPUT_DIR / fn


def ch4_export_likelihood_3d_measurements():
    frames = ch3_build_frames_likelihood_3d_measurements_story()
    fn = "ch4_04_likelihood_3d_measurements.mp4"
    save_mp4(frames, fn, duration=int(CH3_LIK_3D_MS))
    print("wrote", OUTPUT_DIR / fn)
    return OUTPUT_DIR / fn


def ch4_export_likelihood_3d_ball_vectors():
    frames = ch3_build_frames_likelihood_3d_ball_vectors_story()
    fn = "ch4_05_likelihood_3d_ball_vectors.mp4"
    save_mp4(frames, fn, duration=int(CH3_LIK_3D_MS_BALL))
    print("wrote", OUTPUT_DIR / fn)
    return OUTPUT_DIR / fn


def ch4_export_likelihood_3d_gd():
    from ch4_export_pipeline import export_mp4_from_specs

    pack = _ch3_lik_3d_gd_pack()
    return export_mp4_from_specs(
        pack,
        _ch3_lik_gd_frame_specs(pack),
        _ch3_lik_gd_render_frame,
        save_mp4=save_mp4,
        filename="ch4_06_likelihood_3d_gd.mp4",
        duration_ms=int(CH3_LIK_3D_MS_GD),
        story_hold_fn=_ch3_lik_story_hold,
        prewarm="gd",
        progress_label="ch4_06",
        render_fn_name="_ch3_lik_gd_render_frame",
    )


# --- ch4_05a: CT scan — NLL heatmap planes sweeping each 3-D axis ---

CH3_LIK_CT_N_HOLD = 4 if _CH3_DRAFT else max(12, _smooth_n(8))
CH3_LIK_CT_N_SWEEP = 8 if _CH3_DRAFT else max(48, _smooth_n(36))
CH3_LIK_CT_N_PIVOT = 6 if _CH3_DRAFT else max(36, _smooth_n(28))
CH3_LIK_CT_MS = 100 if not _CH3_DRAFT else 120
CH3_LIK_CT_AXES = ("st", "el", "b")
# (from_axis, to_axis): pivot uses shared edge at end of ``from`` sweep → start of ``to``
CH3_LIK_CT_PIVOTS = (("st", "el"), ("el", "b"))


def _ch4_ct_axis_limits(axis, bounds):
    dlo1, dhi1, dlo2, dhi2, dlob, dhib = bounds
    if axis == "st":
        return float(dlo1), float(dhi1)
    if axis == "el":
        return float(dlo2), float(dhi2)
    return float(dlob), float(dhib)


def _ch4_ct_sweep_end_value(axis, bounds):
    _, hi = _ch4_ct_axis_limits(axis, bounds)
    return float(hi)


def _ch4_ct_sweep_start_value(axis, bounds):
    lo, _ = _ch4_ct_axis_limits(axis, bounds)
    return float(lo)


def _ch4_ct_nll_at_grid(study, exam, y, w1, w2, b):
    return _ch3_nll_sum_on_flat_grid(
        study, exam, y,
        np.asarray(w1, dtype=np.float64).ravel(),
        np.asarray(w2, dtype=np.float64).ravel(),
        np.asarray(b, dtype=np.float64).ravel(),
    ).reshape(np.asarray(w1, dtype=np.float64).shape)


def _ch4_ct_sweep_mesh(axis, value, bounds, *, gn=None):
    """Axis-aligned slice mesh + NLL."""
    gn = int(CH3_LIK_CT_GRID if gn is None else gn)
    dlo1, dhi1, dlo2, dhi2, dlob, dhib = bounds
    val = float(value)
    if axis == "st":
        g2 = np.linspace(dlo2, dhi2, gn, dtype=np.float64)
        gb = np.linspace(dlob, dhib, gn, dtype=np.float64)
        w2, b = np.meshgrid(g2, gb, indexing="ij")
        w1 = np.full_like(w2, val)
    elif axis == "el":
        g1 = np.linspace(dlo1, dhi1, gn, dtype=np.float64)
        gb = np.linspace(dlob, dhib, gn, dtype=np.float64)
        w1, b = np.meshgrid(g1, gb, indexing="ij")
        w2 = np.full_like(w1, val)
    else:
        g1 = np.linspace(dlo1, dhi1, gn, dtype=np.float64)
        g2 = np.linspace(dlo2, dhi2, gn, dtype=np.float64)
        w1, w2 = np.meshgrid(g1, g2, indexing="ij")
        b = np.full_like(w1, val)
    return w1, w2, b


def _ch4_ct_pivot_mesh(study, exam, y, from_axis, to_axis, bounds, *, theta_u, gn=None):
    """
    Rotate slice plane from end of ``from_axis`` sweep to start of ``to_axis`` sweep.
    Pivot is the shared edge between the two axis-aligned planes (90° rotation).
    NLL is sampled at each vertex of the rotated plane.
    """
    gn = int(CH3_LIK_CT_GRID if gn is None else gn)
    dlo1, dhi1, dlo2, dhi2, dlob, dhib = bounds
    u = float(np.clip(float(theta_u), 0.0, 1.0))
    u = ch3_knob_smoothstep(u)
    ang = u * (np.pi / 2.0)

    if from_axis == "st" and to_axis == "el":
        # End: w_ST = hi. Start: w_EL = lo. Pivot edge: (hi, lo, b), axis ∥ b.
        hi = dhi1
        lo = dlo2
        w1s, w2s, bs = _ch4_ct_sweep_mesh("st", hi, bounds, gn=gn)
        dw = w2s - lo
        w1 = hi - np.sin(ang) * dw
        w2 = lo + np.cos(ang) * dw
        b = bs
    elif from_axis == "el" and to_axis == "b":
        # End: w_EL = hi. Start: b = lo. Pivot edge: (w_ST, hi, lo), axis ∥ w_ST.
        hi = dhi2
        lob = dlob
        w1, w2s, bs = _ch4_ct_sweep_mesh("el", hi, bounds, gn=gn)
        w2 = w2s
        db = bs - lob
        w2 = hi - np.sin(ang) * db
        b = lob + np.cos(ang) * db
    else:
        raise ValueError(f"unsupported CT pivot {from_axis!r} → {to_axis!r}")
    nll = _ch4_ct_nll_at_grid(study, exam, y, w1, w2, b)
    return w1, w2, b, nll


def _ch4_ct_draw_mesh(ax3d, w1, w2, b, nll, vmin=None, vmax=None):
    _ch4_nll_heatmap_plot_surface(ax3d, w1, w2, b, nll)


def _ch3_lik_ct_scan_pack():
    pack = _ch3_lik_3d_measurements_pack(ball_colormap_limits=False)
    pack["bounds"] = CH3_LIK_CT_VIEW_BOUNDS
    return pack


def _ch4_ct_right_blocks(*, pack=None):
    if pack is None:
        return []
    return ch4_nll_global_legend_blocks()


def ch3_frame_lik_ct_scan(
    pack, *,
    sweep_axis=None,
    plane_val=None,
    show_sweep_range=False,
    pivot_from=None,
    pivot_to=None,
    pivot_u=0.0,
    cam_azim_u=0.0,
):
    from ch4_layout import (
        CH4_FORMULAS_SECTION_TITLE,
        CH4_NOTATION_SECTION_TITLE,
        CH4_NLL_HEATMAP_SECTION_TITLE,
        ch4_cached_formula_blocks_3d_story,
        ch4_cached_notation_corner_blocks,
        ch4_knob_asset_pack,
        compose_tutorial,
    )

    study, exam, y = pack["study"], pack["exam"], pack["y"]
    bounds = pack["bounds"]
    handoff = _ch4_lik_02_03_handoff_state()
    ws, we = handoff["w_st"], handoff["w_el"]
    if str(sweep_axis) == "b" and plane_val is not None:
        bb = float(plane_val)
    else:
        bb = float(CH3_LIK_CH4_PLANE_B)
    fig, ax_data, ax3d, axes_k = ch4_figure_duo_weight3d()
    leg = legend_linear_equation_values_bold_param(ws, we, bb, "all")
    ch3_draw_left_panel(
        ax_data, ws, we, bb, study, exam, y, leg,
        show_colormap=True, highlight_mistakes_flag=False,
    )
    ax_data.set_xlim(*xlim)
    ax_data.set_ylim(*ylim)
    finalize_style_legend_tex(ax_data)
    knob_rgbs, canvas_sides = ch4_knob_asset_pack()
    ch3_draw_knob_row(
        fig, axes_k, ws, we, bb, "all",
        knob_rgbs, canvas_sides,
        rot_strip_deg=0.0, strip_scale=1.0,
        knob_rots=ch3_k1_knob_rots_at(ws, we, bb), knob_scales=[1.0, 1.0, 1.0], ax_data=ax_data,
    )
    dlo1, dhi1, dlo2, dhi2, dlob, dhib = bounds
    ax3d.set_autoscale_on(False)
    _ch3_lik_style_ax3d(ax3d, dlo1, dhi1, dlo2, dhi2, dlob, dhib)
    ch4_lik_ct_view_init(ax3d, cam_azim_u=float(cam_azim_u))
    if pivot_from is not None and pivot_to is not None:
        w1, w2, b, nll = _ch4_ct_pivot_mesh(
            study, exam, y, str(pivot_from), str(pivot_to), bounds, theta_u=float(pivot_u),
        )
    else:
        w1, w2, b = _ch4_ct_sweep_mesh(str(sweep_axis), float(plane_val), bounds)
        nll = _ch4_ct_nll_at_grid(study, exam, y, w1, w2, b)
    _ch4_ct_draw_mesh(ax3d, w1, w2, b, nll)
    _ch3_lik_style_ax3d(ax3d, dlo1, dhi1, dlo2, dhi2, dlob, dhib)
    plot_img = fig_to_image(fig, dpi=CH3_ANIM_DPI)
    plt.close(fig)
    return compose_tutorial(
        plot_img,
        right_blocks=_ch4_ct_right_blocks(pack=pack),
        bottom_blocks=ch4_cached_formula_blocks_3d_story(),
        corner_blocks=ch4_cached_notation_corner_blocks(),
        right_title=CH4_NLL_HEATMAP_SECTION_TITLE,
        right_title_single_line=True,
        bottom_title=CH4_FORMULAS_SECTION_TITLE,
        corner_title=CH4_NOTATION_SECTION_TITLE,
        write_progress=1.0,
        theme="classic_light",
    )


def _ch4_ct_append_pivot(frames, pack, from_axis, to_axis):
    for tv in np.linspace(0.0, 1.0, CH3_LIK_CT_N_PIVOT, endpoint=True):
        frames.append(ch3_frame_lik_ct_scan(
            pack,
            pivot_from=str(from_axis),
            pivot_to=str(to_axis),
            pivot_u=float(tv),
        ))


def ch3_build_frames_likelihood_ct_scan_story():
    pack = _ch3_lik_ct_scan_pack()
    frames = []
    bounds = pack["bounds"]
    pivot_map = dict(CH3_LIK_CT_PIVOTS)
    for _ in range(CH3_LIK_CT_N_HOLD):
        frames.append(ch3_frame_lik_ct_scan(
            pack, sweep_axis="b", plane_val=float(CH3_LIK_CH4_PLANE_B), show_sweep_range=False,
        ))
    for i, axis in enumerate(CH3_LIK_CT_AXES):
        if i > 0:
            prev = CH3_LIK_CT_AXES[i - 1]
            nxt = pivot_map.get(prev)
            if nxt == axis:
                _ch4_ct_append_pivot(frames, pack, prev, axis)
        lo, hi = _ch4_ct_axis_limits(axis, bounds)
        for _ in range(CH3_LIK_CT_N_HOLD):
            frames.append(ch3_frame_lik_ct_scan(
                pack, sweep_axis=axis, plane_val=lo, show_sweep_range=True,
            ))
        for tv in np.linspace(0.0, 1.0, CH3_LIK_CT_N_SWEEP, endpoint=True):
            u = ch3_knob_smoothstep(float(tv))
            val = lo + u * (hi - lo)
            frames.append(ch3_frame_lik_ct_scan(
                pack, sweep_axis=axis, plane_val=val, show_sweep_range=False,
            ))
        for _ in range(max(4, CH3_LIK_CT_N_HOLD // 3)):
            frames.append(ch3_frame_lik_ct_scan(
                pack, sweep_axis=axis, plane_val=hi, show_sweep_range=False,
            ))
    return _ch3_lik_story_hold(frames)


def ch4_preview_likelihood_ct_scan_frame(*, axis="st", plane_u=0.5, show_sweep_range=False, pivot_u=None):
    pack = _ch3_lik_ct_scan_pack()
    bounds = pack["bounds"]
    if pivot_u is not None:
        if axis == "el":
            return ch3_frame_lik_ct_scan(
                pack, pivot_from="st", pivot_to="el", pivot_u=float(pivot_u),
            )
        if axis == "b":
            return ch3_frame_lik_ct_scan(
                pack, pivot_from="el", pivot_to="b", pivot_u=float(pivot_u),
            )
    lo, hi = _ch4_ct_axis_limits(axis, bounds)
    val = lo + float(plane_u) * (hi - lo)
    return ch3_frame_lik_ct_scan(
        pack, sweep_axis=axis, plane_val=val, show_sweep_range=show_sweep_range,
    )


def ch4_export_likelihood_ct_scan():
    frames = ch3_build_frames_likelihood_ct_scan_story()
    fn = "ch4_05a_likelihood_3d_ct_scan.mp4"
    save_mp4(frames, fn, duration=int(CH3_LIK_CT_MS))
    print("wrote", OUTPUT_DIR / fn, f"({len(frames)} frames)")
    return OUTPUT_DIR / fn


# --- ch4_05a2: diagonal voxel fill — checkerboard cubes swept along (−3,3,−3)→(3,−3,3) ---

CH3_LIK_VOXEL_GRID = CH3_LIK_CT_GRID
CH3_LIK_VOXEL_CELL_MULT = 1
CH3_LIK_VOXEL_N_HOLD = 4 if _CH3_DRAFT else max(12, _smooth_n(8))
CH3_LIK_VOXEL_N_SWEEP = 8 if _CH3_DRAFT else max(64, _smooth_n(48))
CH3_LIK_VOXEL_MS = 100 if not _CH3_DRAFT else 120
CH3_LIK_VOXEL_ALPHA = 0.92
# Diagonal sweep: (−3, 3, −3) → (3, −3, 3) in (w_ST, w_EL, b).
CH3_LIK_VOXEL_DIAG_START = (-3.0, 3.0, -3.0)
CH3_LIK_VOXEL_DIAG_END = (3.0, -3.0, 3.0)


def _ch4_voxel_n_cells(cell_mult=None):
    """Voxel count per axis; ``cell_mult=4`` → cubes ¼ the edge length of ``cell_mult=1``."""
    mult = int(CH3_LIK_VOXEL_CELL_MULT if cell_mult is None else cell_mult)
    return max((int(CH3_LIK_CT_GRID) - 1) * mult, 2)


def _ch4_voxel_fill_cache_key(cell_mult, gap_pitch=1):
    gap_pitch = int(gap_pitch)
    if gap_pitch <= 1:
        return f"voxel_fill_cache_x{int(cell_mult)}"
    return f"voxel_fill_cache_x{int(cell_mult)}_g{gap_pitch}"


def _ch4_voxel_checker_mask(n, *, gap_pitch=1):
    """Every-other cells; ``gap_pitch=3`` → 3× center spacing (2× face gap vs pitch 1)."""
    ii, jj, kk = np.indices((int(n), int(n), int(n)))
    if int(gap_pitch) <= 1:
        return ((ii + jj + kk) % 2) == 0
    p = int(gap_pitch)
    mid = p // 2
    bi, bj, bk = ii // p, jj // p, kk // p
    at_block_center = (ii % p == mid) & (jj % p == mid) & (kk % p == mid)
    return at_block_center & ((bi + bj + bk) % 2 == 0)


def _ch4_voxel_cell_edges(lo, hi, n_cells):
    """``n_cells`` voxels along [lo, hi] — same spacing as 05a heatmap grid lines."""
    n_cells = int(n_cells)
    return np.linspace(float(lo), float(hi), n_cells + 1, dtype=np.float64)


def _ch4_voxel_fill_warm_pack(pack, *, cell_mult=None):
    """Precompute NLL + colors on a 3-D checkerboard cell grid (once per export)."""
    if cell_mult is None:
        cell_mult = int(pack.get("voxel_cell_mult", CH3_LIK_VOXEL_CELL_MULT))
    else:
        cell_mult = int(cell_mult)
    gap_pitch = int(pack.get("voxel_gap_pitch", 1))
    cache_key = _ch4_voxel_fill_cache_key(cell_mult, gap_pitch)
    cache = pack.get(cache_key)
    if cache is not None:
        return cache

    from ch4_layout import ch4_nll_heatmap_cmap

    study, exam, y = pack["study"], pack["exam"], pack["y"]
    dlo1, dhi1, dlo2, dhi2, dlob, dhib = CH3_LIK_CT_VIEW_BOUNDS
    n = _ch4_voxel_n_cells(cell_mult)
    x_edges = _ch4_voxel_cell_edges(dlo1, dhi1, n)
    y_edges = _ch4_voxel_cell_edges(dlo2, dhi2, n)
    z_edges = _ch4_voxel_cell_edges(dlob, dhib, n)
    w1 = 0.5 * (x_edges[:-1] + x_edges[1:])
    w2 = 0.5 * (y_edges[:-1] + y_edges[1:])
    bb = 0.5 * (z_edges[:-1] + z_edges[1:])
    W1, W2, B = np.meshgrid(w1, w2, bb, indexing="ij")
    nll = _ch3_nll_sum_on_flat_grid(
        study, exam, y,
        W1.ravel(), W2.ravel(), B.ravel(),
    ).reshape(W1.shape)
    lo, hi = ch4_nll_global_scale()
    cmap = ch4_nll_heatmap_cmap()
    span = max(float(hi) - float(lo), 1e-9)
    normed = np.clip((nll - float(lo)) / span, 0.0, 1.0)
    rgba = cmap(normed)
    rgba[..., 3] = float(CH3_LIK_VOXEL_ALPHA)
    ii, jj, kk = np.indices((n, n, n))
    checker = _ch4_voxel_checker_mask(n, gap_pitch=gap_pitch)
    start = np.array(CH3_LIK_VOXEL_DIAG_START, dtype=np.float64)
    end = np.array(CH3_LIK_VOXEL_DIAG_END, dtype=np.float64)
    diag = end - start
    max_proj = float(np.dot(diag, diag))
    proj = (W1 - start[0]) * diag[0] + (W2 - start[1]) * diag[1] + (B - start[2]) * diag[2]
    cache = {
        "n": n,
        "x_edges": x_edges,
        "y_edges": y_edges,
        "z_edges": z_edges,
        "checker": checker,
        "proj": proj,
        "max_proj": max_proj,
        "diag_start": start,
        "diag": diag,
        "rgba": rgba,
        "bounds": CH3_LIK_CT_VIEW_BOUNDS,
        "cell_mult": cell_mult,
        "gap_pitch": gap_pitch,
        "nll_lo": float(lo),
        "nll_hi": float(hi),
        "right_blocks": ch4_nll_global_legend_blocks(),
    }
    pack[cache_key] = cache
    return cache


def _ch3_lik_voxel_fill_pack(*, cell_mult=1, gap_pitch=1):
    pack = _ch3_lik_3d_measurements_pack(ball_colormap_limits=False)
    pack["bounds"] = CH3_LIK_CT_VIEW_BOUNDS
    pack["voxel_cell_mult"] = int(cell_mult)
    pack["voxel_gap_pitch"] = int(gap_pitch)
    _ch4_voxel_fill_warm_pack(pack, cell_mult=cell_mult)
    return pack


def _ch4_voxel_fill_cut_plane_mesh(cache, sweep_u, *, gn=None):
    """Cutting plane orthogonal to the voxel diagonal sweep (same mesh as CT slices)."""
    gn = int(CH3_LIK_CT_GRID if gn is None else gn)
    start = cache["diag_start"]
    diag = cache["diag"]
    max_proj = float(cache["max_proj"])
    dlo1, dhi1, dlo2, dhi2, dlob, dhib = cache["bounds"]
    u = float(np.clip(float(sweep_u), 0.0, 1.0))
    u = ch3_knob_smoothstep(u)
    target = u * max_proj
    g1 = np.linspace(dlo1, dhi1, gn, dtype=np.float64)
    g2 = np.linspace(dlo2, dhi2, gn, dtype=np.float64)
    w1, w2 = np.meshgrid(g1, g2, indexing="ij")
    b = (
        target
        - (w1 - start[0]) * diag[0]
        - (w2 - start[1]) * diag[1]
    ) / float(diag[2]) + start[2]
    return w1, w2, b


def _ch4_voxel_fill_draw_cut_plane(ax3d, cache, sweep_u):
    w1, w2, b = _ch4_voxel_fill_cut_plane_mesh(cache, sweep_u)
    dlo1, dhi1, dlo2, dhi2, dlob, dhib = cache["bounds"]
    mask = (
        (b >= dlob - 1e-9) & (b <= dhib + 1e-9)
        & (w1 >= dlo1 - 1e-9) & (w1 <= dhi1 + 1e-9)
        & (w2 >= dlo2 - 1e-9) & (w2 <= dhi2 + 1e-9)
    )
    if not np.any(mask):
        return
    face = np.zeros(w1.shape + (4,), dtype=np.float64)
    face[..., :] = (0.85, 0.88, 0.92, float(CH3_LIK_CT_PLANE_ALPHA))
    face[~mask] = (0, 0, 0, 0)
    ax3d.plot_surface(
        w1, w2, b,
        facecolors=face,
        rstride=1,
        cstride=1,
        linewidth=0,
        antialiased=False,
        shade=False,
        zorder=5,
    )


def _ch4_voxel_fill_draw_voxels(ax3d, cache, sweep_u):
    u = float(np.clip(float(sweep_u), 0.0, 1.0))
    u = ch3_knob_smoothstep(u)
    front = u * float(cache["max_proj"])
    visible = (cache["proj"] <= front + 1e-9) & cache["checker"]
    if not np.any(visible):
        return
    x_edges = cache["x_edges"]
    y_edges = cache["y_edges"]
    z_edges = cache["z_edges"]
    dx = float(x_edges[1] - x_edges[0])
    dy = float(y_edges[1] - y_edges[0])
    dz = float(z_edges[1] - z_edges[0])
    idx = np.argwhere(visible)
    xs = x_edges[idx[:, 0]]
    ys = y_edges[idx[:, 1]]
    zs = z_edges[idx[:, 2]]
    colors = cache["rgba"][visible]
    ax3d.bar3d(
        xs, ys, zs, dx, dy, dz,
        color=colors,
        shade=False,
        linewidth=0.0,
        edgecolor=(0.0, 0.0, 0.0, 0.0),
        zorder=6,
    )


def ch3_frame_lik_voxel_fill(pack, *, sweep_u=0.0, show_cut_plane=True, cam_azim_u=0.0, cam_spin_deg=0.0):
    from ch4_layout import (
        CH4_FORMULAS_SECTION_TITLE,
        CH4_NOTATION_SECTION_TITLE,
        CH4_NLL_HEATMAP_SECTION_TITLE,
        ch4_cached_formula_blocks_3d_story,
        ch4_cached_notation_corner_blocks,
        ch4_knob_asset_pack,
        compose_tutorial,
    )

    cache = _ch4_voxel_fill_warm_pack(pack)
    bounds = cache["bounds"]
    study, exam, y = pack["study"], pack["exam"], pack["y"]
    handoff = _ch4_lik_02_03_handoff_state()
    ws, we, bb = handoff["w_st"], handoff["w_el"], float(CH3_LIK_CH4_PLANE_B)
    fig, ax_data, ax3d, axes_k = ch4_figure_duo_weight3d()
    leg = legend_linear_equation_values_bold_param(ws, we, bb, "all")
    ch3_draw_left_panel(
        ax_data, ws, we, bb, study, exam, y, leg,
        show_colormap=True, highlight_mistakes_flag=False,
    )
    ax_data.set_xlim(*xlim)
    ax_data.set_ylim(*ylim)
    finalize_style_legend_tex(ax_data)
    knob_rgbs, canvas_sides = ch4_knob_asset_pack()
    ch3_draw_knob_row(
        fig, axes_k, ws, we, bb, "all",
        knob_rgbs, canvas_sides,
        rot_strip_deg=0.0, strip_scale=1.0,
        knob_rots=ch3_k1_knob_rots_at(ws, we, bb), knob_scales=[1.0, 1.0, 1.0], ax_data=ax_data,
    )
    dlo1, dhi1, dlo2, dhi2, dlob, dhib = bounds
    ax3d.set_autoscale_on(False)
    _ch3_lik_style_ax3d(ax3d, dlo1, dhi1, dlo2, dhi2, dlob, dhib)
    ch4_lik_ct_view_init(ax3d, cam_azim_u=float(cam_azim_u), cam_spin_deg=float(cam_spin_deg))
    _ch4_voxel_fill_draw_voxels(ax3d, cache, sweep_u)
    if show_cut_plane and float(sweep_u) < 1.0 - 1e-6:
        _ch4_voxel_fill_draw_cut_plane(ax3d, cache, sweep_u)
    _ch3_lik_style_ax3d(ax3d, dlo1, dhi1, dlo2, dhi2, dlob, dhib)
    plot_img = fig_to_image(fig, dpi=CH3_ANIM_DPI)
    plt.close(fig)
    return compose_tutorial(
        plot_img,
        right_blocks=cache.get("right_blocks", []),
        bottom_blocks=ch4_cached_formula_blocks_3d_story(),
        corner_blocks=ch4_cached_notation_corner_blocks(),
        right_title=CH4_NLL_HEATMAP_SECTION_TITLE,
        right_title_single_line=True,
        bottom_title=CH4_FORMULAS_SECTION_TITLE,
        corner_title=CH4_NOTATION_SECTION_TITLE,
        write_progress=1.0,
        theme="classic_light",
    )


def ch3_build_frames_likelihood_voxel_fill_story():
    pack = _ch3_lik_voxel_fill_pack()
    frames = []
    for _ in range(CH3_LIK_VOXEL_N_HOLD):
        frames.append(ch3_frame_lik_voxel_fill(pack, sweep_u=0.0))
    for tv in np.linspace(0.0, 1.0, CH3_LIK_VOXEL_N_SWEEP, endpoint=True):
        frames.append(ch3_frame_lik_voxel_fill(pack, sweep_u=float(tv)))
    for _ in range(max(6, CH3_LIK_VOXEL_N_HOLD // 2)):
        frames.append(ch3_frame_lik_voxel_fill(pack, sweep_u=1.0, show_cut_plane=False))
    return _ch3_lik_story_hold(frames)


def ch4_preview_likelihood_voxel_fill_frame(*, sweep_u=0.45):
    pack = _ch3_lik_voxel_fill_pack()
    return ch3_frame_lik_voxel_fill(pack, sweep_u=float(sweep_u))


def ch4_export_likelihood_voxel_fill():
    frames = ch3_build_frames_likelihood_voxel_fill_story()
    fn = "ch4_05a2_diagonal_voxel_fill.mp4"
    save_mp4(frames, fn, duration=int(CH3_LIK_VOXEL_MS))
    print("wrote", OUTPUT_DIR / fn, f"({len(frames)} frames)")
    return OUTPUT_DIR / fn


# --- ch4_05a3: fine diagonal voxel fill (¼ edge length → 4× cells per axis) ---

CH3_LIK_VOXEL_FINE_CELL_MULT = 4
CH3_LIK_VOXEL_FINE_GAP_PITCH = 3  # 2× face gap vs default checker (gap = 2·dx, cube size unchanged)
CH3_LIK_VOXEL_FINE_MS = 100 if not _CH3_DRAFT else 120
CH3_LIK_VOXEL_FINE_N_SPIN = 8 if _CH3_DRAFT else max(24, _smooth_n(18))


def _ch3_lik_voxel_fill_fine_pack():
    return _ch3_lik_voxel_fill_pack(
        cell_mult=CH3_LIK_VOXEL_FINE_CELL_MULT,
        gap_pitch=CH3_LIK_VOXEL_FINE_GAP_PITCH,
    )


def ch3_frame_lik_voxel_fill_fine(pack, *, sweep_u=0.0, show_cut_plane=True, cam_azim_u=0.0, cam_spin_deg=0.0):
    return ch3_frame_lik_voxel_fill(
        pack,
        sweep_u=sweep_u,
        show_cut_plane=show_cut_plane,
        cam_azim_u=cam_azim_u,
        cam_spin_deg=cam_spin_deg,
    )


def ch3_build_frames_likelihood_voxel_fill_fine_story():
    pack = _ch3_lik_voxel_fill_fine_pack()
    frames = []
    for _ in range(CH3_LIK_VOXEL_N_HOLD):
        frames.append(ch3_frame_lik_voxel_fill_fine(pack, sweep_u=0.0))
    for tv in np.linspace(0.0, 1.0, CH3_LIK_VOXEL_N_SWEEP, endpoint=True):
        frames.append(ch3_frame_lik_voxel_fill_fine(pack, sweep_u=float(tv)))
    for _ in range(max(6, CH3_LIK_VOXEL_N_HOLD // 2)):
        frames.append(ch3_frame_lik_voxel_fill_fine(pack, sweep_u=1.0, show_cut_plane=False))
    spin_n = max(int(CH3_LIK_VOXEL_FINE_N_SPIN), 2)
    for tv in np.linspace(0.0, 1.0, spin_n, endpoint=True):
        frames.append(ch3_frame_lik_voxel_fill_fine(
            pack,
            sweep_u=1.0,
            show_cut_plane=False,
            cam_azim_u=float(tv),
            cam_spin_deg=360.0,
        ))
    return _ch3_lik_story_hold(frames)


def ch4_preview_likelihood_voxel_fill_fine_frame(*, sweep_u=0.45):
    pack = _ch3_lik_voxel_fill_fine_pack()
    return ch3_frame_lik_voxel_fill_fine(pack, sweep_u=float(sweep_u))


def ch4_export_likelihood_voxel_fill_fine():
    frames = ch3_build_frames_likelihood_voxel_fill_fine_story()
    fn = "ch4_05a3_diagonal_voxel_fill_fine.mp4"
    save_mp4(frames, fn, duration=int(CH3_LIK_VOXEL_FINE_MS))
    print("wrote", OUTPUT_DIR / fn, f"({len(frames)} frames)")
    return OUTPUT_DIR / fn


# --- ch4_05a4: ball-sized voxel cube at path start → accumulate voxels along ch4_05 path ---

CH3_LIK_BALL_VOXEL_N_GROW = 8 if _CH3_DRAFT else 44
CH3_LIK_BALL_VOXEL_N_POINT_SHRINK = 4 if _CH3_DRAFT else 14
CH3_LIK_BALL_VOXEL_POINT_S_LARGE = 320.0
CH3_LIK_BALL_VOXEL_POINT_S_SMALL = 28.0
CH3_LIK_BALL_VOXEL_POINT_COLOR = "#111111"
CH3_LIK_BALL_VOXEL_MS = int(CH3_LIK_3D_MS_BALL)
CH3_LIK_BALL_VOXEL_HALF_SCALE = 0.5
CH3_LIK_BALL_VOXEL_N_ROT_AFTER = 8 if _CH3_DRAFT else 40
# (w_ST, w_EL, b) — three straight segments across the ±3 cube
CH3_LIK_BALL_VOXEL_PATH_WAYPOINTS = (
    (-3.0, 3.0, -3.0),
    (3.0, 0.0, 3.0),
    (-3.0, -3.0, -3.0),
    (3.0, -3.0, -3.0),
)
CH3_LIK_BALL_VOXEL_N_PATH_PER_SEG = 8 if _CH3_DRAFT else 40


def _ch4_ball_voxel_path_points(n_pts_per_seg=None):
    """Piecewise-linear path with exact waypoint hits at segment junctions."""
    n = int(CH3_LIK_BALL_VOXEL_N_PATH_PER_SEG if n_pts_per_seg is None else n_pts_per_seg)
    waypoints = np.asarray(CH3_LIK_BALL_VOXEL_PATH_WAYPOINTS, dtype=float)
    parts = []
    for i in range(len(waypoints) - 1):
        seg = np.linspace(waypoints[i], waypoints[i + 1], n, endpoint=True)
        if i > 0:
            seg = seg[1:]
        parts.append(seg)
    path = np.vstack(parts).astype(float)
    waypoint_path_indices = [0]
    offset = 0
    for part in parts:
        offset += len(part)
        waypoint_path_indices.append(offset - 1)
    return path, np.asarray(waypoint_path_indices, dtype=int)


def _ch4_ball_voxel_path_us():
    """Equal animation time on each of the three straight segments."""
    n_seg = len(CH3_LIK_BALL_VOXEL_PATH_WAYPOINTS) - 1
    n_frames = max(CH3_LIK_3D_N_PATH // n_seg, 8 if _CH3_DRAFT else 20)
    us = []
    for seg_i in range(n_seg):
        for tv in np.linspace(0.0, 1.0, n_frames, endpoint=True):
            us.append((seg_i + float(tv)) / float(n_seg))
    return us


def _ch4_ball_voxel_path_index(pack, path_u):
    """Map global path_u in [0, 1] to a path sample index (waypoints at u=0, 1/3, 2/3, 1)."""
    path_u = float(np.clip(float(path_u), 0.0, 1.0))
    wp_idx = np.asarray(pack["path_waypoint_indices"], dtype=int)
    n_seg = len(wp_idx) - 1
    seg_f = path_u * float(n_seg)
    seg_i = min(int(np.floor(seg_f)), n_seg - 1)
    t = seg_f - float(seg_i)
    i0 = int(wp_idx[seg_i])
    i1 = int(wp_idx[seg_i + 1])
    return int(round(i0 + t * float(i1 - i0)))


def _ch4_ball_voxel_measurements_pack():
    """Like ``_ch3_lik_3d_measurements_pack`` but with the 05a4 corner-to-corner path."""
    pack = _ch3_lik_3d_measurements_pack(ball_colormap_limits=False)
    bounds = _ch4_ball_voxel_view_bounds()
    path, waypoint_path_indices = _ch4_ball_voxel_path_points()
    global_cache = _ch4_ball_voxel_global_cache(pack)
    ball_vmin = float(global_cache["nll_lo"])
    ball_vmax = float(global_cache["nll_hi"])
    study, exam, y = pack["study"], pack["exam"], pack["y"]
    path_nll = np.array([
        float(-loss_log_likelihood(float(r[0]), float(r[1]), float(r[2]), study, exam, y))
        for r in path
    ], dtype=float)
    span = max(float(ball_vmax) - float(ball_vmin), 1e-9)
    from ch4_layout import ch4_nll_heatmap_cmap

    nll_cmap = ch4_nll_heatmap_cmap()
    pack["path"] = path
    pack["path_waypoint_indices"] = waypoint_path_indices
    pack["path_us"] = _ch4_ball_voxel_path_us()
    pack["path_colors"] = [
        nll_cmap(float(np.clip((v - ball_vmin) / span, 0.0, 1.0)))
        for v in path_nll
    ]
    pack["ball_vmin"] = ball_vmin
    pack["ball_vmax"] = ball_vmax
    pack["ax3d_bounds"] = bounds
    pack["gd_ax3d_bounds"] = bounds
    pack.pop("ball_voxel_ready", None)
    return pack


def _ch4_ball_voxel_view_bounds():
    """Same ±3 cube as ch4_05a3."""
    return CH3_LIK_CT_VIEW_BOUNDS


def _ch4_ball_voxel_global_cache(pack):
    """05a3 fine grid: same bounds, cell size, gap pitch, and NLL colors."""
    cache = pack.get("ball_voxel_global_cache")
    if cache is not None:
        return cache
    cell_mult = int(CH3_LIK_VOXEL_FINE_CELL_MULT)
    gap_pitch = int(CH3_LIK_VOXEL_FINE_GAP_PITCH)
    pack["voxel_cell_mult"] = cell_mult
    pack["voxel_gap_pitch"] = gap_pitch
    cache = _ch4_voxel_fill_warm_pack(pack, cell_mult=cell_mult)
    pack["ball_voxel_global_cache"] = cache
    return cache


def _ch4_ball_voxel_half():
    """Ball framing half-extent using the 05a3 view span."""
    bounds = _ch4_ball_voxel_view_bounds()
    span_ref = float(np.max(_ch3_lik_bounds_span(bounds)))
    r = float(CH3_LIK_3D_BALL_R_SCALE) * span_ref
    return max(r * 2.35, span_ref * 0.055) * float(CH3_LIK_BALL_VOXEL_HALF_SCALE)


def _ch4_ball_voxel_build_cube(global_cache, center, half):
    """Ball-sized subset of the 05a3 fine checkerboard grid."""
    center = np.asarray(center, dtype=np.float64)
    half = float(half)
    x_edges = global_cache["x_edges"]
    y_edges = global_cache["y_edges"]
    z_edges = global_cache["z_edges"]
    w1 = 0.5 * (x_edges[:-1] + x_edges[1:])
    w2 = 0.5 * (y_edges[:-1] + y_edges[1:])
    bb = 0.5 * (z_edges[:-1] + z_edges[1:])
    W1, W2, B = np.meshgrid(w1, w2, bb, indexing="ij")
    in_ball = (
        (np.abs(W1 - center[0]) <= half)
        & (np.abs(W2 - center[1]) <= half)
        & (np.abs(B - center[2]) <= half)
    )
    checker = global_cache["checker"] & in_ball
    ci = int(np.argmin(np.abs(w1 - center[0])))
    cj = int(np.argmin(np.abs(w2 - center[1])))
    ck = int(np.argmin(np.abs(bb - center[2])))
    n = int(global_cache["n"])
    ii, jj, kk = np.indices((n, n, n))
    cheb = np.maximum(np.maximum(np.abs(ii - ci), np.abs(jj - cj)), np.abs(kk - ck))
    max_cheb = int(cheb[checker].max()) if np.any(checker) else 0
    return {
        "x_edges": x_edges,
        "y_edges": y_edges,
        "z_edges": z_edges,
        "checker": checker,
        "cheb": cheb,
        "max_cheb": max_cheb,
        "rgba": global_cache["rgba"],
        "dx": float(x_edges[1] - x_edges[0]),
        "dy": float(y_edges[1] - y_edges[0]),
        "dz": float(z_edges[1] - z_edges[0]),
    }


def _ch4_ball_voxel_cube_draw(
    cube, *, grow_u=1.0, alpha_scale=1.0, emphasize_cheb=None, dim_others=1.0,
):
    grow_u = float(np.clip(float(grow_u), 0.0, 1.0))
    max_cheb = float(cube["max_cheb"])
    lim = grow_u * max_cheb + 1e-9
    visible = cube["checker"] & (cube["cheb"] <= lim)
    if not np.any(visible):
        return None
    idx = np.argwhere(visible)
    xs = cube["x_edges"][idx[:, 0]]
    ys = cube["y_edges"][idx[:, 1]]
    zs = cube["z_edges"][idx[:, 2]]
    colors = cube["rgba"][visible].copy()
    if alpha_scale < 1.0 - 1e-6:
        colors[..., 3] *= float(alpha_scale)
    if emphasize_cheb is not None and float(dim_others) < 1.0 - 1e-6:
        emp = cube["cheb"][visible] == float(emphasize_cheb)
        colors[~emp, 3] *= float(dim_others)
    return {
        "xs": xs,
        "ys": ys,
        "zs": zs,
        "dx": cube["dx"],
        "dy": cube["dy"],
        "dz": cube["dz"],
        "colors": colors,
    }


def _ch4_ball_voxel_merge_draws(draws):
    draws = [d for d in draws if d is not None]
    if not draws:
        return None
    if len(draws) == 1:
        return draws[0]
    return {
        "xs": np.concatenate([d["xs"] for d in draws]),
        "ys": np.concatenate([d["ys"] for d in draws]),
        "zs": np.concatenate([d["zs"] for d in draws]),
        "dx": draws[0]["dx"],
        "dy": draws[0]["dy"],
        "dz": draws[0]["dz"],
        "colors": np.concatenate([d["colors"] for d in draws], axis=0),
    }


def _ch4_ball_voxel_prewarm_pack(pack):
    if pack.get("ball_voxel_ready"):
        return pack
    global_cache = _ch4_ball_voxel_global_cache(pack)
    half = _ch4_ball_voxel_half()
    path = np.asarray(pack["path"], dtype=float)
    start_cube = _ch4_ball_voxel_build_cube(global_cache, path[0], half)
    path_cubes = [
        _ch4_ball_voxel_build_cube(global_cache, r, half)
        for r in path
    ]
    pack["ball_voxel_half"] = half
    pack["ball_voxel_start_cube"] = start_cube
    pack["ball_voxel_path_cubes"] = path_cubes
    pack["ball_voxel_right"] = global_cache["right_blocks"]
    pack["ball_voxel_ready"] = True
    return pack


def _ch4_ball_voxel_path_draw(pack, path_index, *, grow_u=1.0, alpha_scale=1.0):
    cubes = pack["ball_voxel_path_cubes"]
    i = int(np.clip(int(path_index), 0, len(cubes) - 1))
    draws = [_ch4_ball_voxel_cube_draw(cubes[j], grow_u=1.0, alpha_scale=alpha_scale) for j in range(i + 1)]
    return _ch4_ball_voxel_merge_draws(draws)


def _ch4_ball_voxel_frame(
    pack, ws, we, bb, *,
    ax3d_bounds,
    wide_bounds,
    voxel_draw=None,
    point_s=None,
    cam_azim_u=0.0,
    cam_rot_deg=0.0,
    measurements=None,
    right_blocks=None,
    right_title=None,
    right_title_single_line=None,
):
    if right_blocks is not None:
        right = right_blocks
    elif measurements is not None:
        right = measurements
    else:
        right = pack.get("ball_voxel_right", [])
    return ch3_frame_lik_weight3d_measurements(
        pack["study"], pack["exam"], pack["y"], ws, we, bb,
        W1m=pack["W1m"], W2m=pack["W2m"], Bm=pack["Bm"], Lf=pack["Lf"],
        vmin=pack["vmin"], vmax=pack["vmax"],
        path_xyz=pack["path"],
        path_u=0.0,
        notation_condensed=True,
        measurements=right,
        ax3d_bounds=ax3d_bounds,
        wide_bounds=wide_bounds,
        show_ball_vectors=False,
        ball_vmin=pack["ball_vmin"],
        ball_vmax=pack["ball_vmax"],
        cam_azim_u=float(cam_azim_u),
        cam_rot_deg=float(cam_rot_deg),
        show_path_line=False,
        voxel_draw=voxel_draw,
        point_s=point_s,
        point_color=CH3_LIK_BALL_VOXEL_POINT_COLOR,
        write_progress=1.0,
        right_title=right_title,
        right_title_single_line=right_title_single_line,
    )


def _ch3_lik_ball_voxel_intro_frames(pack, *, skip_prewarm=False):
    if not skip_prewarm:
        _ch4_ball_voxel_prewarm_pack(pack)
    path = pack["path"]
    ws0, we0, bb0 = float(path[0, 0]), float(path[0, 1]), float(path[0, 2])
    bounds = _ch4_ball_voxel_view_bounds()
    start_cube = pack["ball_voxel_start_cube"]
    frames = []

    for _ in range(max(4, CH3_LIK_VOXEL_N_HOLD // 3)):
        frames.append(_ch4_ball_voxel_frame(
            pack, ws0, we0, bb0,
            ax3d_bounds=bounds,
            wide_bounds=bounds,
            voxel_draw=None,
            point_s=CH3_LIK_BALL_VOXEL_POINT_S_LARGE,
            right_blocks=pack["ball_voxel_right"],
            right_title=CH4_NLL_HEATMAP_SECTION_TITLE,
            right_title_single_line=True,
        ))

    for tv in np.linspace(0.0, 1.0, CH3_LIK_BALL_VOXEL_N_GROW, endpoint=True):
        u = ch3_knob_smoothstep(float(tv))
        frames.append(_ch4_ball_voxel_frame(
            pack, ws0, we0, bb0,
            ax3d_bounds=bounds,
            wide_bounds=bounds,
            voxel_draw=_ch4_ball_voxel_cube_draw(start_cube, grow_u=u),
            point_s=CH3_LIK_BALL_VOXEL_POINT_S_LARGE,
            right_blocks=pack["ball_voxel_right"],
            right_title=CH4_NLL_HEATMAP_SECTION_TITLE,
            right_title_single_line=True,
        ))

    for tv in np.linspace(0.0, 1.0, CH3_LIK_BALL_VOXEL_N_POINT_SHRINK, endpoint=True):
        u = ch3_knob_smoothstep(float(tv))
        ps = ch3_lerp(CH3_LIK_BALL_VOXEL_POINT_S_LARGE, CH3_LIK_BALL_VOXEL_POINT_S_SMALL, u)
        frames.append(_ch4_ball_voxel_frame(
            pack, ws0, we0, bb0,
            ax3d_bounds=bounds,
            wide_bounds=bounds,
            voxel_draw=_ch4_ball_voxel_cube_draw(start_cube, grow_u=1.0),
            point_s=ps,
            right_blocks=pack["ball_voxel_right"],
            right_title=CH4_NLL_HEATMAP_SECTION_TITLE,
            right_title_single_line=True,
        ))

    return frames


def _ch3_lik_append_ball_voxel_path_frames(frames, pack, *, rotate_during=True):
    _ch4_ball_voxel_prewarm_pack(pack)
    path = pack["path"]
    path_us = pack["path_us"]
    bounds = _ch4_ball_voxel_view_bounds()
    n_path = max(len(path_us) - 1, 1)
    for i, pu in enumerate(path_us):
        pu = float(pu)
        idx = _ch4_ball_voxel_path_index(pack, pu)
        ws_i, we_i, bb_i = path[idx]
        cam_u = float(i) / float(n_path) if rotate_during else 0.0
        cam_rot = float(CH3_LIK_3D_CAM_PATH_ROT) if rotate_during else 0.0
        frames.append(_ch4_ball_voxel_frame(
            pack, float(ws_i), float(we_i), float(bb_i),
            ax3d_bounds=bounds,
            wide_bounds=bounds,
            voxel_draw=_ch4_ball_voxel_path_draw(pack, idx),
            point_s=CH3_LIK_BALL_VOXEL_POINT_S_SMALL,
            cam_azim_u=cam_u,
            cam_rot_deg=cam_rot,
            measurements=_ch3_lik_we_are_here_at(pack, float(ws_i), float(we_i), float(bb_i)),
        ))


def _ch3_lik_append_ball_voxel_rot_after(frames, pack):
    """Single 360° pan after the path completes (05a4 rot-after variant)."""
    _ch4_ball_voxel_prewarm_pack(pack)
    path = pack["path"]
    path_us = pack["path_us"]
    bounds = _ch4_ball_voxel_view_bounds()
    i_end = _ch4_ball_voxel_path_index(pack, float(path_us[-1]))
    ws_e, we_e, bb_e = path[i_end]
    for tv in np.linspace(0.0, 1.0, CH3_LIK_BALL_VOXEL_N_ROT_AFTER, endpoint=True):
        u = ch3_knob_smoothstep(float(tv))
        frames.append(_ch4_ball_voxel_frame(
            pack, float(ws_e), float(we_e), float(bb_e),
            ax3d_bounds=bounds,
            wide_bounds=bounds,
            voxel_draw=_ch4_ball_voxel_path_draw(pack, i_end),
            point_s=CH3_LIK_BALL_VOXEL_POINT_S_SMALL,
            cam_azim_u=u,
            cam_rot_deg=float(CH3_LIK_3D_CAM_PATH_ROT),
            measurements=_ch3_lik_we_are_here_at(pack, float(ws_e), float(we_e), float(bb_e)),
        ))


def _ch3_lik_append_05a4_handoff(frames, pack):
    path = pack["path"]
    path_us = pack["path_us"]
    bounds = _ch4_ball_voxel_view_bounds()
    n_path = max(len(path_us) - 1, 1)
    i_end = _ch3_lik_05_end_path_index(pack)
    pu_end = float(path_us[i_end])
    idx_end = _ch4_ball_voxel_path_index(pack, pu_end)
    ws_e, we_e, bb_e = path[idx_end]
    cam_u_end = float(i_end) / float(n_path)
    ws_t, we_t, bb_t = float(path[0, 0]), float(path[0, 1]), float(path[0, 2])
    for tv in np.linspace(0.0, 1.0, CH3_LIK_3D_N_05_HANDOFF, endpoint=True):
        u = ch3_knob_smoothstep(float(tv))
        ws = ws_e + u * (ws_t - ws_e)
        we = we_e + u * (we_t - we_e)
        bb = bb_e + u * (bb_t - bb_e)
        cam_u = cam_u_end + u * (0.0 - cam_u_end)
        alpha = max(0.0, 1.0 - 1.15 * u)
        frames.append(_ch4_ball_voxel_frame(
            pack, ws, we, bb,
            ax3d_bounds=bounds,
            wide_bounds=bounds,
            voxel_draw=_ch4_ball_voxel_path_draw(pack, idx_end, alpha_scale=alpha),
            point_s=CH3_LIK_BALL_VOXEL_POINT_S_SMALL,
            cam_azim_u=cam_u,
            measurements=_ch3_lik_we_are_here_at(pack, ws, we, bb),
        ))


def ch3_build_frames_likelihood_ball_voxel_path_story(*, rotate_during=True):
    pack = _ch4_ball_voxel_measurements_pack()
    frames = _ch3_lik_ball_voxel_intro_frames(pack)
    _ch3_lik_append_ball_voxel_path_frames(frames, pack, rotate_during=rotate_during)
    if not rotate_during:
        _ch3_lik_append_ball_voxel_rot_after(frames, pack)
    _ch3_lik_append_05a4_handoff(frames, pack)
    return _ch3_lik_story_hold(frames)


def ch4_preview_likelihood_ball_voxel_path_frame(*, intro_grow_u=1.0, path_index=None, path_u=None):
    pack = _ch4_ball_voxel_measurements_pack()
    _ch4_ball_voxel_prewarm_pack(pack)
    path = pack["path"]
    ws0, we0, bb0 = float(path[0, 0]), float(path[0, 1]), float(path[0, 2])
    bounds = _ch4_ball_voxel_view_bounds()
    at_start = path_u is None and (path_index is None or int(path_index) <= 0)
    if float(intro_grow_u) >= 0.0 and at_start:
        return _ch4_ball_voxel_frame(
            pack, ws0, we0, bb0,
            ax3d_bounds=bounds,
            wide_bounds=bounds,
            voxel_draw=_ch4_ball_voxel_cube_draw(pack["ball_voxel_start_cube"], grow_u=float(intro_grow_u)),
            point_s=CH3_LIK_BALL_VOXEL_POINT_S_SMALL if float(intro_grow_u) >= 1.0 else CH3_LIK_BALL_VOXEL_POINT_S_LARGE,
        )
    if path_u is not None:
        idx = _ch4_ball_voxel_path_index(pack, float(path_u))
    else:
        idx = int(np.clip(int(path_index or 0), 0, len(path) - 1))
    ws_i, we_i, bb_i = path[idx]
    return _ch4_ball_voxel_frame(
        pack, float(ws_i), float(we_i), float(bb_i),
        ax3d_bounds=bounds,
        wide_bounds=bounds,
        voxel_draw=_ch4_ball_voxel_path_draw(pack, idx),
        point_s=CH3_LIK_BALL_VOXEL_POINT_S_SMALL,
        measurements=_ch3_lik_we_are_here_at(pack, float(ws_i), float(we_i), float(bb_i)),
    )


def ch4_export_likelihood_ball_voxel_path():
    frames = ch3_build_frames_likelihood_ball_voxel_path_story(rotate_during=True)
    fn = "ch4_05a4_ball_voxel_path.mp4"
    save_mp4(frames, fn, duration=int(CH3_LIK_BALL_VOXEL_MS))
    print("wrote", OUTPUT_DIR / fn, f"({len(frames)} frames)")
    return OUTPUT_DIR / fn


def ch4_export_likelihood_ball_voxel_path_rot_after():
    frames = ch3_build_frames_likelihood_ball_voxel_path_story(rotate_during=False)
    fn = "ch4_05a4b_ball_voxel_path_rot_after.mp4"
    save_mp4(frames, fn, duration=int(CH3_LIK_BALL_VOXEL_MS))
    print("wrote", OUTPUT_DIR / fn, f"({len(frames)} frames)")
    return OUTPUT_DIR / fn


# --- ch4_05a5 / 05a6: GD voxel steps from CH3_LIK_3D_PATH_START (same style as 05a4) ---

CH3_LIK_GD_VOXEL_STEP_SMALL = 0.015
CH3_LIK_GD_VOXEL_HALF_SCALE = 0.5
CH3_LIK_GD_VOXEL_N_EMPHASIS = 8 if _CH3_DRAFT else 22
CH3_LIK_GD_VOXEL_N_STEP = 8 if _CH3_DRAFT else 20
CH3_LIK_GD_VOXEL_N_HOLD = 4 if _CH3_DRAFT else 8
CH3_LIK_GD_VOXEL_EMPHASIS_DIM = 0.32
CH3_LIK_GD_VOXEL_N_ROT_AFTER = 8 if _CH3_DRAFT else 40


def _ch4_ball_voxel_closest_grad_cheb(cube, center, grad):
    """Chebyshev layer of the voxel whose offset best aligns with ``-∇NLL``."""
    center = np.asarray(center, dtype=np.float64)
    g = np.asarray(grad, dtype=np.float64).ravel()
    gn = float(np.linalg.norm(g))
    if gn < 1e-14:
        return int(cube["max_cheb"])
    ghat = -g / gn
    x_edges = cube["x_edges"]
    y_edges = cube["y_edges"]
    z_edges = cube["z_edges"]
    w1 = 0.5 * (x_edges[:-1] + x_edges[1:])
    w2 = 0.5 * (y_edges[:-1] + y_edges[1:])
    bb = 0.5 * (z_edges[:-1] + z_edges[1:])
    visible = np.asarray(cube["checker"], dtype=bool)
    cheb = np.asarray(cube["cheb"], dtype=float)
    best_score = -np.inf
    best_cheb = int(cube["max_cheb"])
    for i, j, k in np.argwhere(visible):
        vc = np.array([w1[i], w2[j], bb[k]], dtype=np.float64) - center
        vn = float(np.linalg.norm(vc))
        if vn < 1e-14:
            continue
        score = float(np.dot(vc / vn, ghat))
        c = int(cheb[i, j, k])
        if score > best_score:
            best_score = score
            best_cheb = c
    return best_cheb


def _ch4_gd_voxel_measurements_pack(*, eta):
    """05a4-style pack with a GD trail from ``CH3_LIK_3D_PATH_START``."""
    pack = _ch4_ball_voxel_measurements_pack()
    study, exam, y = pack["study"], pack["exam"], pack["y"]
    trail = _ch3_lik_gd_path(
        study, exam, y,
        CH3_LIK_3D_PATH_START, CH3_LIK_GD_N_ITERS, float(eta),
    )
    pack["path"] = trail
    pack["path_waypoint_indices"] = np.arange(len(trail), dtype=int)
    pack["gd_eta"] = float(eta)
    pack.pop("ball_voxel_ready", None)
    return pack


def _ch4_gd_voxel_half():
    """Half-extent of each step's voxel ball — 05a5/05a6 use half of 05a4."""
    return float(_ch4_ball_voxel_half()) * float(CH3_LIK_GD_VOXEL_HALF_SCALE)


def _ch4_gd_voxel_prewarm_pack(pack):
    """Like ``_ch4_ball_voxel_prewarm_pack`` but with the smaller GD voxel half."""
    if pack.get("ball_voxel_ready"):
        return pack
    global_cache = _ch4_ball_voxel_global_cache(pack)
    half = _ch4_gd_voxel_half()
    path = np.asarray(pack["path"], dtype=float)
    start_cube = _ch4_ball_voxel_build_cube(global_cache, path[0], half)
    path_cubes = [
        _ch4_ball_voxel_build_cube(global_cache, r, half)
        for r in path
    ]
    pack["ball_voxel_half"] = half
    pack["ball_voxel_start_cube"] = start_cube
    pack["ball_voxel_path_cubes"] = path_cubes
    pack["ball_voxel_right"] = global_cache["right_blocks"]
    pack["ball_voxel_ready"] = True
    return pack


def _ch3_lik_gd_voxel_intro_frames(pack):
    _ch4_gd_voxel_prewarm_pack(pack)
    return _ch3_lik_ball_voxel_intro_frames(pack, skip_prewarm=True)


def _ch4_gd_voxel_step_draw(accumulated, *, grow_u=1.0, emphasize_cheb=None, emphasize_idx=-1):
    """Draw completed voxels; optionally grow/emphasize the cube at ``emphasize_idx``."""
    draws = []
    for j, c in enumerate(accumulated):
        if j == emphasize_idx:
            draws.append(_ch4_ball_voxel_cube_draw(
                c,
                grow_u=float(grow_u),
                emphasize_cheb=emphasize_cheb,
                dim_others=float(CH3_LIK_GD_VOXEL_EMPHASIS_DIM),
            ))
        else:
            draws.append(_ch4_ball_voxel_cube_draw(c, grow_u=1.0))
    return _ch4_ball_voxel_merge_draws(draws)


def _ch4_gd_voxel_append_step_frames(frames, pack, *, rotate_during=True):
    """Ten GD steps: emphasize voxel nearest ``-∇NLL``, step, keep prior voxels."""
    _ch4_gd_voxel_prewarm_pack(pack)
    study, exam, y = pack["study"], pack["exam"], pack["y"]
    trail = np.asarray(pack["path"], dtype=float)
    bounds = _ch4_ball_voxel_view_bounds()
    global_cache = _ch4_ball_voxel_global_cache(pack)
    half = float(pack["ball_voxel_half"])
    n_steps = len(trail) - 1
    n_path = max(n_steps, 1)
    rail = pack["ball_voxel_right"]
    rail_title = CH4_NLL_HEATMAP_SECTION_TITLE

    accumulated = [pack["ball_voxel_start_cube"]]

    for step_i in range(n_steps):
        ws, we, bb = (float(trail[step_i, 0]), float(trail[step_i, 1]), float(trail[step_i, 2]))
        g1, g2, gb = _ch3_nll_sum_grad_at_point(study, exam, y, ws, we, bb)
        grad = (g1, g2, gb)
        cube = accumulated[-1]
        emp = _ch4_ball_voxel_closest_grad_cheb(cube, (ws, we, bb), grad)
        cur_idx = len(accumulated) - 1
        cam_u = float(step_i) / float(n_path) if rotate_during else 0.0
        cam_rot = float(CH3_LIK_3D_CAM_PATH_ROT) if rotate_during else 0.0

        if step_i > 0:
            for tv in np.linspace(0.0, 1.0, CH3_LIK_GD_VOXEL_N_EMPHASIS, endpoint=True):
                u = ch3_knob_smoothstep(float(tv))
                frames.append(_ch4_ball_voxel_frame(
                    pack, ws, we, bb,
                    ax3d_bounds=bounds, wide_bounds=bounds,
                    voxel_draw=_ch4_gd_voxel_step_draw(
                        accumulated, grow_u=u, emphasize_cheb=emp, emphasize_idx=cur_idx,
                    ),
                    point_s=CH3_LIK_BALL_VOXEL_POINT_S_SMALL,
                    cam_azim_u=cam_u,
                    cam_rot_deg=cam_rot,
                    right_blocks=rail,
                    right_title=rail_title,
                    right_title_single_line=True,
                ))

            for _ in range(CH3_LIK_GD_VOXEL_N_HOLD):
                frames.append(_ch4_ball_voxel_frame(
                    pack, ws, we, bb,
                    ax3d_bounds=bounds, wide_bounds=bounds,
                    voxel_draw=_ch4_gd_voxel_step_draw(
                        accumulated, grow_u=1.0, emphasize_cheb=emp, emphasize_idx=cur_idx,
                    ),
                    point_s=CH3_LIK_BALL_VOXEL_POINT_S_SMALL,
                    cam_azim_u=cam_u,
                    cam_rot_deg=cam_rot,
                    right_blocks=rail,
                    right_title=rail_title,
                    right_title_single_line=True,
                ))

        ws1, we1, bb1 = (float(trail[step_i + 1, 0]), float(trail[step_i + 1, 1]), float(trail[step_i + 1, 2]))
        for tv in np.linspace(0.0, 1.0, CH3_LIK_GD_VOXEL_N_STEP, endpoint=True):
            u = ch3_knob_smoothstep(float(tv))
            wi = ws + u * (ws1 - ws)
            ei = we + u * (we1 - we)
            bi = bb + u * (bb1 - bb)
            step_cam_u = cam_u + u / float(n_path) if rotate_during else 0.0
            frames.append(_ch4_ball_voxel_frame(
                pack, wi, ei, bi,
                ax3d_bounds=bounds, wide_bounds=bounds,
                voxel_draw=_ch4_ball_voxel_merge_draws([
                    _ch4_ball_voxel_cube_draw(c, grow_u=1.0) for c in accumulated
                ]),
                point_s=CH3_LIK_BALL_VOXEL_POINT_S_SMALL,
                cam_azim_u=step_cam_u,
                cam_rot_deg=cam_rot,
                right_blocks=rail,
                right_title=rail_title,
                right_title_single_line=True,
            ))

        accumulated.append(_ch4_ball_voxel_build_cube(global_cache, trail[step_i + 1], half))

    ws_f, we_f, bb_f = (float(trail[-1, 0]), float(trail[-1, 1]), float(trail[-1, 2]))
    end_cam_u = 1.0 if rotate_during else 0.0
    end_cam_rot = float(CH3_LIK_3D_CAM_PATH_ROT) if rotate_during else 0.0
    for _ in range(max(4, CH3_LIK_VOXEL_N_HOLD // 3)):
        frames.append(_ch4_ball_voxel_frame(
            pack, ws_f, we_f, bb_f,
            ax3d_bounds=bounds, wide_bounds=bounds,
            voxel_draw=_ch4_ball_voxel_merge_draws([
                _ch4_ball_voxel_cube_draw(c, grow_u=1.0) for c in accumulated
            ]),
            point_s=CH3_LIK_BALL_VOXEL_POINT_S_SMALL,
            cam_azim_u=end_cam_u,
            cam_rot_deg=end_cam_rot,
            right_blocks=rail,
            right_title=rail_title,
            right_title_single_line=True,
        ))


def _ch4_gd_voxel_append_rot_after(frames, pack):
    """Combined 360° pan after all GD steps (05a5/05a6 rot-after variants)."""
    _ch4_gd_voxel_prewarm_pack(pack)
    trail = np.asarray(pack["path"], dtype=float)
    bounds = _ch4_ball_voxel_view_bounds()
    global_cache = _ch4_ball_voxel_global_cache(pack)
    half = float(pack["ball_voxel_half"])
    accumulated = [
        _ch4_ball_voxel_build_cube(global_cache, trail[j], half)
        for j in range(len(trail))
    ]
    ws_f, we_f, bb_f = (float(trail[-1, 0]), float(trail[-1, 1]), float(trail[-1, 2]))
    rail = pack["ball_voxel_right"]
    rail_title = CH4_NLL_HEATMAP_SECTION_TITLE
    for tv in np.linspace(0.0, 1.0, CH3_LIK_GD_VOXEL_N_ROT_AFTER, endpoint=True):
        u = ch3_knob_smoothstep(float(tv))
        frames.append(_ch4_ball_voxel_frame(
            pack, ws_f, we_f, bb_f,
            ax3d_bounds=bounds, wide_bounds=bounds,
            voxel_draw=_ch4_ball_voxel_merge_draws([
                _ch4_ball_voxel_cube_draw(c, grow_u=1.0) for c in accumulated
            ]),
            point_s=CH3_LIK_BALL_VOXEL_POINT_S_SMALL,
            cam_azim_u=u,
            cam_rot_deg=float(CH3_LIK_3D_CAM_PATH_ROT),
            right_blocks=rail,
            right_title=rail_title,
            right_title_single_line=True,
        ))


def ch3_build_frames_likelihood_gd_voxel_steps_story(*, eta, rotate_during=True):
    pack = _ch4_gd_voxel_measurements_pack(eta=float(eta))
    frames = _ch3_lik_gd_voxel_intro_frames(pack)
    _ch4_gd_voxel_append_step_frames(frames, pack, rotate_during=rotate_during)
    if not rotate_during:
        _ch4_gd_voxel_append_rot_after(frames, pack)
    return _ch3_lik_story_hold(frames)


def ch4_preview_likelihood_gd_voxel_steps_end_frame(*, eta=None):
    """Last frame — all 10 GD steps accumulated."""
    eta = float(CH3_LIK_GD_STEP if eta is None else eta)
    pack = _ch4_gd_voxel_measurements_pack(eta=eta)
    _ch4_gd_voxel_prewarm_pack(pack)
    trail = np.asarray(pack["path"], dtype=float)
    bounds = _ch4_ball_voxel_view_bounds()
    global_cache = _ch4_ball_voxel_global_cache(pack)
    half = float(pack["ball_voxel_half"])
    accumulated = [
        _ch4_ball_voxel_build_cube(global_cache, trail[j], half)
        for j in range(len(trail))
    ]
    ws_f, we_f, bb_f = (float(trail[-1, 0]), float(trail[-1, 1]), float(trail[-1, 2]))
    return _ch4_ball_voxel_frame(
        pack, ws_f, we_f, bb_f,
        ax3d_bounds=bounds, wide_bounds=bounds,
        voxel_draw=_ch4_ball_voxel_merge_draws([
            _ch4_ball_voxel_cube_draw(c, grow_u=1.0) for c in accumulated
        ]),
        point_s=CH3_LIK_BALL_VOXEL_POINT_S_SMALL,
        cam_azim_u=1.0,
        cam_rot_deg=float(CH3_LIK_3D_CAM_PATH_ROT),
        right_blocks=pack["ball_voxel_right"],
        right_title=CH4_NLL_HEATMAP_SECTION_TITLE,
        right_title_single_line=True,
    )


def ch4_preview_likelihood_gd_voxel_steps_frame(*, eta=None, step_index=0, emphasis_u=1.0):
    eta = float(CH3_LIK_GD_STEP if eta is None else eta)
    pack = _ch4_gd_voxel_measurements_pack(eta=eta)
    _ch4_gd_voxel_prewarm_pack(pack)
    study, exam, y = pack["study"], pack["exam"], pack["y"]
    trail = np.asarray(pack["path"], dtype=float)
    bounds = _ch4_ball_voxel_view_bounds()
    global_cache = _ch4_ball_voxel_global_cache(pack)
    half = float(pack["ball_voxel_half"])
    step_i = int(np.clip(int(step_index), 0, len(trail) - 2))
    ws, we, bb = (float(trail[step_i, 0]), float(trail[step_i, 1]), float(trail[step_i, 2]))
    g1, g2, gb = _ch3_nll_sum_grad_at_point(study, exam, y, ws, we, bb)
    cube = _ch4_ball_voxel_build_cube(global_cache, trail[step_i], half)
    emp = _ch4_ball_voxel_closest_grad_cheb(cube, (ws, we, bb), (g1, g2, gb))
    accumulated = [
        _ch4_ball_voxel_build_cube(global_cache, trail[j], half)
        for j in range(step_i + 1)
    ]
    return _ch4_ball_voxel_frame(
        pack, ws, we, bb,
        ax3d_bounds=bounds, wide_bounds=bounds,
        voxel_draw=_ch4_gd_voxel_step_draw(
            accumulated, grow_u=float(emphasis_u), emphasize_cheb=emp, emphasize_idx=len(accumulated) - 1,
        ),
        point_s=CH3_LIK_BALL_VOXEL_POINT_S_SMALL,
        right_blocks=pack["ball_voxel_right"],
        right_title=CH4_NLL_HEATMAP_SECTION_TITLE,
        right_title_single_line=True,
    )


def ch4_export_likelihood_gd_voxel_steps():
    frames = ch3_build_frames_likelihood_gd_voxel_steps_story(
        eta=float(CH3_LIK_GD_STEP), rotate_during=True,
    )
    fn = "ch4_05a5_gd_voxel_steps.mp4"
    save_mp4(frames, fn, duration=int(CH3_LIK_BALL_VOXEL_MS))
    print("wrote", OUTPUT_DIR / fn, f"({len(frames)} frames)")
    return OUTPUT_DIR / fn


def ch4_export_likelihood_gd_voxel_steps_rot_after():
    frames = ch3_build_frames_likelihood_gd_voxel_steps_story(
        eta=float(CH3_LIK_GD_STEP), rotate_during=False,
    )
    fn = "ch4_05a5b_gd_voxel_steps_rot_after.mp4"
    save_mp4(frames, fn, duration=int(CH3_LIK_BALL_VOXEL_MS))
    print("wrote", OUTPUT_DIR / fn, f"({len(frames)} frames)")
    return OUTPUT_DIR / fn


def ch4_export_likelihood_gd_voxel_steps_small():
    frames = ch3_build_frames_likelihood_gd_voxel_steps_story(
        eta=float(CH3_LIK_GD_VOXEL_STEP_SMALL), rotate_during=True,
    )
    fn = "ch4_05a6_gd_voxel_steps_small.mp4"
    save_mp4(frames, fn, duration=int(CH3_LIK_BALL_VOXEL_MS))
    print("wrote", OUTPUT_DIR / fn, f"({len(frames)} frames)")
    return OUTPUT_DIR / fn


def ch4_export_likelihood_gd_voxel_steps_small_rot_after():
    frames = ch3_build_frames_likelihood_gd_voxel_steps_story(
        eta=float(CH3_LIK_GD_VOXEL_STEP_SMALL), rotate_during=False,
    )
    fn = "ch4_05a6b_gd_voxel_steps_small_rot_after.mp4"
    save_mp4(frames, fn, duration=int(CH3_LIK_BALL_VOXEL_MS))
    print("wrote", OUTPUT_DIR / fn, f"({len(frames)} frames)")
    return OUTPUT_DIR / fn


# --- ch4_05b: partial derivative motivation (between ball vectors and GD) ---

CH3_LIK_PARTIAL_HALF_W_START = 0.38
CH3_LIK_PARTIAL_HALF_W_END = 0.006
CH3_LIK_PARTIAL_N_CURVE = 72 if not _CH3_DRAFT else 36
CH3_LIK_PARTIAL_VECTOR_SCALE = 0.14
CH3_LIK_PARTIAL_N_HOLD = 4 if _CH3_DRAFT else max(12, _smooth_n(8))
CH3_LIK_PARTIAL_N_DRAW = 6 if _CH3_DRAFT else max(36, _smooth_n(28))
CH3_LIK_PARTIAL_N_ROC_INTRO = 4 if _CH3_DRAFT else max(16, _smooth_n(12))
CH3_LIK_PARTIAL_N_ROC_SHRINK = 10 if _CH3_DRAFT else max(64, _smooth_n(48))
CH3_LIK_PARTIAL_N_VECTORS = 6 if _CH3_DRAFT else max(28, _smooth_n(20))
CH3_LIK_PARTIAL_N_ERASE = 6 if _CH3_DRAFT else max(36, _smooth_n(28))
CH3_LIK_PARTIAL_N_PARTIAL = 6 if _CH3_DRAFT else max(36, _smooth_n(28))
CH3_LIK_PARTIAL_N_UPDATE_REVEAL = 6 if _CH3_DRAFT else max(36, _smooth_n(28))
CH3_LIK_PARTIAL_N_ALPHA = 4 if _CH3_DRAFT else max(24, _smooth_n(18))
CH3_LIK_PARTIAL_N_PLOT = 6 if _CH3_DRAFT else max(28, _smooth_n(20))
CH3_LIK_PARTIAL_MS = 110 if not _CH3_DRAFT else 130
CH3_LIK_PARTIAL_POINT_S = 165.0

_CH4_PARTIAL_PARAM_LABELS = (
    (r"$w_{\mathrm{ST}}$", "st"),
    (r"$w_{\mathrm{EL}}$", "el"),
    (r"$b$", "b"),
)
_CH4_PARTIAL_PARAM_TEX = {
    "st": r"w_{\mathrm{ST}}",
    "el": r"w_{\mathrm{EL}}",
    "b": "b",
}


def _ch4_partial_draw_delta_bracket(ax, x_lo, x_hi, y_lo_p, y_hi_p, color):
    """Horizontal Δparam at ``y_lo_p``; vertical ΔNLL at ``x_hi``."""
    c = str(color)
    ax.plot([x_lo, x_hi], [y_lo_p, y_lo_p], color=c, lw=2.4, solid_capstyle="butt", zorder=6)
    ax.plot([x_hi, x_hi], [y_lo_p, y_hi_p], color=c, lw=4.4, solid_capstyle="butt", zorder=6)


def _ch4_partial_annotate_panel(
    ax, which, x_lo, x_hi, y_lo_p, y_hi_p, color, *, label_fs,
):
    d_nll = abs(float(y_hi_p) - float(y_lo_p))
    d_param = float(x_hi) - float(x_lo)
    pname = _CH4_PARTIAL_PARAM_TEX[which]
    fs = float(label_fs)
    dparam_tex = rf"$\Delta {pname} = {d_param:.2f}$"
    dnll_tex = rf"$\Delta NLL = $" + "\n" + rf"${d_nll:.2f}$"
    y_span = max(float(ax.get_ylim()[1] - ax.get_ylim()[0]), 1e-9)
    x_span = max(float(ax.get_xlim()[1] - ax.get_xlim()[0]), 1e-9)
    mid_x = 0.5 * (float(x_lo) + float(x_hi))
    mid_y = 0.5 * (float(y_lo_p) + float(y_hi_p))
    ax.text(
        mid_x, float(y_lo_p) + 0.028 * y_span, dparam_tex,
        va="bottom", ha="center", fontsize=fs, color=str(color), zorder=15,
    )
    ax.text(
        float(x_hi) + 0.012 * x_span, mid_y, dnll_tex,
        va="center", ha="left", fontsize=fs, color=str(color), zorder=15,
    )


def _ch4_partial_panel_colors():
    from ch4_layout import CH4_GD_ARROW_B_COLOR, CH4_GD_ARROW_EL_COLOR, CH4_GD_ARROW_ST_COLOR
    return (CH4_GD_ARROW_ST_COLOR, CH4_GD_ARROW_EL_COLOR, CH4_GD_ARROW_B_COLOR)


def _ch4_nll_scalar(ws, we, bb, study, exam, y):
    return float(-loss_log_likelihood(ws, we, bb, study, exam, y))


def _ch4_nll_param_value(which, ws, we, bb, val, study, exam, y):
    v = float(val)
    if which == "st":
        return _ch4_nll_scalar(v, we, bb, study, exam, y)
    if which == "el":
        return _ch4_nll_scalar(ws, v, bb, study, exam, y)
    return _ch4_nll_scalar(ws, we, v, study, exam, y)


def _ch4_nll_param_curve(which, ws, we, bb, half_w, study, exam, y, *, n=None):
    n = int(CH3_LIK_PARTIAL_N_CURVE if n is None else n)
    centers = {"st": float(ws), "el": float(we), "b": float(bb)}
    c = centers[which]
    d = float(half_w)
    xs = np.linspace(c - d, c + d, n, dtype=np.float64)
    ys = np.array(
        [_ch4_nll_param_value(which, ws, we, bb, x, study, exam, y) for x in xs],
        dtype=np.float64,
    )
    return xs, ys


def _ch4_nll_avg_roc(which, ws, we, bb, half_w, study, exam, y):
    centers = {"st": float(ws), "el": float(we), "b": float(bb)}
    c = centers[which]
    h = max(float(half_w), 1e-8)
    yp = _ch4_nll_param_value(which, ws, we, bb, c + h, study, exam, y)
    ym = _ch4_nll_param_value(which, ws, we, bb, c - h, study, exam, y)
    return float((yp - ym) / (2.0 * h))


def _ch4_nll_partial_triptych_ylim(ws, we, bb, half_w, study, exam, y):
    ys_all = []
    for which in ("st", "el", "b"):
        _, ys = _ch4_nll_param_curve(which, ws, we, bb, half_w, study, exam, y)
        ys_all.append(ys)
    all_y = np.concatenate(ys_all)
    pad = 0.08 * max(1e-6, float(np.nanmax(all_y) - np.nanmin(all_y)))
    return float(np.nanmin(all_y) - pad), float(np.nanmax(all_y) + pad)


def _ch4_partial_warm_curve_cache(pack, *, plot_half_w=None):
    """Precompute NLL slice curves + y-limits once (fixed axes for whole clip)."""
    plot_half_w = float(CH3_LIK_PARTIAL_HALF_W_START if plot_half_w is None else plot_half_w)
    cache = pack.setdefault("partial_curve_cache", {})
    if plot_half_w in cache:
        return cache[plot_half_w]
    study, exam, y = pack["study"], pack["exam"], pack["y"]
    ws = float(pack["partial_ws"])
    we = float(pack["partial_we"])
    bb = float(pack["partial_bb"])
    curves = {
        which: _ch4_nll_param_curve(which, ws, we, bb, plot_half_w, study, exam, y)
        for which in ("st", "el", "b")
    }
    y_lo, y_hi = _ch4_nll_partial_triptych_ylim(ws, we, bb, plot_half_w, study, exam, y)
    cache[plot_half_w] = (curves, y_lo, y_hi)
    return cache[plot_half_w]


def _ch4_partial_bottom_fully_written(bottom_prog):
    if not bottom_prog:
        return True
    return all(float(v) >= 1.0 - 1e-9 for v in bottom_prog.values())


def _ch4_partial_rails_keys(*, gd_bottom: bool, bottom_prog):
    from ch4_layout import (
        CH4_RAILS_CACHE_SHELL,
        ch4_rails_cache_key,
        ch4_rails_cache_key_gd,
    )

    if not _ch4_partial_bottom_fully_written(bottom_prog):
        return None, CH4_RAILS_CACHE_SHELL
    if gd_bottom:
        return ch4_rails_cache_key_gd(), None
    return ch4_rails_cache_key(gd_formulas=False), None


def ch3_figure_nllkeras_partial_triptych():
    """Left: dataset + knobs; right: three stacked NLL-vs-parameter panels."""
    fig = plt.figure(figsize=EXPORT_FIGSIZE)
    gs = fig.add_gridspec(1, 2, width_ratios=CH3_LIK_W12_WIDTH_RATIOS, wspace=CH3_DUO_WSPACE)
    g_left = GridSpecFromSubplotSpec(
        2, 1, subplot_spec=gs[0, 0], height_ratios=CH3_LEFT_HEIGHT_RATIOS, hspace=CH3_LEFT_HSPACE
    )
    ax_data = fig.add_subplot(g_left[0, 0])
    g_k = GridSpecFromSubplotSpec(1, 3, subplot_spec=g_left[1, 0], wspace=CH3_KNOB_WSPACE)
    axes_k = tuple(fig.add_subplot(g_k[0, j]) for j in range(3))
    g_right = GridSpecFromSubplotSpec(3, 1, subplot_spec=gs[0, 1], hspace=0.42)
    axes_partial = tuple(fig.add_subplot(g_right[i, 0]) for i in range(3))
    fig.subplots_adjust(left=0.05, right=0.97, top=0.93, bottom=0.06)
    _ch3_align_knob_axes_under_data(fig, ax_data, axes_k)
    ch3_layout_knob_axes_like_bridge_end(fig, ax_data, axes_k)
    return fig, ax_data, axes_partial, axes_k


def _ch4_partial_draw_gradient_vector(ax, x0, y0, grad, half_w, color, *, grad_ref):
    """Horizontal arrow from ``(x0, y0)``; length proportional to ``|grad|`` vs ``grad_ref``."""
    g = float(grad)
    g_ref = max(abs(float(grad_ref)), 1e-12)
    if abs(g) < 1e-12:
        return
    span = 2.0 * max(float(half_w), 1e-6)
    max_frac = 0.40
    dx = float(np.sign(g) * (abs(g) / g_ref) * max_frac * span)
    ax.annotate(
        "",
        xy=(x0 + dx, y0),
        xytext=(x0, y0),
        arrowprops=dict(
            arrowstyle="-|>",
            color=str(color),
            lw=3.0,
            mutation_scale=14.0,
            shrinkA=0.0,
            shrinkB=0.0,
        ),
        zorder=12,
    )


def ch3_frame_lik_partial_motivation(
    study, exam, y, ws, we, bb, *,
    half_w,
    plot_half_w=None,
    curve_u=1.0,
    show_secant=True,
    show_vectors=False,
    rocs=None,
    show_partials=False,
    partials=None,
    partial_lines_show=3,
    show_alpha=False,
    step_size=None,
    bottom_blocks=None,
    progress_override=None,
    write_progress=1.0,
    right_blocks=None,
    curve_cache=None,
    rails_cache_key=None,
    shell_cache_key=None,
    gd_bottom=False,
):
    from ch4_layout import (
        CH4_FORMULAS_SECTION_TITLE,
        CH4_HERE_SECTION_TITLE,
        CH4_NOTATION_SECTION_TITLE,
        ch4_cached_notation_corner_blocks,
        ch4_duo_partial_layout_tune,
        ch4_knob_asset_pack,
        ch4_we_are_here_roc_blocks,
        ch4_we_are_here_grad_line_colors,
        compose_tutorial,
    )

    ws, we, bb = float(ws), float(we), float(bb)
    half_w = float(half_w)
    plot_half_w = float(CH3_LIK_PARTIAL_HALF_W_START if plot_half_w is None else plot_half_w)
    fig, ax_data, axes_partial, axes_k = ch3_figure_nllkeras_partial_triptych()
    ch4_duo_partial_layout_tune(
        fig, ax_data, axes_partial,
        y_lift_mm=10.0, x_shift_mm=-10.0, subplot_height_frac=0.88,
    )
    leg = legend_linear_equation_values_bold_param(ws, we, bb, "all")
    ch3_draw_left_panel(
        ax_data, ws, we, bb, study, exam, y, leg,
        show_colormap=True, highlight_mistakes_flag=False,
    )
    ax_data.set_xlim(*xlim)
    ax_data.set_ylim(*ylim)
    finalize_style_legend_tex(ax_data)
    knob_rgbs, canvas_sides = ch4_knob_asset_pack()
    ch3_draw_knob_row(
        fig, axes_k, ws, we, bb, "all",
        knob_rgbs, canvas_sides,
        rot_strip_deg=0.0, strip_scale=1.0,
        knob_rots=ch3_k1_knob_rots_at(ws, we, bb), knob_scales=[1.0, 1.0, 1.0], ax_data=ax_data,
    )
    if curve_cache is None:
        curves, y_lo, y_hi = _ch4_partial_warm_curve_cache(
            {"study": study, "exam": exam, "y": y,
             "partial_ws": ws, "partial_we": we, "partial_bb": bb},
            plot_half_w=plot_half_w,
        )
    else:
        curves, y_lo, y_hi = curve_cache
    nll0 = _ch4_nll_scalar(ws, we, bb, study, exam, y)
    if partials is None:
        g1, g2, gb = _ch3_nll_sum_grad_at_point(study, exam, y, ws, we, bb)
        partials = (g1, g2, gb)
    centers = {"st": ws, "el": we, "b": bb}
    grads = {"st": partials[0], "el": partials[1], "b": partials[2]}
    cu = float(np.clip(curve_u, 0.0, 1.0))
    grad_ref = max(abs(float(partials[0])), abs(float(partials[1])), abs(float(partials[2])), 1e-12)
    label_fs = FONT_SIZE * 0.88 + 2.0
    for ax, (_xlabel, which), color in zip(
        axes_partial, _CH4_PARTIAL_PARAM_LABELS, _ch4_partial_panel_colors(),
    ):
        xs, ys = curves[which]
        n_keep = max(2, int(round(cu * (len(xs) - 1))) + 1)
        ax.plot(xs[:n_keep], ys[:n_keep], color="#333333", lw=2.4, zorder=2)
        xc = float(centers[which])
        yc = _ch4_nll_param_value(which, ws, we, bb, xc, study, exam, y)
        ax.scatter([xc], [yc], s=float(CH3_LIK_PARTIAL_POINT_S), c=[color], edgecolors="white", linewidths=1.6, zorder=8)
        ax.set_xlim(xc - plot_half_w, xc + plot_half_w)
        ax.set_ylim(y_lo, y_hi)
        if show_secant and cu >= 0.99 and not show_vectors:
            h = min(float(half_w), plot_half_w)
            x_lo, x_hi = xc - h, xc + h
            y_lo_p = _ch4_nll_param_value(which, ws, we, bb, x_lo, study, exam, y)
            y_hi_p = _ch4_nll_param_value(which, ws, we, bb, x_hi, study, exam, y)
            _ch4_partial_draw_delta_bracket(ax, x_lo, x_hi, y_lo_p, y_hi_p, color)
            ax.scatter([x_lo, x_hi], [y_lo_p, y_hi_p], s=36, c=[str(color)], zorder=7, alpha=0.9)
            _ch4_partial_annotate_panel(
                ax, which, x_lo, x_hi, y_lo_p, y_hi_p, color, label_fs=label_fs,
            )
        if show_vectors:
            _ch4_partial_draw_gradient_vector(
                ax, xc, yc, grads[which], plot_half_w, color, grad_ref=grad_ref,
            )
        ax.tick_params(labelsize=FONT_SIZE * 0.82)
        for spine in ax.spines.values():
            spine.set_linewidth(0.9)
    plot_img = fig_to_image(fig, dpi=CH3_ANIM_DPI)
    plt.close(fig)
    grad_colors = ch4_we_are_here_grad_line_colors()
    if right_blocks is None:
        right = ch4_we_are_here_roc_blocks(
            ws, we, bb, nll0, rocs,
            point_color=CH3_LIK_3D_POINT_COLOR,
            show_partials=show_partials,
            partials=partials,
            grad_line_colors=grad_colors,
            partial_lines_show=partial_lines_show,
            show_alpha=show_alpha,
            step_size=step_size,
        )
    else:
        right = right_blocks
    if bottom_blocks is None:
        from ch4_layout import ch4_cached_formula_blocks_3d_story
        bottom = ch4_cached_formula_blocks_3d_story()
    else:
        bottom = bottom_blocks
    return compose_tutorial(
        plot_img,
        right_blocks=right,
        bottom_blocks=bottom,
        corner_blocks=ch4_cached_notation_corner_blocks(),
        right_title=CH4_HERE_SECTION_TITLE,
        bottom_title=CH4_FORMULAS_SECTION_TITLE,
        corner_title=CH4_NOTATION_SECTION_TITLE,
        right_title_color=CH3_LIK_3D_POINT_COLOR,
        write_progress=write_progress,
        theme="classic_light",
        progress_override=progress_override,
        rails_cache_key=rails_cache_key,
        shell_cache_key=shell_cache_key,
    )


def _ch3_lik_partial_gd_start_pack():
    """Shared pose at ``CH3_LIK_3D_PATH_START`` (ch4_05 handoff end / ch4_06 start)."""
    pack = _ch3_lik_3d_measurements_pack(ball_colormap_limits=True)
    ws = float(CH3_LIK_3D_PATH_START[0])
    we = float(CH3_LIK_3D_PATH_START[1])
    bb = float(CH3_LIK_3D_PATH_START[2])
    pack["partial_ws"] = ws
    pack["partial_we"] = we
    pack["partial_bb"] = bb
    _ch4_partial_warm_curve_cache(pack)
    return pack


def _ch3_lik_emit_partial_frame(
    pack, *,
    half_w,
    plot_half_w=None,
    curve_u=1.0,
    show_secant=True,
    show_vectors=False,
    rocs=None,
    show_partials=False,
    partials=None,
    partial_lines_show=3,
    show_alpha=False,
    step_size=None,
    bottom_blocks=None,
    bottom_prog=None,
    right_blocks=None,
    right_prog=None,
    gd_bottom=False,
):
    from ch4_layout import ch4_bottom_per_block_progress

    study, exam, y = pack["study"], pack["exam"], pack["y"]
    ws = float(pack["partial_ws"])
    we = float(pack["partial_we"])
    bb = float(pack["partial_bb"])
    plot_hw = float(CH3_LIK_PARTIAL_HALF_W_START if plot_half_w is None else plot_half_w)
    curve_cache = pack.get("partial_curve_cache", {}).get(plot_hw)
    rails_key, shell_key = _ch4_partial_rails_keys(
        gd_bottom=gd_bottom, bottom_prog=bottom_prog,
    )
    po = {}
    if bottom_blocks and bottom_prog and not _ch4_partial_bottom_fully_written(bottom_prog):
        po["bottom"] = ch4_bottom_per_block_progress(bottom_blocks, bottom_prog)
    if right_prog is not None:
        po["right"] = right_prog
    return ch3_frame_lik_partial_motivation(
        study, exam, y, ws, we, bb,
        half_w=half_w,
        plot_half_w=plot_half_w,
        curve_u=curve_u,
        show_secant=show_secant,
        show_vectors=show_vectors,
        rocs=rocs,
        show_partials=show_partials,
        partials=partials,
        partial_lines_show=partial_lines_show,
        show_alpha=show_alpha,
        step_size=step_size,
        bottom_blocks=bottom_blocks,
        right_blocks=right_blocks,
        progress_override=po if po else None,
        write_progress=1.0,
        curve_cache=curve_cache,
        rails_cache_key=rails_key,
        shell_cache_key=shell_key,
        gd_bottom=gd_bottom,
    )


def _ch3_lik_partial_render_frame(pack, spec, *, cam_azim_u=0.0):
    """Worker entry for fast parallel export."""
    return _ch3_lik_emit_partial_frame(pack, **spec)


def ch3_partial_motivation_build_specs(pack):
    """Frame specs for partial-motivation plot phases (before 3D crossfade)."""
    from ch4_layout import ch4_formula_blocks_3d_story, ch4_formula_blocks_gd_progressive

    study, exam, y = pack["study"], pack["exam"], pack["y"]
    ws = float(pack["partial_ws"])
    we = float(pack["partial_we"])
    bb = float(pack["partial_bb"])
    g1, g2, gb = _ch3_nll_sum_grad_at_point(study, exam, y, ws, we, bb)
    partials = (g1, g2, gb)
    half_w0 = float(CH3_LIK_PARTIAL_HALF_W_START)
    half_w1 = float(CH3_LIK_PARTIAL_HALF_W_END)
    bottom_3d = ch4_formula_blocks_3d_story()
    specs = []

    def _spec(**kw):
        specs.append(kw)

    def _vec_spec(**extra):
        base = dict(
            half_w=half_w1, curve_u=1.0, show_secant=False, show_vectors=True,
            rocs=partials, show_partials=True, partials=partials,
        )
        base.update(extra)
        _spec(**base)

    for tv in np.linspace(0.0, 1.0, CH3_LIK_PARTIAL_N_DRAW, endpoint=True):
        _spec(
            half_w=half_w0, curve_u=ch3_knob_smoothstep(float(tv)), show_secant=False,
            bottom_blocks=bottom_3d, bottom_prog={0: 1.0, 1: 1.0}, gd_bottom=False,
        )

    for _ in range(CH3_LIK_PARTIAL_N_HOLD):
        _spec(
            half_w=half_w0, curve_u=1.0, show_secant=False,
            bottom_blocks=bottom_3d, bottom_prog={0: 1.0, 1: 1.0}, gd_bottom=False,
        )

    rocs_wide = (
        _ch4_nll_avg_roc("st", ws, we, bb, half_w0, study, exam, y),
        _ch4_nll_avg_roc("el", ws, we, bb, half_w0, study, exam, y),
        _ch4_nll_avg_roc("b", ws, we, bb, half_w0, study, exam, y),
    )
    for _ in range(CH3_LIK_PARTIAL_N_ROC_INTRO):
        _spec(
            half_w=half_w0, curve_u=1.0, show_secant=True, rocs=rocs_wide,
            bottom_blocks=bottom_3d, bottom_prog={0: 1.0, 1: 1.0}, gd_bottom=False,
        )

    for tv in np.linspace(0.0, 1.0, CH3_LIK_PARTIAL_N_ROC_SHRINK, endpoint=True):
        u = ch3_knob_smoothstep(float(tv))
        hw = half_w0 + u * (half_w1 - half_w0)
        rocs = (
            _ch4_nll_avg_roc("st", ws, we, bb, hw, study, exam, y),
            _ch4_nll_avg_roc("el", ws, we, bb, hw, study, exam, y),
            _ch4_nll_avg_roc("b", ws, we, bb, hw, study, exam, y),
        )
        _spec(
            half_w=hw, curve_u=1.0, show_secant=True, show_vectors=False, rocs=rocs,
            bottom_blocks=bottom_3d, bottom_prog={0: 1.0, 1: 1.0}, gd_bottom=False,
        )

    for tv in np.linspace(0.0, 1.0, CH3_LIK_PARTIAL_N_VECTORS, endpoint=True):
        u = ch3_knob_smoothstep(float(tv))
        _spec(
            half_w=half_w1, curve_u=1.0, show_secant=False,
            show_vectors=u >= 0.08, rocs=rocs_wide, show_partials=False,
            bottom_blocks=bottom_3d, bottom_prog={0: 1.0, 1: 1.0}, gd_bottom=False,
        )

    for _ in range(max(6, CH3_SCRIPT_N_HOLD // 5)):
        _spec(
            half_w=half_w1, curve_u=1.0, show_secant=False, show_vectors=True,
            rocs=rocs_wide, show_partials=False,
            bottom_blocks=bottom_3d, bottom_prog={0: 1.0, 1: 1.0}, gd_bottom=False,
        )

    for tv in np.linspace(1.0, 0.0, CH3_LIK_PARTIAL_N_ERASE, endpoint=True):
        u = ch3_knob_smoothstep(float(tv))
        _spec(
            half_w=half_w1, curve_u=1.0, show_secant=False, show_vectors=True,
            rocs=rocs_wide, show_partials=False,
            bottom_blocks=bottom_3d, bottom_prog={0: 1.0, 1: u}, gd_bottom=False,
        )

    for n_p in (1, 2, 3):
        bottom = ch4_formula_blocks_gd_progressive(n_grad_lines=n_p)
        for tv in np.linspace(0.0, 1.0, CH3_LIK_PARTIAL_N_PARTIAL, endpoint=True):
            u = ch3_knob_smoothstep(float(tv))
            _vec_spec(
                partial_lines_show=n_p,
                bottom_blocks=bottom,
                bottom_prog={0: 1.0, 1: u},
                gd_bottom=True,
            )

    for n_u in (1, 2, 3):
        bottom = ch4_formula_blocks_gd_progressive(n_grad_lines=3, n_update_lines=n_u)
        for tv in np.linspace(0.0, 1.0, CH3_LIK_PARTIAL_N_UPDATE_REVEAL, endpoint=True):
            u = ch3_knob_smoothstep(float(tv))
            blk = 2 if n_u > 0 else 1
            _vec_spec(
                partial_lines_show=3,
                bottom_blocks=bottom,
                bottom_prog={0: 1.0, 1: 1.0, blk: u},
                gd_bottom=True,
            )

    for tv in np.linspace(0.0, 1.0, CH3_LIK_PARTIAL_N_ALPHA, endpoint=True):
        u = ch3_knob_smoothstep(float(tv))
        bottom = ch4_formula_blocks_gd_progressive(n_grad_lines=3, n_update_lines=3)
        _vec_spec(
            partial_lines_show=3,
            show_alpha=u >= 0.5,
            step_size=CH3_LIK_GD_STEP,
            bottom_blocks=bottom,
            bottom_prog={0: 1.0, 1: 1.0, 2: 1.0},
            gd_bottom=True,
        )

    _vec_spec(
        partial_lines_show=3,
        show_alpha=True,
        step_size=CH3_LIK_GD_STEP,
        bottom_blocks=ch4_formula_blocks_gd_progressive(n_grad_lines=3, n_update_lines=3),
        bottom_prog={0: 1.0, 1: 1.0, 2: 1.0},
        gd_bottom=True,
    )
    pack["partial_tail"] = {
        "partials": partials,
        "half_w1": half_w1,
    }
    return specs


def ch3_build_frames_likelihood_partial_motivation_story(*, parallel=None):
    """Motivate partial derivatives: 1-D NLL slices → shrinking Δ → ∂ → GD formulas → 3D."""
    from ch4_export_pipeline import build_tutorial_frames

    pack = _ch3_lik_partial_gd_start_pack()
    gd_pack = _ch3_lik_3d_gd_pack()
    specs = ch3_partial_motivation_build_specs(pack)

    frames = build_tutorial_frames(
        pack,
        specs,
        _ch3_lik_partial_render_frame,
        prewarm="partial",
        parallel=parallel,
        progress_label="ch4_05b partial",
        render_fn_name="_ch3_lik_partial_render_frame",
    )

    frame_gd_open = _ch3_lik_gd_render_frame(
        gd_pack, _ch3_lik_gd_frame_specs(gd_pack)[0], cam_azim_u=0.0,
    )
    frame_plot_from = frames[-1]
    for tv in np.linspace(0.0, 1.0, CH3_LIK_PARTIAL_N_PLOT, endpoint=True):
        u = ch3_knob_smoothstep(float(tv))
        frames.append(ch4_blend_images(frame_plot_from, frame_gd_open, u))

    for _ in range(max(8, CH3_SCRIPT_N_HOLD // 4)):
        frames.append(frame_gd_open.copy())
    return frames


def ch4_preview_likelihood_partial_motivation_frame(*, phase="roc_wide"):
    """Preview one partial-motivation frame."""
    from ch4_layout import ch4_formula_blocks_3d_story, ch4_formula_blocks_gd_story

    pack = _ch3_lik_partial_gd_start_pack()
    study, exam, y = pack["study"], pack["exam"], pack["y"]
    ws = float(pack["partial_ws"])
    we = float(pack["partial_we"])
    bb = float(pack["partial_bb"])
    g1, g2, gb = _ch3_nll_sum_grad_at_point(study, exam, y, ws, we, bb)
    partials = (g1, g2, gb)
    half_w0 = float(CH3_LIK_PARTIAL_HALF_W_START)
    half_w1 = float(CH3_LIK_PARTIAL_HALF_W_END)
    bottom_3d = ch4_formula_blocks_3d_story()
    bottom_gd = ch4_formula_blocks_gd_story()
    if phase == "draw":
        return _ch3_lik_emit_partial_frame(
            pack, half_w=half_w0, curve_u=0.45, show_secant=False,
            bottom_blocks=bottom_3d, bottom_prog={0: 1.0, 1: 1.0},
        )
    if phase == "roc_wide":
        rocs = (
            _ch4_nll_avg_roc("st", ws, we, bb, half_w0, study, exam, y),
            _ch4_nll_avg_roc("el", ws, we, bb, half_w0, study, exam, y),
            _ch4_nll_avg_roc("b", ws, we, bb, half_w0, study, exam, y),
        )
        return _ch3_lik_emit_partial_frame(
            pack, half_w=half_w0, curve_u=1.0, show_secant=True, rocs=rocs,
            bottom_blocks=bottom_3d, bottom_prog={0: 1.0, 1: 1.0},
        )
    if phase == "roc_tight":
        return _ch3_lik_emit_partial_frame(
            pack, half_w=half_w1, curve_u=1.0, show_secant=True,
            rocs=partials, show_partials=True, partials=partials,
            bottom_blocks=bottom_3d, bottom_prog={0: 1.0, 1: 1.0},
        )
    if phase == "vectors":
        rocs = (
            _ch4_nll_avg_roc("st", ws, we, bb, half_w1, study, exam, y),
            _ch4_nll_avg_roc("el", ws, we, bb, half_w1, study, exam, y),
            _ch4_nll_avg_roc("b", ws, we, bb, half_w1, study, exam, y),
        )
        return _ch3_lik_emit_partial_frame(
            pack, half_w=half_w1, curve_u=1.0, show_vectors=True,
            rocs=rocs, show_partials=False,
            bottom_blocks=bottom_3d, bottom_prog={0: 1.0, 1: 1.0},
        )
    if phase == "mid_grad":
        from ch4_layout import ch4_formula_blocks_gd_progressive
        return _ch3_lik_emit_partial_frame(
            pack, half_w=half_w1, curve_u=1.0, show_vectors=True,
            rocs=partials, show_partials=True, partials=partials, partial_lines_show=2,
            bottom_blocks=ch4_formula_blocks_gd_progressive(n_grad_lines=2),
            bottom_prog={0: 1.0, 1: 1.0}, gd_bottom=True,
        )
    gd_pack = _ch3_lik_3d_gd_pack()
    return _ch3_lik_gd_render_frame(gd_pack, _ch3_lik_gd_frame_specs(gd_pack)[0], cam_azim_u=0.0)


def ch4_export_likelihood_partial_motivation(*, parallel=None):
    frames = ch3_build_frames_likelihood_partial_motivation_story(parallel=parallel)
    fn = "ch4_05b_partial_derivative_motivation.mp4"
    save_mp4(frames, fn, duration=int(CH3_LIK_PARTIAL_MS))
    print("wrote", OUTPUT_DIR / fn, f"({len(frames)} frames)")
    return OUTPUT_DIR / fn


### Likelihood story clips (ch4_02–07)

One export cell per clip — run **`ch4-likelihood-02`** through **`ch4-likelihood-07`** in order.

1. **ch4_02** — likelihood w₁₂ landscape (knob labels)
2. **ch4_03** — Ch4 template + Notation/Formulas + log → NLL + p(y|x)
3. **ch4_04** — 3D weight space + We are here + NLL trajectory (90° camera pan)
4. **ch4_05a** — CT scan: NLL heatmap planes sweep w_ST, w_EL, b
5. **ch4_05a2** — diagonal voxel fill: checkerboard cubes swept corner → corner
6. **ch4_05a3** — same as 05a2 with ¼-size voxels (4× resolution per axis)
7. **ch4_05a4** — ball-sized voxel cube at path start, accumulate voxels along ch4_05 path
8. **ch4_05a4b** — same as 05a4 but camera rotates after path completes
9. **ch4_05a5** — GD voxel steps (η = ch4_06), emphasize voxel nearest −∇NLL, keep trail
10. **ch4_05a5b** — same as 05a5 but camera rotates after all GD steps
11. **ch4_05a6** — same as 05a5 with smaller step size
12. **ch4_05a6b** — same as 05a6 but camera rotates after all GD steps
13. **ch4_05** — ball of heatmapped gradient vectors + intro zoom/spin
14. **ch4_05b** — partial derivative motivation: 1-D NLL slices → Δ → ∂ → GD formulas
15. **ch4_06** — sequential GD: colored axis arrows, per-parameter updates (10 steps)
16. **ch4_07** — like ch4_06 opening, then red combined gradient arrow + simultaneous GD


In [ ]:
# ch4_02 — likelihood w₁₂ landscape (knob labels)
ch4_export_likelihood_w12_landscape()


In [212]:
# ch4_03 — Ch4 template, notation, log/NLL morph, p(y|x)
ch4_export_likelihood_notation_nll()


IMAGEIO FFMPEG_WRITER WARNING: input image is not divisible by macro_block_size=16, resizing from (3000, 1900) to (3008, 1904) to ensure video compatibility with most codecs and players. To prevent resizing, make your input image divisible by the macro_block_size or set the macro_block_size to 1 (risking incompatibility).


wrote renders/ch4_03_likelihood_notation_nll.mp4


PosixPath('renders/ch4_03_likelihood_notation_nll.mp4')

In [ ]:
# ch4_04 — 3D (w_ST, w_EL, b) measurements + colormap path
ch4_export_likelihood_3d_measurements()


In [213]:
# ch4_05a — CT scan: NLL heatmap planes along each axis
ch4_export_likelihood_ct_scan()


IMAGEIO FFMPEG_WRITER WARNING: input image is not divisible by macro_block_size=16, resizing from (3000, 1900) to (3008, 1904) to ensure video compatibility with most codecs and players. To prevent resizing, make your input image divisible by the macro_block_size or set the macro_block_size to 1 (risking incompatibility).


wrote renders/ch4_05a_likelihood_3d_ct_scan.mp4 (830 frames)


PosixPath('renders/ch4_05a_likelihood_3d_ct_scan.mp4')

In [ ]:
# ch4_05a2 — diagonal checkerboard voxel fill (−3…3 cube)
ch4_export_likelihood_voxel_fill()


In [214]:
# ch4_05a3 — fine voxel fill (¼ cube size, 4× cells per axis)
ch4_export_likelihood_voxel_fill_fine()


IMAGEIO FFMPEG_WRITER WARNING: input image is not divisible by macro_block_size=16, resizing from (3000, 1900) to (3008, 1904) to ensure video compatibility with most codecs and players. To prevent resizing, make your input image divisible by the macro_block_size or set the macro_block_size to 1 (risking incompatibility).


wrote renders/ch4_05a3_diagonal_voxel_fill_fine.mp4 (328 frames)


PosixPath('renders/ch4_05a3_diagonal_voxel_fill_fine.mp4')

In [ ]:
# ch4_05a4 — ball-sized voxel cube grows at path start, accumulates along path
ch4_export_likelihood_ball_voxel_path()


In [ ]:
# ch4_05a4b — same path; camera pan after path completes
ch4_export_likelihood_ball_voxel_path_rot_after()


In [ ]:
# ch4_05a5 — GD voxel steps (same η as ch4_06), accumulate voxels
ch4_export_likelihood_gd_voxel_steps()


IMAGEIO FFMPEG_WRITER WARNING: input image is not divisible by macro_block_size=16, resizing from (3000, 1900) to (3008, 1904) to ensure video compatibility with most codecs and players. To prevent resizing, make your input image divisible by the macro_block_size or set the macro_block_size to 1 (risking incompatibility).


wrote renders/ch4_05a5_gd_voxel_steps.mp4 (594 frames)


PosixPath('renders/ch4_05a5_gd_voxel_steps.mp4')

In [ ]:
# ch4_05a5b — same GD steps; camera pan after all steps
ch4_export_likelihood_gd_voxel_steps_rot_after()


In [ ]:
# ch4_05a6 — GD voxel steps (small η), accumulate voxels
ch4_export_likelihood_gd_voxel_steps_small()


IMAGEIO FFMPEG_WRITER WARNING: input image is not divisible by macro_block_size=16, resizing from (3000, 1900) to (3008, 1904) to ensure video compatibility with most codecs and players. To prevent resizing, make your input image divisible by the macro_block_size or set the macro_block_size to 1 (risking incompatibility).


wrote renders/ch4_05a6_gd_voxel_steps_small.mp4 (594 frames)


PosixPath('renders/ch4_05a6_gd_voxel_steps_small.mp4')

In [ ]:
# ch4_05a6b — same small-step GD; camera pan after all steps
ch4_export_likelihood_gd_voxel_steps_small_rot_after()


In [ ]:
# ch4_05 — ball of NLL-colored gradient vectors + intro zoom/spin
ch4_export_likelihood_3d_ball_vectors()


IMAGEIO FFMPEG_WRITER WARNING: input image is not divisible by macro_block_size=16, resizing from (3000, 1900) to (3008, 1904) to ensure video compatibility with most codecs and players. To prevent resizing, make your input image divisible by the macro_block_size or set the macro_block_size to 1 (risking incompatibility).


wrote renders/ch4_05_likelihood_3d_ball_vectors.mp4


PosixPath('renders/ch4_05_likelihood_3d_ball_vectors.mp4')

In [ ]:
# ch4_05b — partial derivative motivation (1-D NLL → ∂ → GD opening)
ch4_export_likelihood_partial_motivation()


Chapter 4 layout OK — handwriting: Patrick Hand
Chapter 4 layout OK — handwriting: Patrick Hand
Chapter 4 layout OK — handwriting: Patrick Hand
Chapter 4 layout OK — handwriting: Patrick Hand
Chapter 4 layout OK — handwriting: Patrick Hand
Chapter 4 layout OK — handwriting: Patrick Hand
Chapter 3 setup OK — 20 clean, 26 with noise.
Chapter 4 layout OK — handwriting: Patrick Hand
Chapter 3 setup OK — 20 clean, 26 with noise.
Chapter 3 setup OK — 20 clean, 26 with noise.
Chapter 3 setup OK — 20 clean, 26 with noise.
Chapter 4 layout OK — handwriting: Patrick Hand
Chapter 3 setup OK — 20 clean, 26 with noise.
Chapter 3 setup OK — 20 clean, 26 with noise.


ch3:580: SyntaxWarning: invalid escape sequence '\s'
ch3:584: SyntaxWarning: invalid escape sequence '\s'
ch3:580: SyntaxWarning: invalid escape sequence '\s'
ch3:584: SyntaxWarning: invalid escape sequence '\s'
ch3:580: SyntaxWarning: invalid escape sequence '\s'
ch3:584: SyntaxWarning: invalid escape sequence '\s'
ch3:580: SyntaxWarning: invalid escape sequence '\s'
ch3:584: SyntaxWarning: invalid escape sequence '\s'
ch3:580: SyntaxWarning: invalid escape sequence '\s'
ch3:584: SyntaxWarning: invalid escape sequence '\s'
ch3:580: SyntaxWarning: invalid escape sequence '\s'
ch3:584: SyntaxWarning: invalid escape sequence '\s'
ch3:580: SyntaxWarning: invalid escape sequence '\s'
ch3:584: SyntaxWarning: invalid escape sequence '\s'
ch3:580: SyntaxWarning: invalid escape sequence '\s'
ch3:584: SyntaxWarning: invalid escape sequence '\s'
Exception in initializer:
Traceback (most recent call last):
  File "/opt/homebrew/Cellar/python@3.12/3.12.13_2/Frameworks/Python.framework/Versions/3.1

Chapter 3 setup OK — 20 clean, 26 with noise.


BrokenProcessPool: A process in the process pool was terminated abruptly while the future was running or pending.

In [ ]:
# ch4_06 — gradient descent on NLL with animated weight updates
ch4_export_likelihood_3d_gd()


/Users/lance/Documents/BostonUniversity/PROJECTS/reproduce-those-animations/007-full-logistic-regression/handwrite_tutorial.py:367: UserWarning: Glyph 8592 (\N{LEFTWARDS ARROW}) missing from font(s) Patrick Hand.
  w_px, h_px, d_px = renderer.get_text_width_height_descent(str(text), fp, ismath=False)
/Users/lance/Documents/BostonUniversity/PROJECTS/reproduce-those-animations/007-full-logistic-regression/handwrite_tutorial.py:367: UserWarning: Glyph 945 (\N{GREEK SMALL LETTER ALPHA}) missing from font(s) Patrick Hand.
  w_px, h_px, d_px = renderer.get_text_width_height_descent(str(text), fp, ismath=False)
/Users/lance/Documents/BostonUniversity/PROJECTS/reproduce-those-animations/007-full-logistic-regression/handwrite_tutorial.py:367: UserWarning: Glyph 8706 (\N{PARTIAL DIFFERENTIAL}) missing from font(s) Patrick Hand.
  w_px, h_px, d_px = renderer.get_text_width_height_descent(str(text), fp, ismath=False)
/Users/lance/Documents/BostonUniversity/PROJECTS/reproduce-those-animations/007-f

Chapter 4 layout OK — handwriting: Patrick Hand
Chapter 4 layout OK — handwriting: Patrick Hand
Chapter 4 layout OK — handwriting: Patrick Hand
Chapter 4 layout OK — handwriting: Patrick Hand
Chapter 4 layout OK — handwriting: Patrick Hand
Chapter 4 layout OK — handwriting: Patrick Hand
Chapter 4 layout OK — handwriting: Patrick Hand
Chapter 3 setup OK — 20 clean, 26 with noise.
Chapter 3 setup OK — 20 clean, 26 with noise.
Chapter 3 setup OK — 20 clean, 26 with noise.
Chapter 3 setup OK — 20 clean, 26 with noise.
Chapter 3 setup OK — 20 clean, 26 with noise.
Chapter 4 layout OK — handwriting: Patrick Hand
Chapter 3 setup OK — 20 clean, 26 with noise.
Chapter 3 setup OK — 20 clean, 26 with noise.


ch3:580: SyntaxWarning: invalid escape sequence '\s'
ch3:584: SyntaxWarning: invalid escape sequence '\s'
ch3:580: SyntaxWarning: invalid escape sequence '\s'
ch3:584: SyntaxWarning: invalid escape sequence '\s'
ch3:580: SyntaxWarning: invalid escape sequence '\s'
ch3:584: SyntaxWarning: invalid escape sequence '\s'
ch3:580: SyntaxWarning: invalid escape sequence '\s'
ch3:584: SyntaxWarning: invalid escape sequence '\s'
ch3:580: SyntaxWarning: invalid escape sequence '\s'
ch3:584: SyntaxWarning: invalid escape sequence '\s'
ch3:580: SyntaxWarning: invalid escape sequence '\s'
ch3:584: SyntaxWarning: invalid escape sequence '\s'
ch3:580: SyntaxWarning: invalid escape sequence '\s'
ch3:584: SyntaxWarning: invalid escape sequence '\s'
ch3:580: SyntaxWarning: invalid escape sequence '\s'
ch3:584: SyntaxWarning: invalid escape sequence '\s'


Chapter 3 setup OK — 20 clean, 26 with noise.


/Users/lance/Documents/BostonUniversity/PROJECTS/reproduce-those-animations/007-full-logistic-regression/handwrite_tutorial.py:367: UserWarning: Glyph 8592 (\N{LEFTWARDS ARROW}) missing from font(s) Patrick Hand.
  w_px, h_px, d_px = renderer.get_text_width_height_descent(str(text), fp, ismath=False)
/Users/lance/Documents/BostonUniversity/PROJECTS/reproduce-those-animations/007-full-logistic-regression/handwrite_tutorial.py:367: UserWarning: Glyph 945 (\N{GREEK SMALL LETTER ALPHA}) missing from font(s) Patrick Hand.
  w_px, h_px, d_px = renderer.get_text_width_height_descent(str(text), fp, ismath=False)
/Users/lance/Documents/BostonUniversity/PROJECTS/reproduce-those-animations/007-full-logistic-regression/handwrite_tutorial.py:367: UserWarning: Glyph 8706 (\N{PARTIAL DIFFERENTIAL}) missing from font(s) Patrick Hand.
  w_px, h_px, d_px = renderer.get_text_width_height_descent(str(text), fp, ismath=False)
/Users/lance/Documents/BostonUniversity/PROJECTS/reproduce-those-animations/007-f

  ch4_06: 42/420  (38s, 8 workers)
  ch4_06: 84/420  (66s, 8 workers)
  ch4_06: 126/420  (94s, 8 workers)
  ch4_06: 168/420  (122s, 8 workers)
  ch4_06: 210/420  (155s, 8 workers)
  ch4_06: 252/420  (183s, 8 workers)
  ch4_06: 294/420  (211s, 8 workers)
  ch4_06: 336/420  (240s, 8 workers)
  ch4_06: 378/420  (271s, 8 workers)
  ch4_06: 420/420  (299s, 8 workers)


IMAGEIO FFMPEG_WRITER WARNING: input image is not divisible by macro_block_size=16, resizing from (3000, 1900) to (3008, 1904) to ensure video compatibility with most codecs and players. To prevent resizing, make your input image divisible by the macro_block_size or set the macro_block_size to 1 (risking incompatibility).


wrote renders/ch4_06_likelihood_3d_gd.mp4 (436 frames)


PosixPath('renders/ch4_06_likelihood_3d_gd.mp4')

In [ ]:
# ch4_07 — combined gradient arrow GD (all params at once)
ch4_export_likelihood_3d_gd_combined()


Chapter 4 layout OK — handwriting: Patrick Hand
Chapter 4 layout OK — handwriting: Patrick Hand
Chapter 4 layout OK — handwriting: Patrick Hand
Chapter 4 layout OK — handwriting: Patrick Hand
Chapter 4 layout OK — handwriting: Patrick Hand
Chapter 4 layout OK — handwriting: Patrick Hand
Chapter 3 setup OK —Chapter 3 setup OK —  2020 clean,  clean, 2626 with noise.
 with noise.
Chapter 3 setup OK — 20 clean, 26 with noise.
Chapter 3 setup OK — 20 clean, 26 with noise.
Chapter 4 layout OK — handwriting: Patrick Hand
Chapter 3 setup OK — 20 clean, 26 with noise.
Chapter 4 layout OK — handwriting: Patrick Hand
Chapter 3 setup OK — 20 clean, 26 with noise.


ch3:580: SyntaxWarning: invalid escape sequence '\s'
ch3:584: SyntaxWarning: invalid escape sequence '\s'
ch3:580: SyntaxWarning: invalid escape sequence '\s'
ch3:584: SyntaxWarning: invalid escape sequence '\s'
ch3:580: SyntaxWarning: invalid escape sequence '\s'
ch3:584: SyntaxWarning: invalid escape sequence '\s'
ch3:580: SyntaxWarning: invalid escape sequence '\s'
ch3:584: SyntaxWarning: invalid escape sequence '\s'
ch3:580: SyntaxWarning: invalid escape sequence '\s'
ch3:584: SyntaxWarning: invalid escape sequence '\s'
ch3:580: SyntaxWarning: invalid escape sequence '\s'
ch3:584: SyntaxWarning: invalid escape sequence '\s'
ch3:580: SyntaxWarning: invalid escape sequence '\s'
ch3:584: SyntaxWarning: invalid escape sequence '\s'
ch3:580: SyntaxWarning: invalid escape sequence '\s'
ch3:584: SyntaxWarning: invalid escape sequence '\s'


Chapter 3 setup OK — 20 clean, 26 with noise.
Chapter 3 setup OK — 20 clean, 26 with noise.
  ch4_07: 24/248  (21s, 8 workers)


/Users/lance/Documents/BostonUniversity/PROJECTS/reproduce-those-animations/007-full-logistic-regression/handwrite_tutorial.py:367: UserWarning: Glyph 8592 (\N{LEFTWARDS ARROW}) missing from font(s) Patrick Hand.
  w_px, h_px, d_px = renderer.get_text_width_height_descent(str(text), fp, ismath=False)
/Users/lance/Documents/BostonUniversity/PROJECTS/reproduce-those-animations/007-full-logistic-regression/handwrite_tutorial.py:367: UserWarning: Glyph 945 (\N{GREEK SMALL LETTER ALPHA}) missing from font(s) Patrick Hand.
  w_px, h_px, d_px = renderer.get_text_width_height_descent(str(text), fp, ismath=False)
/Users/lance/Documents/BostonUniversity/PROJECTS/reproduce-those-animations/007-full-logistic-regression/handwrite_tutorial.py:367: UserWarning: Glyph 8706 (\N{PARTIAL DIFFERENTIAL}) missing from font(s) Patrick Hand.
  w_px, h_px, d_px = renderer.get_text_width_height_descent(str(text), fp, ismath=False)
/Users/lance/Documents/BostonUniversity/PROJECTS/reproduce-those-animations/007-f

  ch4_07: 48/248  (38s, 8 workers)
  ch4_07: 72/248  (55s, 8 workers)
  ch4_07: 96/248  (72s, 8 workers)
  ch4_07: 120/248  (88s, 8 workers)
  ch4_07: 144/248  (105s, 8 workers)
  ch4_07: 168/248  (122s, 8 workers)
  ch4_07: 192/248  (139s, 8 workers)
  ch4_07: 216/248  (156s, 8 workers)
  ch4_07: 240/248  (173s, 8 workers)
  ch4_07: 248/248  (178s, 8 workers)


IMAGEIO FFMPEG_WRITER WARNING: input image is not divisible by macro_block_size=16, resizing from (3000, 1900) to (3008, 1904) to ensure video compatibility with most codecs and players. To prevent resizing, make your input image divisible by the macro_block_size or set the macro_block_size to 1 (risking incompatibility).


wrote renders/ch4_07_likelihood_3d_gd_combined.mp4 (264 frames)


PosixPath('renders/ch4_07_likelihood_3d_gd_combined.mp4')